# অলীকবচন — Bengali LLM Hallucination Detection (Final Inference)

Self-contained inference notebook (2×T4, Internet ON). Judges: TigerLLM-9B + Qwen3.5-9B (first-token logits) + a fine-tuned BanglaBERT encoder (**attached checkpoint — no training here**), combined by a per-track meta-judge fitted on the labeled sample, then overridden by deterministic layers (reference banks, BenHalluEval exact pairs, idiom/context tiers, sympy). Attach: `clf_finetuned/`, the 5 bank CSVs, `benhallu-data`. Test-set path is fixed below.


In [1]:
# ---- Competition data paths (organizers may swap the test set at this path) ----
DATA_DIR = "/kaggle/input/competitions/bengali-hallucination"
TEST_PATH_FIXED    = f"{DATA_DIR}/test set.csv"
LABELED_PATH_FIXED = f"{DATA_DIR}/dataset samples.json"
SAMPLE_PATH_FIXED  = f"{DATA_DIR}/sample submission.csv"


In [2]:
# Internet-first setup. With Kaggle Internet ON this installs the runtime bits
# previously prepared by the loader notebooks.
import socket, glob, os

def has_internet(host="pypi.org", port=443, timeout=3):
    try:
        socket.setdefaulttimeout(timeout)
        socket.create_connection((host, port)).close()
        return True
    except OSError:
        return False

ONLINE = has_internet()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
_wheel_dirs = glob.glob("/kaggle/input/**/wheels", recursive=True)
if _wheel_dirs:
    get_ipython().system(f'pip install -q --no-index --find-links={_wheel_dirs[0]} bitsandbytes')
    if ONLINE:
        get_ipython().system('pip install -q -U "transformers>=4.50" accelerate hf_transfer datasets huggingface_hub kagglehub')
elif ONLINE:
    get_ipython().system('pip install -q -U "transformers>=4.50" accelerate "bitsandbytes>=0.46.1" hf_transfer datasets huggingface_hub kagglehub')
else:
    print("no wheels attached, no internet - relying on the pre-installed image")

if ONLINE:
    # csebuetnlp normalizer - BanglaBERT preprocessing best practice (fail-soft)
    get_ipython().system('pip install -q "git+https://github.com/csebuetnlp/normalizer" || true')
import bitsandbytes as _bnb_check
print("bitsandbytes:", _bnb_check.__version__, "| online:", ONLINE)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re, json, glob, time, math, unicodedata, shutil, ast
import numpy as np
import pandas as pd
import torch
import transformers
from packaging.version import Version

print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}     :", torch.cuda.get_device_name(i))

assert Version(transformers.__version__) >= Version("4.50"), "Gemma 3 needs transformers >= 4.50"
assert torch.cuda.is_available(), "Enable a GPU accelerator"


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


transformers: 5.14.1
torch       : 2.10.0+cu128
GPU 0     : Tesla T4
GPU 1     : Tesla T4


In [4]:
# ---- Configuration ---------------------------------------------------------
MAX_CTX_CHARS  = 767
BATCH_SIZE     = 4      # v5: OOM-free on 2xT4; auto-backoff handles OOM
MAX_LEN        = 2048
CHUNK_SIZE     = 200
USE_MATH_COT   = True   # v6.18 A/B: OFF - math rows = pure Qwen zero-shot (p_q > 0.5)
                         # + sympy only. BenHalluEval: Qwen zero-shot 26.80 BHS beat
                         # Qwen+CoT 28.90 on Reasoning. Flip back to True to restore CoT.
USE_SQUAD_REFS = True
USE_SYMPY      = True
USE_QA_VERIFY  = False  # v5.1: labeled gate invalid - QA model trained on squad_bn train split
USE_RETRIEVAL  = True
USE_ENSEMBLE   = True    # Qwen Bengali judge; auto-disabled on single GPU for 16GB safety
USE_GRAMMAR    = True    # optional: uses local/Kaggle files if present, otherwise disables cleanly
USE_REL_ROUTE  = True
USE_CTX_MATCH  = True
USE_CB_REFS    = True    # optional: uses local/Kaggle files if present, otherwise disables cleanly
USE_EXTRA_QA_REFS = True # BanglaRQA/BanglaQuAD/NCTB-QA exact QA references
USE_KAGGLE_BAGDHARA = False # off: Kaggle cannot attach new datasets in a committed
                            # (non-interactive) run; the HF bagdhara set is used instead
USE_GROUPS     = True

# v6 additions
USE_CLF        = True    # encoder classifier fine-tuned on synthetic squad_bn contrastive pairs
USE_XLING_EN   = True    # English-instruction Qwen logit pass (organizers: strongest signal)
USE_XLING_COT  = False   # v6.19 A/B: translate-verify generation OFF. Gate passed only
                         # marginally (+0.054 on 55 compass rows = noise-tier) at 30-60 min
                         # test cost, and its turning ON this run confounds the math A/B.
                         # USE_XLING_EN (cheap English-logit p_qe) stays ON. Flip to True to restore.    # translate-then-verify generation on uncertain rows (budget-capped)
XLING_COT_CAP  = 450
XLING_VAL_N    = 60
MATH_SC_N      = 1
QWEN_ONLY_MATH = True    # v6.18: math rows are decided by Qwen alone. BenHalluEval
                         # measured TigerLLM as the WORST math judge of 7 models
                         # (BHS 53.45, 85.5% miss rate) while Qwen was among the best;
                         # the closed meta-judge (80% Tiger) is also no longer fitted
                         # on math rows (v6.17). Base verdict: p_q > 0.5 (uncalibrated
                         # 0.5 - only 13 labeled math rows, too few to fit honestly);
                         # sympy still outranks it, Qwen-EN CoT still overrides, and
                         # the Tiger Bengali-CoT fallback is disabled for math.
CLF_EPOCHS, CLF_MAX_LEN, CLF_BATCH, CLF_LR, CLF_MAX_TRAIN = 2, 256, 32, 2e-5, 40000
CLF_CTX_DROP   = 0.0     # v6.21: share of encoder-training pairs with context DROPPED
                         # (teaches closed-book). p_clf is only USED on grounded, so set
                         # 0.0 to train grounded-only. A/B: compare printed grounded AUC.
BLEND3_MANUAL = {"closed": {"p_t": 0.25, "p_q": 0.65, "p_qe": 0.1}}       # v6.22: force blend3 weights per track, e.g.
                         # {"closed": {"p_t": 0.3, "p_q": 0.6, "p_qe": 0.1}}
                         # Weights used VERBATIM (should sum to 1); the decision
                         # THRESHOLD is re-fit by CV on the overridden blend, and the
                         # track is FORCED to blend3 (skips challenger selection).
                         # With val_scores.json attached + USE_VAL_SCORE_CACHE=True,
                         # weight changes re-fit instantly from cached judge scores.
CLF_SAVE_NAME = "clf_finetuned"
USE_VAL_SCORE_CACHE = False # v6.19: OFF while experimenting -> validation always re-scores fresh.
                            # Set True + attach val_scores.json to skip the ~31 min pass-1 scoring
                            # once the config is frozen (fitting/gates always re-run either way).
                            # of pass-1 scoring on reruns; the FITTING (weights/thresholds/
                            # gates) still re-runs live, so bank/tier/threshold changes are
                            # always respected. Guard invalidates on prompt/model/ctx change.  # v6.14: save/reuse the fine-tuned encoder (skips ~18 min + freezes p_clf)
MATH_COT_TOKENS = 256    # v6.2: was 320; batched generation makes CoT ~7x faster
GEN_BATCH       = 8      # prompts per qwen.generate call (auto-halves on OOM)
USE_BENHALLU    = True   # v6.2: embedded BenHalluEval refs + exact-pair tiers + real clf pairs

# Hub resources from the loader notebooks. These are open-weight / public data.
TIGER_ID = "md-nishat-008/TigerLLM-9B-it"
QWEN_ID  = "Qwen/Qwen3.5-9B"
CLF_ID   = "csebuetnlp/banglabert"
SQUAD_URL = "https://huggingface.co/datasets/csebuetnlp/squad_bn/resolve/main/data/squad_bn.tar.bz2"
WIKI_DATASET = ("wikimedia/wikipedia", "20231101.bn")
GRAMMAR_REPOS = {
    "ek_kothay_prokash": "ishtiakmoin/bangla-ek-kothay-prokash",
    "shomash": "ishtiakmoin/bangla-shomash",
    "bagdhara": "ishtiakmoin/bangla-bagdhara",
}
CB_REF_REPOS = {
    # "bcs_qs": removed on request (azminetoushikwasi/bangla-bcs-qs) - noisy
    "bangla_math": "kawchar85/Bangla-Math",
}
# Recommended online additions from the Bangla NLP catalog / challenge discussion.
# Exact repo IDs are used where known; search specs fail softly if HF mirrors are unavailable.
EXTRA_QA_REPOS = {
    "banglarqa": "sartajekram/BanglaRQA",       # answerable + unanswerable Bengali RQA
    "nctb_qa":   "ShihabReza/nctb-qa",          # 12k NCTB textbook QA (Phase-2 source)
}
EXTRA_QA_SEARCH = {
    "banglaquad": {"queries": ["BanglaQuAD", "Bangla QuAD Bengali"], "must": ["banglaquad"]},
}
CB_REF_SEARCH = {
    "bluck": {"queries": ["BLUCK Bengali", "Bangla linguistic cultural knowledge BLUCK"], "must": ["bluck"]},
}
KAGGLE_GRAMMAR_DATASETS = {
    "kaggle_bagdhara": "sakhadib/bagdhara-bangla-idioms-dataset",
}

# Set True if you explicitly want full HF snapshots copied into /kaggle/working first.
# False avoids an extra copy; AutoModel/AutoTokenizer still download from the Hub cache on demand.
DOWNLOAD_MODEL_SNAPSHOTS = False
ALLOW_QWEN_ON_SINGLE_GPU = False

# grammar dataset column names - leave None to auto-detect, set if the guess is wrong:
GRAMMAR_TERM_COL = None
GRAMMAR_ANS_COL  = None
GRAMMAR_PATH_RE  = r"bagdhara|baghdara|shomash|somash|prokash|idiom|grammar|byakoron|bakko|phrase"

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RESOURCE_DIR = os.path.join(WORK_DIR, "olikbochon_resources")
os.makedirs(RESOURCE_DIR, exist_ok=True)
CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints_v6_online")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if torch.cuda.device_count() < 2 and not ALLOW_QWEN_ON_SINGLE_GPU:
    if USE_ENSEMBLE or USE_XLING_EN or USE_XLING_COT:
        print("single GPU detected -> Qwen judge/xling passes disabled for 16GB VRAM safety")
    USE_ENSEMBLE = False
    USE_XLING_EN = False
    USE_XLING_COT = False

NEED_QWEN = USE_ENSEMBLE or USE_XLING_EN or USE_XLING_COT

def find_file(*names):
    roots = ["/kaggle/input", "/kaggle/working", "Dataset", "."]
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for name in names:
                if name in filenames:
                    return os.path.join(dirpath, name)
    return None

# fixed paths first (see top cell) - find_file fallback keeps local runs working
LABELED_PATH = LABELED_PATH_FIXED if os.path.exists(LABELED_PATH_FIXED) else find_file("dataset samples.json", "dataset_samples.json")
TEST_PATH    = TEST_PATH_FIXED    if os.path.exists(TEST_PATH_FIXED)    else find_file("test set.csv", "test_set.csv")
SAMPLE_PATH  = SAMPLE_PATH_FIXED  if os.path.exists(SAMPLE_PATH_FIXED)  else find_file("sample submission.csv", "sample_submission.csv")
print("labeled :", LABELED_PATH)
print("test    :", TEST_PATH)
print("sample  :", SAMPLE_PATH)
assert LABELED_PATH and TEST_PATH and SAMPLE_PATH, "competition data files not found in /kaggle/input or working directory"

def maybe_snapshot(repo_id, local_name, needed=True):
    if not needed:
        return None
    if not DOWNLOAD_MODEL_SNAPSHOTS:
        return repo_id
    out = os.path.join(RESOURCE_DIR, "models", local_name)
    if os.path.exists(os.path.join(out, "config.json")):
        return out
    assert ONLINE, f"Internet is required to download {repo_id}"
    from huggingface_hub import snapshot_download
    os.makedirs(out, exist_ok=True)
    print(f"downloading snapshot {repo_id} -> {out}")
    snapshot_download(repo_id, local_dir=out, local_dir_use_symlinks=False, resume_download=True)
    return out

def ensure_squad_tarball():
    p = find_file("squad_bn.tar.bz2")
    if p:
        return p
    if not (USE_SQUAD_REFS or USE_CTX_MATCH or USE_CLF):
        return None
    assert ONLINE, "Internet is required to download squad_bn.tar.bz2"
    import urllib.request
    out_dir = os.path.join(RESOURCE_DIR, "refs")
    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, "squad_bn.tar.bz2")
    if not os.path.exists(out):
        print("downloading squad_bn (~8 MB)...")
        urllib.request.urlretrieve(SQUAD_URL, out)
    print("squad_bn tarball:", out, f"{os.path.getsize(out)/1e6:.1f} MB")
    return out

def ensure_bnwiki():
    # Reuse any attached/local saved dataset first, otherwise reproduce wikibn.ipynb online.
    for root in ["/kaggle/input", "/kaggle/working", "."]:
        if not os.path.isdir(root):
            continue
        for hit in glob.glob(os.path.join(root, "**", "dataset_info.json"), recursive=True):
            d = os.path.dirname(hit)
            if "bnwiki" in d.lower() or "wikipedia" in d.lower():
                return d
    if not USE_RETRIEVAL:
        return None
    out = os.path.join(RESOURCE_DIR, "bnwiki")
    if os.path.exists(os.path.join(out, "dataset_info.json")):
        return out
    assert ONLINE, "Internet is required to download Bengali Wikipedia"
    from datasets import load_dataset
    print('downloading Bengali Wikipedia:', WIKI_DATASET)
    wiki = load_dataset(WIKI_DATASET[0], WIKI_DATASET[1], split="train")
    print("bnwiki articles:", len(wiki))
    wiki.save_to_disk(out)
    return out

MODEL_PATH  = maybe_snapshot(TIGER_ID, "tigerllm-9b-it", needed=True)
QWEN_PATH   = maybe_snapshot(QWEN_ID,  "qwen3-8b", needed=NEED_QWEN)
QA_PATH     = None   # QA verifier intentionally disabled unless you set USE_QA_VERIFY and provide/choose a QA model
CLF_PATH    = maybe_snapshot(CLF_ID, "banglabert", needed=USE_CLF)
SQUAD_TARBALL_PATH = ensure_squad_tarball() if (USE_SQUAD_REFS or USE_CTX_MATCH or USE_CLF) else None
BNWIKI_PATH = ensure_bnwiki() if USE_RETRIEVAL else None

print("tigerllm:", MODEL_PATH)
print("qwen    :", QWEN_PATH)
print("encoder :", CLF_PATH)
print("squad   :", SQUAD_TARBALL_PATH)
print("bnwiki  :", BNWIKI_PATH)

if USE_QA_VERIFY and not QA_PATH:
    USE_QA_VERIFY = False; print("QA model missing -> graft 2 disabled")
if USE_RETRIEVAL and not BNWIKI_PATH:
    USE_RETRIEVAL = False; print("bnwiki unavailable -> retrieval disabled")
if USE_CLF and not CLF_PATH:
    USE_CLF = False; print("encoder unavailable -> classifier member disabled")
if NEED_QWEN and not QWEN_PATH:
    USE_ENSEMBLE = USE_XLING_EN = USE_XLING_COT = False
    NEED_QWEN = False
    print("qwen unavailable -> Qwen-dependent passes disabled")


def maybe_hf_login():
    """Use HF_TOKEN when available. Some grammar datasets may require it; public datasets work without it."""
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
            os.environ["HF_TOKEN"] = token
        except Exception:
            token = None
    if token:
        try:
            from huggingface_hub import login
            login(token=token, add_to_git_credential=False)
            print("HF login: token available")
        except Exception as e:
            print("HF login skipped:", e)
    else:
        print("HF login: no token found; using public/anonymous access")
    return token

def ensure_grammar_refs():
    """Reproduce grammar.ipynb: download grammar datasets and save csv/parquet/jsonl outputs."""
    if not USE_GRAMMAR:
        return None
    base = os.path.join(WORK_DIR, "bangla_grammar_datasets")
    expected = [os.path.join(base, name, f"{name}.parquet") for name in GRAMMAR_REPOS]
    if all(os.path.exists(x) for x in expected):
        print("grammar refs already present:", base)
        return base
    if not ONLINE:
        print("grammar refs unavailable: no internet")
        return None
    maybe_hf_login()
    from datasets import load_dataset
    os.makedirs(base, exist_ok=True)
    for name, repo_id in GRAMMAR_REPOS.items():
        out_dir = os.path.join(base, name)
        os.makedirs(out_dir, exist_ok=True)
        try:
            ds = load_dataset(repo_id)
            train = ds["train"] if "train" in ds else next(iter(ds.values()))
            train.to_csv(os.path.join(out_dir, f"{name}.csv"), index=False)
            train.to_parquet(os.path.join(out_dir, f"{name}.parquet"))
            try:
                train.to_json(os.path.join(out_dir, f"{name}.jsonl"), orient="records", lines=True, force_ascii=False)
            except TypeError:
                train.to_json(os.path.join(out_dir, f"{name}.jsonl"), orient="records", lines=True)
            print("grammar", name, "saved", train.num_rows, "rows ->", out_dir)
        except Exception as e:
            print("grammar", repo_id, "load failed:", e)
    return base

def ensure_kaggle_grammar_refs():
    """Download Kaggle idiom datasets into working so the grammar scanner can use them."""
    if not (USE_GRAMMAR and USE_KAGGLE_BAGDHARA and KAGGLE_GRAMMAR_DATASETS):
        return []
    base = os.path.join(WORK_DIR, "kaggle_grammar_refs")
    made = []
    os.makedirs(base, exist_ok=True)
    if not ONLINE:
        print("Kaggle grammar refs unavailable: no internet")
        return made
    try:
        import kagglehub
    except Exception as e:
        print("kagglehub unavailable; Kaggle Bagdhara skipped:", e)
        return made
    for name, slug in KAGGLE_GRAMMAR_DATASETS.items():
        dest = os.path.join(base, name)
        if os.path.isdir(dest) and any(glob.glob(os.path.join(dest, "**", "*"), recursive=True)):
            print("Kaggle grammar", name, "already present:", dest)
            made.append(dest)
            continue
        try:
            raw = kagglehub.dataset_download(slug)
            if os.path.isdir(dest):
                shutil.rmtree(dest)
            shutil.copytree(raw, dest)
            print("Kaggle grammar", slug, "->", dest)
            made.append(dest)
        except Exception as e:
            print("Kaggle grammar", slug, "download failed:", e)
    return made

def _dataset_to_frame(ds):
    frames = []
    for split in ds.keys():
        try:
            frames.append(ds[split].to_pandas())
        except Exception as e:
            print("  split", split, "to_pandas failed:", e)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _flatten_qa_json(obj):
    """v6.3 fix: SQuAD / BanglaRQA style nested json -> flat QA rows the
    reference mapper understands (question / answers / context / is_answerable)."""
    if not (isinstance(obj, dict) and isinstance(obj.get("data"), list)):
        return None
    rows = []
    for art in obj["data"]:
        if not isinstance(art, dict):
            continue
        for para in (art.get("paragraphs") or [art]):
            if not isinstance(para, dict):
                continue
            ctx = str(para.get("context", ""))
            for qa in para.get("qas", []) or []:
                if not isinstance(qa, dict):
                    continue
                q = qa.get("question_text") or qa.get("question") or ""
                ans, texts = qa.get("answers"), []
                if isinstance(ans, dict):
                    texts = [t for t in (ans.get("answer_text") or ans.get("text") or []) if t]
                elif isinstance(ans, list):
                    for a in ans:
                        t = (a.get("answer_text") or a.get("text")) if isinstance(a, dict) else a
                        if t:
                            texts.append(t)
                rows.append({"question": str(q),
                             "answers": json.dumps([str(t) for t in texts], ensure_ascii=False),
                             "context": ctx,
                             "is_answerable": str(qa.get("is_answerable", ""))})
    return pd.DataFrame(rows) if rows else None

def _save_hf_dataset(repo_id, out_path):
    try:
        from datasets import load_dataset
        ds = load_dataset(repo_id)
        df = _dataset_to_frame(ds)
    except Exception as e:
        # v6.3 fix: metadata-version mismatches (e.g. SplitInfo 'description')
        # break load_dataset on older repos -> read the raw data files directly
        print(f"{repo_id}: load_dataset failed ({e}) -> raw-file fallback")
        from huggingface_hub import snapshot_download
        raw = snapshot_download(repo_id, repo_type="dataset")
        frames = []
        for p in sorted(glob.glob(os.path.join(raw, "**", "*"), recursive=True)):
            base = os.path.basename(p).lower()
            try:
                if p.endswith(".parquet"):
                    frames.append(pd.read_parquet(p))
                elif p.endswith(".csv"):
                    frames.append(pd.read_csv(p))
                elif p.endswith(".tsv"):
                    frames.append(pd.read_csv(p, sep="\t"))
                elif p.endswith(".jsonl"):
                    frames.append(pd.read_json(p, lines=True))
                elif p.endswith(".json") and "dataset_info" not in base:
                    with open(p, encoding="utf-8") as f:
                        obj = json.load(f)
                    flat = _flatten_qa_json(obj)
                    frames.append(flat if flat is not None
                                  else pd.DataFrame(obj if isinstance(obj, list) else [obj]))
            except Exception:
                pass
        if not frames:
            df = pd.DataFrame()
        else:
            try:
                df = pd.concat(frames, ignore_index=True)
            except Exception:
                df = max(frames, key=len)
        # parquet-safe: stringify non-scalar object values
        for c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].map(lambda v: v if isinstance(v, str) or pd.isna(v)
                                  else json.dumps(v, ensure_ascii=False, default=str))
    if df.empty:
        raise ValueError("empty dataset")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_parquet(out_path)
    print("saved", repo_id, df.shape, "->", out_path)
    return out_path

def _hf_search_ids(queries, must=(), limit=12):
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        seen = []
        for q in queries:
            for item in api.list_datasets(search=q, limit=limit):
                rid = getattr(item, "id", None) or getattr(item, "repo_id", None)
                if not rid or rid in seen:
                    continue
                low = rid.lower()
                if all(m.lower() in low for m in must):
                    seen.append(rid)
        return seen
    except Exception as e:
        print("HF dataset search failed:", e)
        return []

def ensure_extra_qa_refs():
    """Download recommended Bengali QA references for exact prompt-answer matching."""
    if not USE_EXTRA_QA_REFS:
        return []
    base = os.path.join(WORK_DIR, "extra_qa_refs")
    os.makedirs(base, exist_ok=True)
    files = []
    if not ONLINE:
        print("extra QA refs unavailable: no internet")
        return files
    maybe_hf_login()
    repo_map = dict(EXTRA_QA_REPOS)
    for name, spec in EXTRA_QA_SEARCH.items():
        out = os.path.join(base, f"{name}.parquet")
        if os.path.exists(out):
            files.append(out); continue
        hits = _hf_search_ids(spec.get("queries", []), spec.get("must", []))
        if hits:
            repo_map[name] = hits[0]
            print("HF search", name, "->", hits[0])
        else:
            print("HF search", name, "found no loadable candidate")
    for name, repo_id in repo_map.items():
        out = os.path.join(base, f"{name}.parquet")
        if os.path.exists(out):
            files.append(out); continue
        try:
            files.append(_save_hf_dataset(repo_id, out))
        except Exception as e:
            print("extra QA", repo_id, "load failed:", e)
    return files

def ensure_optional_cb_refs():
    """Download optional closed-book culture/linguistic benchmarks such as BLUCK."""
    if not USE_CB_REFS:
        return []
    base = os.path.join(WORK_DIR, "cb_refs_optional")
    os.makedirs(base, exist_ok=True)
    files = []
    if not ONLINE:
        print("optional cb refs unavailable: no internet")
        return files
    maybe_hf_login()
    for name, spec in CB_REF_SEARCH.items():
        out = os.path.join(base, f"{name}.parquet")
        if os.path.exists(out):
            files.append(out); continue
        hits = _hf_search_ids(spec.get("queries", []), spec.get("must", []))
        if not hits:
            print("optional cb", name, "not found on HF search")
            continue
        try:
            print("optional cb", name, "->", hits[0])
            files.append(_save_hf_dataset(hits[0], out))
        except Exception as e:
            print("optional cb", hits[0], "load failed:", e)
    return files

def ensure_cb_refs():
    """Reproduce cb-refs.ipynb: download BCS and Bangla-Math references to parquet."""
    if not USE_CB_REFS:
        return None
    base = os.path.join(WORK_DIR, "cb_refs")
    expected = [os.path.join(base, f"{stem}.parquet") for stem in CB_REF_REPOS]
    if all(os.path.exists(x) for x in expected):
        print("closed-book refs already present:", base)
        return base
    if not ONLINE:
        print("closed-book refs unavailable: no internet")
        return None
    maybe_hf_login()
    from datasets import load_dataset
    os.makedirs(base, exist_ok=True)
    for stem, repo in CB_REF_REPOS.items():
        try:
            ds = load_dataset(repo)
            df = pd.concat([ds[split].to_pandas() for split in ds.keys()], ignore_index=True)
            out = os.path.join(base, f"{stem}.parquet")
            df.to_parquet(out)
            print("cb-ref", stem, df.shape, "| columns:", list(df.columns))
        except Exception as e:
            print(repo, "load_dataset failed:", e, "-> raw snapshot fallback")
            try:
                from huggingface_hub import snapshot_download
                snapshot_download(repo, repo_type="dataset",
                                  local_dir=os.path.join(base, f"{stem}_raw"),
                                  local_dir_use_symlinks=False, resume_download=True)
            except Exception as e2:
                print(repo, "raw snapshot fallback failed:", e2)
    return base

# Build optional references before scanning for files. These are no longer attachments.
GRAMMAR_BASE_DIR = ensure_grammar_refs() if USE_GRAMMAR else None
KAGGLE_GRAMMAR_DIRS = ensure_kaggle_grammar_refs() if USE_GRAMMAR else []
CB_REF_BASE_DIR = ensure_cb_refs() if USE_CB_REFS else None
OPTIONAL_CB_FILES = ensure_optional_cb_refs() if USE_CB_REFS else []
EXTRA_QA_FILES = ensure_extra_qa_refs() if USE_EXTRA_QA_REFS else []

# ---- v6.5: knowledge datasets (manually-uploaded local CSVs) -----------------
# NCTB textbook QA (passage+Q+A) and a Bangla knowledge MCQ set. Measured exact
# test-overlap is small (NCTB 0, bqad ~8 rows), so these help mainly as: closed-book
# reference coverage, REAL classifier pairs in the C1/C2 cultural domain (MCQ
# distractors are human-written plausible wrong answers = natural hallucinations),
# retrieval passages, and Phase-2 generalization. Attach the CSVs as a Kaggle dataset;
# each piece disables itself if its file is absent.
USE_KNOWLEDGE_DS = True
USE_BCS = True          # BCS answer-key MCQs (Bangla language & literature)
USE_SATT = True         # v6.13: SATT GK MCQ bank (satt_gk_mcq.csv, attached as a Kaggle dataset)
# v6.11: MCQ pairs (bqad + BCS) as CLASSIFIER training data measurably hurt it. Two runs:
#   KD=11,263 pairs -> labeled AUC all .822 | grounded .971 | closed .584
#   KD=19,129 pairs -> labeled AUC all .803 | grounded .954 | closed .558  (p_clf closed coef -0.13)
# More closed-book MCQ noise degraded BOTH tracks and crowded the same-shape synthetic
# squad pairs out of the cap (15,562 -> 7,696). Their REFERENCES and exact-match TIERS are
# valuable and stay on; only the classifier pairs are dropped. Set True to A/B.
KD_CLF_MCQ = False
BCS_URL = ("https://huggingface.co/datasets/azminetoushikwasi/bangla-bcs-qs/"
           "resolve/main/cleaned_bangla-data_20240901_001128.json")
KD_CLF_ROWS, KD_PASSAGE_FILE, BCS_RAW = [], None, []
SATT_RAW = []   # v6.13: (question, gold_option_TEXT, [distractor_TEXTs])

_ALLNONE = re.compile(r"উপরের\s*সব|সবক[টয়]ি|সবগুলো|সব[গক]ুলি|কোনটিই\s*ন[য়া]|কোনোটিই\s*ন[য়া]|উপরের\s*কোনটিই|all\s+of\s+the\s+above|none\s+of\s+the\s+above", re.I)

def _samestr(a, b):
    return re.sub(r"\s+", " ", str(a)).strip().casefold() == re.sub(r"\s+", " ", str(b)).strip().casefold()

def ensure_knowledge_datasets():
    global KD_PASSAGE_FILE
    if not USE_KNOWLEDGE_DS:
        return []
    ref_files, passages = [], []
    # -- Bangla knowledge MCQ: question + correct option (+ distractors as negatives) --
    bq = find_file("bqad2025.csv")
    if bq:
        try:
            d = pd.read_csv(bq)
            rr, npair = [], 0
            for _, r in d.iterrows():
                q = str(r.get("Question", "")).strip()
                L = str(r.get("Answer", "")).strip().upper()
                gold = str(r.get(L, "")).strip() if L in ("A", "B", "C", "D") else ""
                if not (q and gold and gold.lower() != "nan"):
                    continue
                rr.append({"question": q, "answers": gold})
                if KD_CLF_MCQ:
                    KD_CLF_ROWS.append(("", q, gold, 1)); npair += 1
                    # ONE distractor per MCQ keeps the classifier pool balanced (a full
                    # A/B/C/D expansion is 3:1 hallucinated and biases the member)
                    for opt_l in ("A", "B", "C", "D"):
                        opt = str(r.get(opt_l, "")).strip()
                        if opt and opt.lower() != "nan" and not _samestr(opt, gold):
                            KD_CLF_ROWS.append(("", q, opt, 0)); npair += 1
                            break
            out = os.path.join(RESOURCE_DIR, "bqad_refs.parquet")
            pd.DataFrame(rr).to_parquet(out); ref_files.append(out)
            print(f"knowledge MCQ (bqad): {len(rr)} refs + {npair} classifier pairs")
        except Exception as e:
            print("bqad2025 load failed:", e)
    # -- NCTB textbook QA (local): passage + question + answer --
    nc = find_file("Textbook Dataset from NCTB.csv", "nctb.csv", "nctb_textbook.csv")
    if nc:
        try:
            d = pd.read_csv(nc)
            qcol = next((c for c in d.columns if re.search(r"question", str(c), re.I)), None)
            acol = next((c for c in d.columns if re.search(r"ans", str(c), re.I)), None)
            # v6.9: 'AnsText' also matches the passage regex via `text`; exclude the
            # already-claimed columns so column ORDER can never decide the mapping
            pcol = next((c for c in d.columns
                         if c not in (qcol, acol)
                         and re.search(r"passage|context|evidence|text", str(c), re.I)), None)
            rr = []
            if qcol and acol:
                for _, r in d.iterrows():
                    q, a = str(r[qcol]).strip(), str(r[acol]).strip()
                    p = str(r[pcol]).strip() if pcol else ""
                    if q and a and a.lower() != "nan":
                        rr.append({"question": q, "answers": a})
                        KD_CLF_ROWS.append((p[:1200] if (p and p.lower() != "nan") else "", q, a, 1))
                    if p and p.lower() != "nan":
                        passages.append((q[:60], p))
                out = os.path.join(RESOURCE_DIR, "nctb_local_refs.parquet")
                pd.DataFrame(rr).to_parquet(out); ref_files.append(out)
                print(f"NCTB local: {len(rr)} QA refs + {len(passages)} passages")
        except Exception as e:
            print("NCTB local load failed:", e)
    # -- BCS answer-key MCQs: `answer` is the 1-BASED OPTION INDEX, not the text.
    #    Mapping it naively (the old cb_refs path) put "1".."4" in the reference pool,
    #    which is why this source was junk. Resolve index -> option text here.
    if USE_BCS:
        bp = find_file("cleaned_bangla-data_20240901_001128.json", "bangla-bcs-qs.json")
        if not bp and ONLINE:
            try:
                import urllib.request
                # name must NOT match find_cb_files() - the raw file has an `options` column
                bp = os.path.join(RESOURCE_DIR, "bcs_answerkey_raw.json")
                if not os.path.exists(bp):
                    urllib.request.urlretrieve(BCS_URL, bp)
            except Exception as e:
                print("BCS download failed:", e); bp = None
        if bp:
            try:
                with open(bp, encoding="utf-8") as f:
                    recs = json.load(f)
                rr, bad = [], 0
                for r in recs:
                    q = str(r.get("question", "")).strip()
                    opts = [str(o).strip() for o in (r.get("options") or []) if str(o).strip()]
                    try:
                        idx = int(str(r.get("answer", "")).strip()) - 1
                    except Exception:
                        bad += 1; continue
                    if not (q and opts and 0 <= idx < len(opts)):
                        bad += 1; continue
                    gold = opts[idx]
                    dist = [o for j, o in enumerate(opts) if j != idx and not _samestr(o, gold)]
                    BCS_RAW.append((q, gold, dist))
                    rr.append({"question": q, "answers": gold})
                    if KD_CLF_MCQ:
                        KD_CLF_ROWS.append(("", q, gold, 1))
                        if dist:
                            KD_CLF_ROWS.append(("", q, dist[0], 0))   # 1 distractor keeps balance
                out = os.path.join(RESOURCE_DIR, "bcs_refs.parquet")
                pd.DataFrame(rr).to_parquet(out); ref_files.append(out)
                print(f"BCS answer-key: {len(BCS_RAW)} questions resolved "
                      f"(index->option text), {bad} unparseable")
            except Exception as e:
                print("BCS load failed:", e)

    # -- v6.13: SATT GK MCQ bank. `Answer` is the option LETTER (A/B/C/D), NOT the
    #    answer text and NOT a 1-based index. Resolve letter -> option TEXT here, the
    #    same discipline Layer 1g applies to the BCS bank (whose `answer` is an index;
    #    mapping it naively once put "1".."4" into the reference pool). Distractors are
    #    the OTHER options' TEXT. Classifier pairs are deliberately NOT built from these
    #    (v6.11 measured MCQ distractor pairs as harmful: closed AUC .584 -> .558).
    for _bank_name, _bank_file in (("SATT GK", "satt_gk_mcq.csv"), ("BCS GK", "bcs_gk_mcq.csv"),
                                   ("BCS Bangla", "bcs_bangla_mcq.csv"),
                                   ("BCS Geography", "bcs_geo_mcq.csv"),
                                   ("BCS Science", "bcs_science_mcq.csv")):
        sp = find_file(_bank_file) if USE_SATT else None
        if not sp:
            print(f"{_bank_name}: {_bank_file} not found under /kaggle/input - skipped")
        else:
            try:
                d = pd.read_csv(sp)
                _n0 = len(SATT_RAW)
                need = {"Question", "A", "B", "C", "D", "Answer"}
                missing = need - set(d.columns)
                if missing:
                    raise ValueError(f"missing columns {sorted(missing)}")
                rr, bad = [], 0
                for _, r in d.iterrows():
                    q = str(r.get("Question", "")).strip()
                    L = str(r.get("Answer", "")).strip().upper()
                    if L not in ("A", "B", "C", "D"):
                        bad += 1; continue
                    gold = str(r.get(L, "")).strip()
                    if not (q and gold and gold.lower() != "nan"):
                        bad += 1; continue
                    opts = [str(r.get(c, "")).strip() for c in ("A", "B", "C", "D")]
                    opts = [o for o in opts if o and o.lower() != "nan"]
                    if len(opts) < 2:
                        bad += 1; continue
                    # v6.13 guard: a gold of "all/none of the above" breaks BOTH tiers.
                    # If gold == "সবগুলো"/"all of the above", every OTHER option is TRUE,
                    # so distractor->0 would manufacture false hallucinations (observed on
                    # the Bangla bank: "বাগযন্ত্রের অংশ কোনটি?" gold "উপরের সবকটি" -> the
                    # true answer স্বরযন্ত্র gets called a distractor). If gold ==
                    # "কোনটিই নয়", the gold TEXT is not an answer and is junk in GOLD.
                    # Either way the row is unusable -> drop it.
                    if _ALLNONE.search(gold):
                        bad += 1; continue
                    dist = [o for o in opts if not _samestr(o, gold)]
                    SATT_RAW.append((q, gold, dist))
                    rr.append({"question": q, "answers": gold})
                out = os.path.join(RESOURCE_DIR, f"{_bank_file.replace('.csv','')}_refs.parquet")
                pd.DataFrame(rr).to_parquet(out); ref_files.append(out)
                _added = len(SATT_RAW) - _n0
                print(f"{_bank_name}: {_added} questions resolved (letter->option TEXT), {bad} skipped"
                      + (f" | e.g. gold={SATT_RAW[_n0][1]!r} dist={SATT_RAW[_n0][2][:2]}" if _added else ""))
            except Exception as e:
                print(f"{_bank_name} load failed:", e)

    if passages:
        KD_PASSAGE_FILE = os.path.join(RESOURCE_DIR, "nctb_passages.parquet")
        pd.DataFrame(passages, columns=["title", "text"]).drop_duplicates("text").to_parquet(KD_PASSAGE_FILE)
    return ref_files

KD_REF_FILES = ensure_knowledge_datasets() if USE_KNOWLEDGE_DS else []
# v6.13: hand SATT's (question, gold, distractors) to Layer 1g - it already builds
# exact gold/distractor pairs, quoted-term tiers, drops answer-key conflicts across
# exams, and reports labeled precision. No parallel plumbing needed.
if SATT_RAW:
    BCS_RAW.extend(SATT_RAW)
    print(f"Layer 1g pool: +{len(SATT_RAW)} SATT questions (BCS_RAW now {len(BCS_RAW)})")
EXTRA_QA_FILES = list(EXTRA_QA_FILES) + list(KD_REF_FILES)

def find_grammar_files():
    out, pat = [], re.compile(GRAMMAR_PATH_RE, re.I)
    for root in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            if "competitions" in dirpath:
                continue
            for fn in files:
                if fn.lower().endswith((".csv", ".tsv", ".json", ".jsonl", ".parquet")):
                    full = os.path.join(dirpath, fn)
                    if pat.search(full):
                        out.append(full)
    return out

GRAMMAR_FILES = find_grammar_files() if USE_GRAMMAR else []

def merge_itemized_json_dirs(files, min_files=50):
    """v6.3 fix: Kaggle idiom corpora ship as thousands of one-record .json files
    (e.g. sakhadib/bagdhara-bangla-idioms-dataset). pd.read_json chokes on scalar
    dicts and the per-file loop would spam/skip - merge each such directory into
    one parquet and hand that to the grammar layer instead."""
    from collections import defaultdict
    by_dir = defaultdict(list)
    for p in files:
        if p.lower().endswith(".json"):
            by_dir[os.path.dirname(p)].append(p)
    big_dirs = {d for d, fs in by_dir.items() if len(fs) >= min_files}
    out_files = [p for p in files if os.path.dirname(p) not in big_dirs]
    merged_dir = os.path.join(WORK_DIR, "merged_grammar")
    for d in sorted(big_dirs):
        os.makedirs(merged_dir, exist_ok=True)
        out = os.path.join(merged_dir,
                           re.sub(r"\W+", "_", os.path.basename(d)) + "_grammar_merged.parquet")
        if os.path.exists(out):
            print("merged grammar file already present:", os.path.basename(out))
        else:
            rows, bad = [], 0
            for p in by_dir[d]:
                try:
                    with open(p, encoding="utf-8") as f:
                        obj = json.load(f)
                    for it in (obj if isinstance(obj, list) else [obj]):
                        if isinstance(it, dict):
                            rows.append({k: (v if isinstance(v, (str, int, float, bool)) or v is None
                                             else json.dumps(v, ensure_ascii=False))
                                         for k, v in it.items()})
                except Exception:
                    bad += 1
            if not rows:
                print(f"merge {d}: no readable records ({bad} bad files)")
                continue
            pd.DataFrame(rows).to_parquet(out)
            print(f"merged {len(rows)} records from {len(by_dir[d])} json files "
                  f"({bad} unreadable) -> {os.path.basename(out)}")
        out_files.append(out)
    return sorted(set(out_files))

if USE_GRAMMAR and GRAMMAR_FILES:
    GRAMMAR_FILES = merge_itemized_json_dirs(GRAMMAR_FILES)
print(f"grammar files: {len(GRAMMAR_FILES)}"
      + (" | e.g. " + ", ".join(os.path.basename(p) for p in GRAMMAR_FILES[:5])
         if GRAMMAR_FILES else " (none)"))
if USE_GRAMMAR and not GRAMMAR_FILES:
    USE_GRAMMAR = False
    print("-> grammar layer disabled (no local/public grammar file found)")

def find_cb_files():
    out, pat = [], re.compile(r"cb_refs|bcs[-_ ]?qs|bcs[-_ ]?question|bangla[-_ ]?math", re.I)
    for root in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            if "competitions" in dirpath:
                continue
            for fn in files:
                if fn.lower().endswith((".csv", ".json", ".jsonl", ".parquet")):
                    full = os.path.join(dirpath, fn)
                    if pat.search(full):
                        out.append(full)
    return out

CB_FILES = find_cb_files() if USE_CB_REFS else []
CB_FILES = sorted(set(CB_FILES + list(globals().get("OPTIONAL_CB_FILES", []))))
print("cb-ref files:", CB_FILES or "none")
if USE_CB_REFS and not CB_FILES:
    USE_CB_REFS = False
    print("-> closed-book reference layer disabled (no local/public cb-ref file found)")


# ---- v6.3: Phase-2 source-base expansion + starter-notebook resources ---------
USE_NEWS_CORPUS = True    # XL-Sum Bengali (BBC Bangla) -> retrieval index (newspapers domain)
USE_RETR_SEARCH = True    # runtime HF search: Banglapedia / bdlaws / NCTB corpora (fail-soft)
RETRIEVAL_SEARCH = {
    "banglapedia": {"queries": ["banglapedia"], "must": ["banglapedia"]},
    # v6.12: known repo id first (47,429 sections of bdlaws.minlaw.gov.bd - an
    # announced Phase-2 source the HF search missed); search stays as fallback.
    "bdlaws":      {"repo": "anisafifi/bd-laws",
                    "queries": ["bdlaws bangladesh law", "bangladesh legislation corpus"], "must": ["bdlaw"]},
    "nctb":        {"queries": ["NCTB textbook bangla", "nctb bengali"], "must": ["nctb"]},
}

def ensure_news_corpus():
    if not (USE_NEWS_CORPUS and USE_RETRIEVAL):
        return None
    out = os.path.join(RESOURCE_DIR, "xlsum_bn.parquet")
    if os.path.exists(out):
        print("news corpus already present:", out)
        return out
    if not ONLINE:
        print("news corpus unavailable: no internet")
        return None
    try:
        # v6.3 fix: xlsum is a script-based repo (unsupported by datasets>=4) ->
        # pull the raw language tarball and parse the jsonl splits directly
        import tarfile, urllib.request
        tb = os.path.join(RESOURCE_DIR, "xlsum_bn.tar.bz2")
        if not os.path.exists(tb):
            urllib.request.urlretrieve(
                "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/"
                "data/bengali_XLSum_v2.0.tar.bz2", tb)
        rows = []
        with tarfile.open(tb, "r:bz2") as tf:
            for name in tf.getnames():
                if not name.endswith(".jsonl"):
                    continue
                with tf.extractfile(name) as fh:
                    for line in fh:
                        try:
                            o = json.loads(line)
                            if o.get("title") and o.get("text"):
                                rows.append((o["title"], o["text"]))
                        except Exception:
                            pass
        df_n = pd.DataFrame(rows, columns=["title", "text"])
        df_n.to_parquet(out)
        print(f"news corpus: {len(df_n)} BBC Bangla articles -> {out}")
        return out
    except Exception as e:
        print("news corpus (xlsum) failed:", e)
        return None

def ensure_retrieval_extras():
    if not (USE_RETR_SEARCH and USE_RETRIEVAL):
        return []
    base = os.path.join(RESOURCE_DIR, "retrieval_extras")
    os.makedirs(base, exist_ok=True)
    files = []
    if not ONLINE:
        return files
    for name, spec in RETRIEVAL_SEARCH.items():
        out = os.path.join(base, f"{name}.parquet")
        if os.path.exists(out):
            files.append(out); continue
        hits = ([spec["repo"]] if spec.get("repo") else []) \
               + _hf_search_ids(spec["queries"], spec["must"])
        if not hits:
            print("retrieval extra", name, "not found on HF search")
            continue
        for hid in hits:
            try:
                print("retrieval extra", name, "->", hid)
                files.append(_save_hf_dataset(hid, out))
                break
            except Exception as e:
                print("retrieval extra", hid, "failed:", e)
    return files

NEWS_CORPUS_PATH      = ensure_news_corpus()
RETRIEVAL_EXTRA_FILES = ensure_retrieval_extras()
# v6.5: NCTB textbook passages join the retrieval corpus (defined above; appended
# here because RETRIEVAL_EXTRA_FILES only exists from this point on)
if KD_PASSAGE_FILE and USE_RETRIEVAL:
    RETRIEVAL_EXTRA_FILES = list(RETRIEVAL_EXTRA_FILES) + [KD_PASSAGE_FILE]
    print(f"retrieval corpus += NCTB passages ({os.path.basename(KD_PASSAGE_FILE)})")

# BenHalluEval files come from the attached `benhallu-data` Kaggle dataset
# (zip the folder and attach it; files are found by name anywhere under /kaggle/input)
BH_QA4000_PATH = find_file("benhallu_qa_4000.csv", "qa_4000.csv")
BH_QADS_PATH   = find_file("benhallu_qa_dataset_2509.csv", "banglahallueval_qa_dataset.csv")
BH_GT1000_PATH = find_file("benhallu_qa_gt_1000.csv", "qa_gt_1000.csv")
print("bh qa_4000 :", BH_QA4000_PATH)
print("bh qa_2509 :", BH_QADS_PATH)
print("bh gt_1000 :", BH_GT1000_PATH, "(optional)")
if USE_BENHALLU and not (BH_QA4000_PATH and BH_QADS_PATH):
    USE_BENHALLU = False
    print("=" * 70)
    print("!! BenHalluEval layer DISABLED - benhallu_qa_4000.csv / "
          "benhallu_qa_dataset_2509.csv NOT found under /kaggle/input.")
    print("!! ~900 test rows (36%) will be judged instead of resolved exactly.")
    print("!! Attach the zipped benhallu-data folder as a Kaggle dataset.")
    print("=" * 70)

labeled : /kaggle/input/competitions/bengali-hallucination/dataset samples.json
test    : /kaggle/input/competitions/bengali-hallucination/test set.csv
sample  : /kaggle/input/competitions/bengali-hallucination/sample submission.csv
downloading squad_bn (~8 MB)...
squad_bn tarball: /kaggle/working/olikbochon_resources/refs/squad_bn.tar.bz2 8.4 MB
downloading Bengali Wikipedia: ('wikimedia/wikipedia', '20231101.bn')


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.bn/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  192MB            

20231101.bn/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

20231101.bn/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  151MB            

...

dataset_infos.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

bd_laws_all.jsonl: reconstructing file:   0%|          |  0.00B / 82.3MB            

bd_laws_all.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

saved anisafifi/bd-laws (47429, 7) -> /kaggle/working/olikbochon_resources/retrieval_extras/bdlaws.parquet
retrieval extra nctb not found on HF search
bh qa_4000 : /kaggle/input/datasets/zarifmahir/dataset1/benhallu_qa_4000.csv
bh qa_2509 : /kaggle/input/datasets/zarifmahir/dataset1/benhallu_qa_dataset_2509.csv
bh gt_1000 : /kaggle/input/datasets/zarifmahir/dataset1/benhallu_qa_gt_1000.csv (optional)


In [5]:
NO_CONTEXT_VALUES = {"", "nan", "NaN", "[NULL]", "None"}

def nfc(s):
    return unicodedata.normalize("NFC", str(s))

def clean_context(value):
    if pd.isna(value) or str(value).strip() in NO_CONTEXT_VALUES:
        return ""
    return str(value).strip()

with open(LABELED_PATH, encoding="utf-8") as f:
    labeled = pd.DataFrame(json.load(f))
test   = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

for df in (labeled, test):
    df["context"]     = df["context"].map(clean_context).map(nfc)
    df["prompt_bn"]   = df["prompt_bn"].astype(str).str.strip().map(nfc)
    df["response_bn"] = df["response_bn"].astype(str).str.strip().map(nfc)
    df["grounded"]    = df["context"] != ""

print(f"labeled: {len(labeled)} rows | grounded {int(labeled.grounded.sum())} / closed-book {int((~labeled.grounded).sum())}")
print(f"test   : {len(test)} rows | grounded {int(test.grounded.sum())} / closed-book {int((~test.grounded).sum())}")
print("labeled label balance:", labeled.label.value_counts().to_dict())


labeled: 299 rows | grounded 130 / closed-book 169
test   : 2516 rows | grounded 1361 / closed-book 1155
labeled label balance: {1: 163, 0: 136}


In [6]:
def normalize(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[\u2010-\u2015\-\u2013\u2014]+", "-", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.rstrip("-\u2014 ?\u0964").strip()

for df in (labeled, test):
    df["p_norm"] = df.prompt_bn.map(normalize)
    df["r_norm"] = df.response_bn.map(normalize)

pair_labels = (labeled.groupby(["p_norm", "r_norm"]).label
                      .agg(lambda s: int(s.iloc[0]) if s.nunique() == 1 else -1)
                      .to_dict())

hits = pd.Series([pair_labels.get((p, r)) for p, r in zip(test.p_norm, test.r_norm)],
                 index=test.index, dtype="float")
test["lookup_label"] = hits.where(hits.isin([0.0, 1.0]))

n_prompt_match = test.p_norm.isin(set(labeled.p_norm)).sum()
print(f"prompt matches labeled set : {n_prompt_match} rows")
print(f"exact (prompt,response) label transfers: {int(test.lookup_label.notna().sum())} rows")


prompt matches labeled set : 190 rows
exact (prompt,response) label transfers: 11 rows


In [7]:
import tarfile, urllib.request

SQUAD_URL = "https://huggingface.co/datasets/csebuetnlp/squad_bn/resolve/main/data/squad_bn.tar.bz2"

def locate_squad_tarball():
    if globals().get("SQUAD_TARBALL_PATH") and os.path.exists(SQUAD_TARBALL_PATH):
        return SQUAD_TARBALL_PATH
    p = find_file("squad_bn.tar.bz2")
    if p:
        return p
    if ONLINE:
        out = os.path.join(RESOURCE_DIR, "refs", "squad_bn.tar.bz2")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        print("downloading squad_bn (~8 MB) from the HF Hub...")
        urllib.request.urlretrieve(SQUAD_URL, out)
        return out
    return None

BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def norm_ans(s):
    """Aggressive normalization for answer comparison (digits, case, punctuation)."""
    s = unicodedata.normalize("NFC", str(s)).translate(BN_DIGITS).casefold()
    s = re.sub(r"[‐-―\-–—]+", "-", s)
    s = re.sub(r"[\"'“”‘’()\[\],;:.!।?]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

SUB_RATIO = 0.70   # v6.9: see below
# v6.10 - `gold appears inside the response`. The match audit measured this sub-rule at
# ~2 correct of 7 fires on the test set (truth taken from BenHalluEval pair labels):
#   gold 'মোহর'  <- response 'ডাক নাম ছিল **আকবরী মোহর**'   (hallucinated)
#   gold 'সি'    <- response 'সি++'                          (C++ is not C)
# vs legitimate:  gold 'খাদিজা' <- 'খাদিজা বিনতে খুওয়াইলিদ'
# No length/ratio guard separates them. It fires 0 times on the labeled set, so there is
# no honest way to tune it. Left ON (unchanged LB behaviour); today all 4 known-bad fires
# are corrected downstream because the BenHallu hall-pair tier overrides containment.
# Set False to A/B it - `match_audit.csv` reports exactly which rows it changes.
ALLOW_GOLD_IN_RESPONSE = True

def contains_gold(response, prompt, golds):
    """Echo-safe containment: a gold answer that already appears in the QUESTION text
    is no evidence (e.g. a person's own name inside a "father's name" question).

    v6.9 - the `response inside gold` direction now needs the response to cover most
    of the gold (SUB_RATIO). It exists to absorb Bengali inflection
    ('রশীদা হামিদ' vs 'রশীদা হামিদের', ratio 0.89), NOT to let a fragment match a long
    sentence gold: with NCTB/BanglaRQA answers in the pool (median 41 chars, 62% over
    30), a bare 'বাংলাদেশ' was matching a 47-char gold it is merely a word of.
    Measured on the labeled set: keeps both legitimate fires, blocks the fragments."""
    r, q = norm_ans(response), norm_ans(prompt)
    if not r:
        return False
    for g0 in golds:
        g = norm_ans(g0)
        if len(g) < 2:
            continue
        if g == r:
            return True
        if ALLOW_GOLD_IN_RESPONSE and g in r and g not in q:
            return True
        if r in g and len(r) >= SUB_RATIO * len(g):
            return True
    return False

GOLD = {}
GOLD_SRC = {}          # v6.10: p_norm -> {source names} for the match audit
tarball = locate_squad_tarball() if USE_SQUAD_REFS else None
if tarball:
    with tarfile.open(tarball, "r:bz2") as tf:
        for split in ("train", "validation", "test"):
            with tf.extractfile(f"squad_bn/{split}.json") as fh:
                data = json.load(fh)
            for art in data["data"]:
                for para in art["paragraphs"]:
                    for qa in para["qas"]:
                        answers = [a["text"] for a in qa.get("answers", []) if a.get("text")]
                        if answers:
                            _k = normalize(qa["question"])
                            GOLD.setdefault(_k, set()).update(answers)
                            GOLD_SRC.setdefault(_k, set()).add("squad_bn")
    print(f"squad_bn: {len(GOLD)} unique answerable questions loaded")
else:
    print("squad_bn unavailable (no tarball and no internet) - reference layer disabled")

QA_Q_HINT = re.compile(r"^(question|question_text|query|prompt|problem|proshno)(_|$)|প্রশ্ন", re.I)
QA_A_HINT = re.compile(r"answers?|answer_text|correct|solution|label|output|উত্তর", re.I)
QA_ANSWERABLE_HINT = re.compile(r"answerable|is_answerable|has_answer|unanswerable", re.I)

def _truthy_answerable(v):
    if pd.isna(v) if not isinstance(v, (list, tuple, dict, np.ndarray)) else False:
        return False
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, float, np.integer, np.floating)):
        return bool(v)
    return str(v).strip().lower() not in {"0", "false", "no", "না", "nan", "none", "unanswerable"}

def _answer_texts(v):
    out = []
    if v is None:
        return out
    if isinstance(v, float) and pd.isna(v):
        return out
    if isinstance(v, str):
        s0 = v.strip()
        if not s0 or s0.lower() in {"nan", "none", "[]", "{}"}:
            return out
        if s0[:1] in "[{":
            try:
                return _answer_texts(json.loads(s0))
            except Exception:
                try:
                    return _answer_texts(ast.literal_eval(s0))
                except Exception:
                    pass
        return [s0]
    if isinstance(v, dict):
        for key in ("answer_text", "text", "answers", "answer", "value", "label", "correct_answer"):
            if key in v:
                out.extend(_answer_texts(v[key]))
        return out
    if isinstance(v, (list, tuple, set, np.ndarray, pd.Series)):
        for x in list(v):
            out.extend(_answer_texts(x))
        return out
    return [str(v).strip()]

def _load_ref_frame(path):
    try:
        if path.endswith(".parquet"): return pd.read_parquet(path)
        if path.endswith(".csv"):     return pd.read_csv(path)
        if path.endswith(".jsonl"):   return pd.read_json(path, lines=True)
        if path.endswith(".json"):    return pd.read_json(path)
    except Exception as e:
        print("extra QA skip", path, "->", e)
    return None

def add_extra_qa_refs(paths):
    for path in paths or []:
        df_q = _load_ref_frame(path)
        if df_q is None or df_q.empty:
            continue
        q_col = next((c for c in df_q.columns if QA_Q_HINT.search(str(c))), None)
        a_col = next((c for c in df_q.columns if QA_A_HINT.search(str(c)) and c != q_col), None)
        ans_flag = next((c for c in df_q.columns if QA_ANSWERABLE_HINT.search(str(c))), None)
        if q_col is None or a_col is None:
            print(f"{os.path.basename(path)}: could not map QA columns {list(df_q.columns)} - skipped")
            continue
        n0 = len(GOLD); pairs = 0
        for _, row_q in df_q.iterrows():
            if ans_flag is not None and not _truthy_answerable(row_q[ans_flag]):
                continue
            q = str(row_q[q_col]).strip()
            answers = [a for a in _answer_texts(row_q[a_col]) if 0 < len(str(a).strip()) <= 180]
            if q and answers:
                _k = normalize(q)
                GOLD.setdefault(_k, set()).update(map(str, answers))
                GOLD_SRC.setdefault(_k, set()).add(os.path.basename(path))
                pairs += 1
        print(f"{os.path.basename(path)}: q={q_col!r} a={a_col!r} answerable={ans_flag!r}"
              f" -> {pairs} rows, +{len(GOLD) - n0} unique questions")

add_extra_qa_refs(globals().get("EXTRA_QA_FILES", []))
print(f"total exact QA references: {len(GOLD)} unique questions")

LABELED_CORRECT = labeled[labeled.label == 1].groupby("p_norm").response_bn.agg(set).to_dict()
LABELED_WRONG   = labeled[labeled.label == 0].groupby("p_norm").response_bn.agg(set).to_dict()

def make_ref_text(golds, limit=3):
    return " / ".join(sorted(golds, key=len)[:limit])

labeled["ref_text"]    = labeled.p_norm.map(lambda p: make_ref_text(GOLD[p]) if p in GOLD else "")
labeled["ref_contain"] = labeled.apply(
    lambda r: bool(r.ref_text) and contains_gold(r.response_bn, r.prompt_bn, GOLD[r.p_norm]), axis=1)

test["ref_pool"]    = test.p_norm.map(lambda p: set(GOLD.get(p, set())) | set(LABELED_CORRECT.get(p, set())))
test["ref_text"]    = test.ref_pool.map(lambda g: make_ref_text(g) if g else "")
test["ref_contain"] = test.apply(
    lambda r: bool(r.ref_text) and contains_gold(r.response_bn, r.prompt_bn, r.ref_pool), axis=1)
test["wrong_match"] = test.apply(
    lambda r: any(norm_ans(r.response_bn) == norm_ans(w) for w in LABELED_WRONG.get(r.p_norm, ())), axis=1)
test.loc[test.ref_contain, "wrong_match"] = False

n_ref = int((test.ref_text != "").sum())
print(f"\ntest rows with a reference answer     : {n_ref} ({n_ref/len(test):.1%} of test)")
print(f"  resolved by containment -> label 1  : {int(test.ref_contain.sum())}")
print(f"  known-wrong match -> label 0        : {int(test.wrong_match.sum())}")
print(f"  residual -> reference-augmented judge: {int(((test.ref_text != '') & ~test.ref_contain & ~test.wrong_match).sum())}")

lm = labeled[labeled.ref_text != ""]
if len(lm):
    print(f"\nlabeled sanity: {len(lm)} rows matched squad_bn | containment-as-label accuracy "
          f"{(lm.ref_contain.astype(int) == lm.label).mean():.3f}")
    p1, p0 = lm[lm.ref_contain], lm[~lm.ref_contain]
    print(f"  containment -> 1 branch: n={len(p1)}, precision {(p1.label == 1).mean():.3f}")
    print(f"  residual rows: n={len(p0)}, {(p0.label == 0).mean():.1%} truly hallucinated")


squad_bn: 71030 unique answerable questions loaded
banglarqa.parquet: q='question' a='answers' answerable='is_answerable' -> 11022 rows, +11006 unique questions
nctb_qa.parquet: q='question' a='answer' answerable=None -> 12329 rows, +12320 unique questions
bqad_refs.parquet: q='question' a='answers' answerable=None -> 4234 rows, +4209 unique questions
bcs_refs.parquet: q='question' a='answers' answerable=None -> 3938 rows, +3733 unique questions
bcs_gk_mcq_refs.parquet: q='question' a='answers' answerable=None -> 1371 rows, +1321 unique questions
bcs_bangla_mcq_refs.parquet: q='question' a='answers' answerable=None -> 855 rows, +675 unique questions
bcs_geo_mcq_refs.parquet: q='question' a='answers' answerable=None -> 87 rows, +82 unique questions
bcs_science_mcq_refs.parquet: q='question' a='answers' answerable=None -> 502 rows, +498 unique questions
total exact QA references: 104874 unique questions

test rows with a reference answer     : 1290 (51.3% of test)
  resolved by containme

In [8]:
# ---- v6.2/v6.3: BenHalluEval data from the attached `benhallu-data` dataset ---
# (replaces the embedded payload of earlier revisions - attach the zipped folder
#  as a Kaggle dataset; paths were resolved in the configuration cell)
if USE_BENHALLU:
    bh_hall_raw = pd.read_csv(BH_QA4000_PATH)
    bh_qa = pd.read_csv(BH_QADS_PATH)[
        ["id", "question", "correct_answer", "deepseek_answer", "deepseek_score",
         "gemma_answer", "gemma_score", "qwen_answer", "qwen_score"]]
    bh_ctx = {}
    for sid, c in zip(bh_hall_raw.source_id, bh_hall_raw.context):
        bh_ctx.setdefault(str(sid), str(c) if str(c).lower() != "nan" else "")
    if BH_GT1000_PATH:
        _gt = pd.read_csv(BH_GT1000_PATH)
        for sid, c in zip(_gt.id, _gt.context):
            bh_ctx.setdefault(str(sid), str(c) if str(c).lower() != "nan" else "")
        del _gt
    bh_hall = bh_hall_raw[["source_id", "pattern", "question", "hallucinated_answer"]].copy()
    del bh_hall_raw
    print(f"BenHalluEval loaded: {len(bh_qa)} QA rows | {len(bh_hall)} hallucinated candidates | "
          f"{len(bh_ctx)} contexts")
else:
    print("BenHalluEval data not attached - layer off")


BenHalluEval loaded: 2509 QA rows | 4000 hallucinated candidates | 997 contexts


In [9]:
# ---- v6.2: Layer 1e - BenHalluEval QA references --------------------------------
for df in (labeled, test):
    for c in ("bh_gold", "bh_hall", "bh_m1", "bh_m0"):
        df[c] = False
BH_CLF_ROWS = []

if not USE_BENHALLU:
    print("BenHalluEval layer disabled by flag")
else:
    bh_qa["pn"]   = bh_qa.question.map(normalize)
    bh_hall["pn"] = bh_hall.question.map(normalize)

    def _clean(a):
        a = str(a).strip()
        return a if a and a.lower() != "nan" else ""

    # 1) gold answers extend the existing GOLD reference pool (feeds ref_text /
    #    containment / the ref-judge branch exactly like squad_bn refs)
    n0 = len(GOLD)
    for pn, a in zip(bh_qa.pn, bh_qa.correct_answer):
        a = _clean(a)
        if a:
            GOLD.setdefault(pn, set()).add(a)
            GOLD_SRC.setdefault(pn, set()).add("benhallu")
    print(f"BenHalluEval gold refs: +{len(GOLD) - n0} new questions (GOLD now {len(GOLD)})")

    # 2) exact-pair lookups. On the labeled sample these measured 100% precision
    #    (gold-pair 66/66 -> 1, hall-pair 31/31 -> 0); still gated in section 7.
    BH_GOLD_PAIRS = {(pn, norm_ans(a)) for pn, a in zip(bh_qa.pn, bh_qa.correct_answer) if _clean(a)}
    BH_HALL_PAIRS = {(pn, norm_ans(h)) for pn, h in zip(bh_hall.pn, bh_hall.hallucinated_answer) if _clean(h)}
    _both = BH_GOLD_PAIRS & BH_HALL_PAIRS
    BH_GOLD_PAIRS -= _both
    BH_HALL_PAIRS -= _both
    if _both:
        print(f"dropped {len(_both)} ambiguous pairs present in both pools")

    # 3) real zero-shot LLM answers with judge scores -> weaker, separately gated tiers
    BH_M1_PAIRS, BH_M0_PAIRS = set(), set()
    for m in ("deepseek", "gemma", "qwen"):
        for pn, a, s in zip(bh_qa.pn, bh_qa[f"{m}_answer"], bh_qa[f"{m}_score"]):
            a = _clean(a)
            if not a or pd.isna(s):
                continue
            (BH_M1_PAIRS if int(s) == 1 else BH_M0_PAIRS).add((pn, norm_ans(a)))
    BH_M0_PAIRS -= BH_GOLD_PAIRS | BH_M1_PAIRS   # gold / any judged-correct outranks judged-wrong
    BH_M1_PAIRS -= BH_HALL_PAIRS

    for df in (labeled, test):
        keys = list(zip(df.p_norm, df.response_bn.map(norm_ans)))
        df["bh_gold"] = [k in BH_GOLD_PAIRS for k in keys]
        df["bh_hall"] = [k in BH_HALL_PAIRS for k in keys]
        df["bh_m1"]   = [k in BH_M1_PAIRS and k not in BH_GOLD_PAIRS for k in keys]
        df["bh_m0"]   = [k in BH_M0_PAIRS and k not in BH_HALL_PAIRS for k in keys]

    for name, mask, want in (("gold-pair", labeled.bh_gold, 1), ("hall-pair", labeled.bh_hall, 0),
                             ("model-correct", labeled.bh_m1, 1), ("model-wrong", labeled.bh_m0, 0)):
        n = int(mask.sum())
        if n:
            print(f"labeled {name:14s}: n={n:3d} precision={(labeled.label[mask] == want).mean():.3f}")
    print(f"test fires: gold {int(test.bh_gold.sum())} | hall {int(test.bh_hall.sum())} | "
          f"m1 {int(test.bh_m1.sum())} | m0 {int(test.bh_m0.sum())}")

    # 4) recompute the reference columns now that GOLD grew (same code as layer 1b)
    labeled["ref_text"]    = labeled.p_norm.map(lambda p: make_ref_text(GOLD[p]) if p in GOLD else "")
    labeled["ref_contain"] = labeled.apply(
        lambda r: bool(r.ref_text) and contains_gold(r.response_bn, r.prompt_bn, GOLD[r.p_norm]), axis=1)
    test["ref_pool"]    = test.p_norm.map(lambda p: set(GOLD.get(p, set())) | set(LABELED_CORRECT.get(p, set())))
    test["ref_text"]    = test.ref_pool.map(lambda g: make_ref_text(g) if g else "")
    test["ref_contain"] = test.apply(
        lambda r: bool(r.ref_text) and contains_gold(r.response_bn, r.prompt_bn, r.ref_pool), axis=1)
    test["wrong_match"] = test.apply(
        lambda r: any(norm_ans(r.response_bn) == norm_ans(w) for w in LABELED_WRONG.get(r.p_norm, ())), axis=1)
    test.loc[test.ref_contain, "wrong_match"] = False

    n_ref = int((test.ref_text != "").sum())
    print(f"\nafter BenHalluEval merge: {n_ref} test rows with references "
          f"| containment -> 1: {int(test.ref_contain.sum())} "
          f"| conflicts vs hall-pair: {int((test.ref_contain & test.bh_hall).sum())} "
          f"(hall-pair wins at override time)")
    lm = labeled[labeled.ref_text != ""]
    if len(lm):
        print(f"labeled sanity: {len(lm)} rows with refs | containment-as-label accuracy "
              f"{(lm.ref_contain.astype(int) == lm.label).mean():.3f}")

    # 5) real training pairs for the classifier member (labeled prompts excluded so
    #    the section-7 gates stay uncontaminated; test-prompt overlap is deliberate -
    #    this is public external data, same as the squad_bn ref layer)
    _labeled_prompts = set(labeled.p_norm)
    _gold_seen = set()
    for r in bh_qa.itertuples():
        if r.pn in _labeled_prompts:
            continue
        c = bh_ctx.get(str(r.id), "")
        g = _clean(r.correct_answer)
        if g and r.pn not in _gold_seen:
            _gold_seen.add(r.pn)
            BH_CLF_ROWS.append((c, str(r.question), g, 1))
        for m in ("deepseek", "gemma", "qwen"):
            a, s = _clean(getattr(r, f"{m}_answer")), getattr(r, f"{m}_score")
            if a and not pd.isna(s):
                BH_CLF_ROWS.append(("", str(r.question), a, int(int(s) == 1)))
    for r in bh_hall.itertuples():
        if r.pn in _labeled_prompts:
            continue
        BH_CLF_ROWS.append((bh_ctx.get(str(r.source_id), ""), str(r.question),
                            _clean(r.hallucinated_answer), 0))
    BH_CLF_ROWS = [x for x in BH_CLF_ROWS if x[2]]
    _n1 = sum(1 for x in BH_CLF_ROWS if x[3] == 1)
    print(f"classifier real pairs: {len(BH_CLF_ROWS)} ({_n1} faithful / {len(BH_CLF_ROWS) - _n1} hallucinated)")


BenHalluEval gold refs: +0 new questions (GOLD now 104874)
labeled gold-pair     : n= 66 precision=1.000
labeled hall-pair     : n= 31 precision=1.000
test fires: gold 550 | hall 351 | m1 1 | m0 2

after BenHalluEval merge: 1290 test rows with references | containment -> 1: 679 | conflicts vs hall-pair: 4 (hall-pair wins at override time)
labeled sanity: 143 rows with refs | containment-as-label accuracy 0.930
classifier real pairs: 13175 (4508 faithful / 8667 hallucinated)


In [10]:
# ---- Layer 1c: grammar references (bagdhara / shomash / ek-kothay) ----------
GRAMMAR_Q = re.compile(
    r"বাগধারা|ভাবার্থ|সমাস|ব্যাসবাক্য|এক\s*কথায়|সমার্থক|বিপরীত(?:ার্থক)?\s*শব্দ|সন্ধি|প্রকৃতি\s*ও\s*প্রত্যয়")
QUOTE_RE = re.compile("[\"'\u2018\u201c]([^\"'\u2019\u201d]{1,60})[\"'\u2019\u201d]")

GRAM = {}
if USE_GRAMMAR:
    def _load_any(path):
        try:
            if path.endswith(".csv"):     return pd.read_csv(path)
            if path.endswith(".tsv"):     return pd.read_csv(path, sep="\t")
            if path.endswith(".parquet"): return pd.read_parquet(path)
            if path.endswith(".jsonl"):   return pd.read_json(path, lines=True)
            if path.endswith(".json"):
                try:
                    return pd.read_json(path)
                except ValueError:
                    with open(path, encoding="utf-8") as f:
                        obj = json.load(f)
                    return pd.DataFrame(obj if isinstance(obj, list) else [obj])
        except Exception as e:
            print("skip", path, "->", e)
        return None

    TERM_HINT  = re.compile(r"term|word|idiom|bagdhara|বাগধারা|প্রবাদ|somash|shomash|samas|সমাস|"
                            r"pada|phrase|বাক্যাংশ|proshno|question|প্রশ্ন|title|শিরোনাম|key", re.I)
    ANS_HINT   = re.compile(r"mean|meaning|figurative|answer|artho|ortho|অর্থ|মানে|ভাবার্থ|bekkha|ব্যাখ্যা|"
                            r"bashbakko|ব্যাস|byas|one_word|expres|প্রকাশ|value|solution|explan", re.I)
    # v6.9: `category` was matching bagdhara's idiom-taxonomy column, so values like
    # "cultural"/"religious" were added as REFERENCE ANSWERS - and make_ref_text sorts
    # by length, so the junk led the reference note shown to the judge. Only genuine
    # answer-bearing extras (shomash's samas_type) qualify.
    EXTRA_HINT = re.compile(r"samas_type|samastype|_type$|^type$|ধরন|প্রকার", re.I)
    SKIP_COLS  = re.compile(r"serial|source|url|category|^id$|^index$", re.I)

    _seen_stems = set()
    for path in GRAMMAR_FILES:
        stem = os.path.splitext(os.path.basename(path))[0]
        if stem in _seen_stems:          # csv/jsonl/parquet triplets: load once
            continue
        df_g = _load_any(path)
        if df_g is None or df_g.empty:
            continue
        _seen_stems.add(stem)
        str_cols = [c for c in df_g.columns
                    if df_g[c].dtype == object and not SKIP_COLS.search(str(c))]
        if len(str_cols) < 2:
            continue
        term_col = GRAMMAR_TERM_COL or next((c for c in str_cols if TERM_HINT.search(str(c))), None)
        ans_col  = GRAMMAR_ANS_COL  or next((c for c in str_cols
                                             if ANS_HINT.search(str(c)) and c != term_col), None)
        if term_col is None or ans_col is None:
            lens = {c: df_g[c].astype(str).str.len().mean() for c in str_cols}
            ordered = sorted(lens, key=lens.get)
            term_col = term_col or ordered[0]
            ans_col  = ans_col  or ordered[-1]
        if term_col == ans_col:
            continue
        extra_col = next((c for c in str_cols
                          if EXTRA_HINT.search(str(c)) and c not in (term_col, ans_col)), None)
        n0 = len(GRAM)
        for _, row_g in df_g.iterrows():
            t, a = str(row_g[term_col]).strip(), str(row_g[ans_col]).strip()
            if not (0 < len(t) <= 60 and a and a.lower() != "nan"):
                continue
            refs = {a}
            if extra_col is not None:
                x = str(row_g[extra_col]).strip()
                if x and x.lower() != "nan":
                    refs.add(x)
            for part in re.split(r"\s*/\s*", t):
                part = part.strip()
                if part:
                    GRAM.setdefault(normalize(part), set()).update(refs)
        print(f"{os.path.basename(path)}: term={term_col!r} ans={ans_col!r}"
              f"{' extra=' + repr(extra_col) if extra_col else ''} -> +{len(GRAM) - n0} terms")
    print(f"grammar lookup: {len(GRAM)} terms")

GRAM_KEYS = sorted(GRAM, key=len, reverse=True)

def grammar_refs(prompt):
    """Reference answers for a grammar-style question, else empty set."""
    if not GRAM or not GRAMMAR_Q.search(prompt):
        return set()
    m = QUOTE_RE.search(prompt)
    if m:
        k = normalize(m.group(1))
        if k in GRAM:
            return GRAM[k]
    p_n = normalize(prompt)
    for k in GRAM_KEYS:
        if len(k) >= 3 and k in p_n:
            return GRAM[k]
    return set()

for df in (labeled, test):
    refs = df.prompt_bn.map(grammar_refs)
    df["gram_hit"] = refs.map(bool)
    df["gram_contain"] = [contains_gold(r, p, g) if g else False
                          for r, p, g in zip(df.response_bn, df.prompt_bn, refs)]
    fill = df.gram_hit & (df.ref_text == "")
    df.loc[fill, "ref_text"] = refs[fill].map(make_ref_text)

for name, df in (("labeled", labeled), ("test", test)):
    n_gq = int(df.prompt_bn.map(lambda p: bool(GRAMMAR_Q.search(p))).sum())
    print(f"{name}: grammar-style prompts {n_gq} | with reference {int(df.gram_hit.sum())} "
          f"| containment hits {int(df.gram_contain.sum())}")
if USE_GRAMMAR and labeled.gram_hit.sum():
    gl = labeled[labeled.gram_hit]
    print(f"labeled grammar sanity: containment-as-label accuracy "
          f"{(gl.gram_contain.astype(int) == gl.label).mean():.3f} on n={len(gl)}")

bagdhara.csv: term='bagdhara' ans='meaning' -> +1299 terms
ek_kothay_prokash.csv: term='phrase' ans='one_word' -> +1058 terms
shomash.csv: term='samasta_pada' ans='byasabakya' extra='samas_type' -> +189 terms
grammar lookup: 2546 terms
labeled: grammar-style prompts 14 | with reference 2 | containment hits 0
test: grammar-style prompts 293 | with reference 57 | containment hits 3
labeled grammar sanity: containment-as-label accuracy 1.000 on n=2


In [11]:
# ---- Layer 1h: field-aware bagdhara idiom tiers (v6.11) -----------------------
# Measured 12/12 on labeled closed rows; ~116 test verdicts. Uses only helpers
# defined above (normalize, norm_ans). Self-disables without the dictionary.
USE_IDIOM_TIERS = True
IDM_LIT, IDM_FIG = {}, {}
IDIOM1_OK = IDIOM0_OK = False    # precision-gated in section 7

def _idm_clean(v):
    v = re.sub(r"\s*\([^)]*\)\s*", " ", str(v)).strip()   # strip English parentheticals
    parts = {v} | {p.strip() for p in re.split(r"[;,।]", v) if len(p.strip()) >= 4}
    return {p for p in parts if p}

def _idm_source():
    import glob as _g
    hits = sorted(d for d, _, _ in os.walk("/kaggle/input")
                  if "bagdhara-bangla-idioms" in os.path.basename(d).lower())
    if hits:
        if any("bagdhara_bangla_idioms" in os.path.basename(p) for p in GRAMMAR_FILES):
            print("NOTE: the attached idiom dictionary was ALSO merged into the grammar "
                  "layer (GRAM differs from the 0.839 run). Prefer not attaching it - "
                  "this layer downloads its own copy.")
        return hits[0]
    if ONLINE:
        try:
            import urllib.request, zipfile
            _dst = os.path.join(WORK_DIR, "idm_bagdhara")
            if not os.path.isdir(_dst):
                req = urllib.request.Request(
                    "https://www.kaggle.com/api/v1/datasets/download/"
                    "sakhadib/bagdhara-bangla-idioms-dataset",
                    headers={"User-Agent": "Mozilla/5.0"})
                _zp = os.path.join(WORK_DIR, "idm_bagdhara.zip")
                with urllib.request.urlopen(req, timeout=300) as r, open(_zp, "wb") as f:
                    f.write(r.read())
                with zipfile.ZipFile(_zp) as z:
                    z.extractall(_dst)
            return _dst
        except Exception as e:
            print(f"idiom dictionary download failed ({e})")
    return None

if USE_IDIOM_TIERS:
    import glob as _idm_glob
    _src = _idm_source()
    if _src:
        _n_bad = 0
        for _fp in _idm_glob.glob(os.path.join(_src, "**", "*.json"), recursive=True):
            try:
                with open(_fp, encoding="utf-8") as f:
                    _r = json.load(f)
            except Exception:
                _n_bad += 1
                continue
            if not isinstance(_r, dict) or "idiom" not in _r:
                continue
            _lit = _idm_clean(_r.get("literal_meaning") or "")
            _fig = _idm_clean(_r.get("figurative_meaning_bn") or "")
            for _key in [_r.get("idiom", "")] + list(_r.get("alternative_idioms") or []):
                _key = normalize(_key)
                if _key:
                    IDM_LIT.setdefault(_key, set()).update(_lit)
                    IDM_FIG.setdefault(_key, set()).update(_fig)
        print(f"idiom dictionary: {len(IDM_LIT)} keys loaded"
              + (f" ({_n_bad} unparseable files skipped)" if _n_bad else ""))
    else:
        print("idiom dictionary unavailable - idiom tiers disabled")
else:
    print("idiom tiers disabled by flag")

_IDMQ_RE = re.compile(r'[\"“”‘’\'](.+?)[\"“”‘’\']')
_IDML_RE = re.compile(r"শাব্দিক|আভিধানিক")

def _idm_match(response, golds):
    r_n = norm_ans(response)
    for g0 in golds:
        g = norm_ans(g0)
        if len(g) < 4:
            continue
        if g == r_n or (g in r_n and len(g) >= 6) or (r_n in g and len(r_n) >= 8):
            return True
    return False

def idiom_verdict(prompt, response):
    """1 = response matches the ASKED field's meaning, 0 = it matches only the
    OTHER field (the literal/figurative swap trap), None = no signal."""
    m = _IDMQ_RE.search(str(prompt))
    if not m:
        return None
    t = normalize(m.group(1))
    if t not in IDM_LIT and t not in IDM_FIG:
        return None
    lit, fig = IDM_LIT.get(t, set()), IDM_FIG.get(t, set())
    right, wrong = (lit, fig) if _IDML_RE.search(str(prompt)) else (fig, lit)
    if _idm_match(response, right):
        return 1
    if _idm_match(response, wrong):
        return 0
    return None

for _df in (labeled, test):
    _df["idiom_v"] = np.nan
    if IDM_LIT:
        for _i in _df.index[~_df.grounded]:
            _v = idiom_verdict(_df.at[_i, "prompt_bn"], _df.at[_i, "response_bn"])
            if _v is not None:
                _df.at[_i, "idiom_v"] = _v
if IDM_LIT:
    print(f"idiom verdicts: labeled {int(labeled.idiom_v.notna().sum())} "
          f"(->1 {int((labeled.idiom_v == 1).sum())} / ->0 {int((labeled.idiom_v == 0).sum())}) | "
          f"test {int(test.idiom_v.notna().sum())} "
          f"(->1 {int((test.idiom_v == 1).sum())} / ->0 {int((test.idiom_v == 0).sum())})")


idiom dictionary: 11051 keys loaded
idiom verdicts: labeled 12 (->1 8 / ->0 4) | test 116 (->1 76 / ->0 40)


In [12]:
BN_STOP = set("কী কি কে কোন কত কোথায় কবে কেন হয় হলো ছিল এর ও এবং থেকে জন্য করে একটি টি টির".split())
TOK_RE  = re.compile(r"[\w\u0980-\u09FF]+")

def bm25_tokenize(s):
    s = nfc(s).translate(BN_DIGITS).lower()
    return [t for t in TOK_RE.findall(s) if t not in BN_STOP and len(t) > 1]

class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.N = len(docs_tokens)
        self.dl = np.array([len(d) for d in docs_tokens], dtype=np.float32)
        self.avgdl = max(float(self.dl.mean()), 1.0)
        postings = {}
        for i, d in enumerate(docs_tokens):
            seen = {}
            for t in d:
                seen[t] = seen.get(t, 0) + 1
            for t, tf in seen.items():
                postings.setdefault(t, ([], []))
                postings[t][0].append(i); postings[t][1].append(tf)
        self.post = {t: (np.array(ids, dtype=np.int32), np.array(tfs, dtype=np.float32))
                     for t, (ids, tfs) in postings.items()}
        self.idf = {t: math.log(1 + (self.N - len(ids) + 0.5) / (len(ids) + 0.5))
                    for t, (ids, _) in self.post.items()}

    def top(self, query_tokens, k=2):
        scores = np.zeros(self.N, dtype=np.float32)
        for t in set(query_tokens):
            if t not in self.post:
                continue
            ids, tfs = self.post[t]
            denom = tfs + self.k1 * (1 - self.b + self.b * self.dl[ids] / self.avgdl)
            scores[ids] += self.idf[t] * tfs * (self.k1 + 1) / denom
        k = min(k, self.N)
        idx = np.argpartition(scores, -k)[-k:]
        idx = idx[np.argsort(-scores[idx])]
        return idx, scores[idx]

labeled["ret_text"] = ""; labeled["ret_conf"] = np.nan
test["ret_text"]    = ""; test["ret_conf"]    = np.nan

if USE_RETRIEVAL:
    import gc
    try:
        from datasets import load_from_disk
        wiki = load_from_disk(BNWIKI_PATH)
        wiki_iter = ({"title": a["title"], "text": a["text"]} for a in wiki)
        n_articles = len(wiki)
    except Exception as e:
        print("load_from_disk failed:", e, "-> pyarrow fallback")
        import pyarrow as pa
        tables = [pa.ipc.open_stream(open(p, "rb")).read_all()
                  for p in sorted(glob.glob(os.path.join(BNWIKI_PATH, "*.arrow")))]
        tbl = pa.concat_tables(tables)
        n_articles = tbl.num_rows
        wiki_iter = ({"title": t, "text": x} for t, x in
                     zip(tbl.column("title").to_pylist(), tbl.column("text").to_pylist()))

    from tqdm.auto import tqdm
    chunks, chunk_tok = [], []
    for art in tqdm(wiki_iter, total=n_articles, desc="chunking bn-wiki"):
        title, lead = str(art["title"]), str(art["text"])[:1300]
        parts = [p.strip() for p in lead.split("\u0964") if p.strip()]
        buf, taken = "", 0
        for p in parts:
            if len(buf) + len(p) > 600 and buf:
                chunks.append(title + " \u2014 " + buf + "\u0964")
                chunk_tok.append(bm25_tokenize(title + " " + buf))
                taken += 1; buf = p
                if taken >= 2:
                    buf = ""; break
            else:
                buf = (buf + "\u0964 " + p) if buf else p
        if buf and taken < 2:
            chunks.append(title + " \u2014 " + buf + "\u0964")
            chunk_tok.append(bm25_tokenize(title + " " + buf))

    # ---- v6.3: Phase-2 source-base expansion (news / searched corpora) ----------
    def _chunk_doc(title, text):
        title, text = str(title).strip(), str(text)[:1300]
        if not re.search(r"[\u0980-\u09FF]", text):
            return   # Bengali-script guard: searched repos may return junk
        parts = [p.strip() for p in text.split("\u0964") if p.strip()]
        buf, taken = "", 0
        for p in parts:
            if len(buf) + len(p) > 600 and buf:
                chunks.append(title + " \u2014 " + buf + "\u0964")
                chunk_tok.append(bm25_tokenize(title + " " + buf))
                taken += 1; buf = p
                if taken >= 2:
                    buf = ""; break
            else:
                buf = (buf + "\u0964 " + p) if buf else p
        if buf and taken < 2:
            chunks.append(title + " \u2014 " + buf + "\u0964")
            chunk_tok.append(bm25_tokenize(title + " " + buf))

    if globals().get("NEWS_CORPUS_PATH"):
        n0 = len(chunks)
        _news = pd.read_parquet(NEWS_CORPUS_PATH)
        for ttl, txt in zip(_news.title.astype(str), _news.text.astype(str)):
            _chunk_doc(ttl[:60], txt)
        del _news
        print(f"retrieval += {len(chunks) - n0} chunks from BBC Bangla news (XL-Sum)")
    for _path in globals().get("RETRIEVAL_EXTRA_FILES", []) or []:
        try:
            df_x = pd.read_parquet(_path)
        except Exception as e:
            print("retrieval extra skip", _path, e); continue
        _scols = [c for c in df_x.columns if df_x[c].dtype == object]
        if not _scols:
            continue
        _lens = {c: df_x[c].astype(str).str.len().mean() for c in _scols}
        _tcol = max(_lens, key=_lens.get)
        _ttlcol = next((c for c in _scols if re.search(r"title|name|heading", str(c), re.I)), None)
        _texts = df_x[_tcol].astype(str).tolist()
        _titles = (df_x[_ttlcol].astype(str).tolist() if _ttlcol
                   else [t[:40] for t in _texts])
        n0, _cap = len(chunks), 120000
        for ttl, txt in zip(_titles, _texts):
            if len(chunks) - n0 >= _cap:
                break
            _chunk_doc(ttl[:60], txt)
        del df_x
        print(f"retrieval += {len(chunks) - n0} chunks from {os.path.basename(_path)}")

    print(len(chunks), "chunks; building BM25 index...")
    bm25 = BM25(chunk_tok)
    del chunk_tok; gc.collect()

    for df, dname in ((labeled, "labeled"), (test, "test")):
        closed_idx = df.index[~df.grounded]
        for i in tqdm(closed_idx, desc=f"retrieve {dname}"):
            q = bm25_tokenize(df.at[i, "prompt_bn"])
            if not q:
                continue
            idx, sc = bm25.top(q, k=2)
            if sc[0] <= 0:
                continue
            df.at[i, "ret_text"] = " ".join(chunks[j] for j in idx)[:1500]
            df.at[i, "ret_conf"] = float(sc[0])
    confs = pd.concat([labeled.ret_conf, test.ret_conf]).dropna().values
    print(f"retrieved for {int((labeled.ret_text != '').sum())} labeled + "
          f"{int((test.ret_text != '').sum())} test closed rows | "
          f"conf p50={np.percentile(confs, 50):.1f} p75={np.percentile(confs, 75):.1f} p90={np.percentile(confs, 90):.1f}")
else:
    print("retrieval disabled")


# ---- v4: context relevance (share of question content-words found in context) ----
def ctx_relevance(prompt, context):
    pt = set(bm25_tokenize(prompt))
    if not pt:
        return 1.0
    ct = set(bm25_tokenize(context))
    return len(pt & ct) / len(pt)

for df in (labeled, test):
    df["ctx_rel"] = [ctx_relevance(p, c) if g else np.nan
                     for p, c, g in zip(df.prompt_bn, df.context, df.grounded)]
_rel = pd.concat([labeled.ctx_rel, test.ctx_rel]).dropna()
print(f"context relevance: p10={_rel.quantile(.10):.2f} p25={_rel.quantile(.25):.2f} "
      f"median={_rel.quantile(.50):.2f} | rows < 0.30: {int((_rel < 0.30).sum())}")


chunking bn-wiki:   0%|          | 0/143069 [00:00<?, ?it/s]

retrieval += 20151 chunks from BBC Bangla news (XL-Sum)
retrieval += 33282 chunks from bdlaws.parquet
306911 chunks; building BM25 index...


retrieve labeled:   0%|          | 0/169 [00:00<?, ?it/s]

retrieve test:   0%|          | 0/1155 [00:00<?, ?it/s]

retrieved for 169 labeled + 1154 test closed rows | conf p50=23.1 p75=29.3 p90=37.1
context relevance: p10=0.20 p25=0.33 median=0.50 | rows < 0.30: 261


In [13]:
# ---- Layer 1b-ctx: paragraph-level squad_bn matching (v5) -------------------
PARA, PARA_PREFIX = {}, {}
_NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")

def _num(s):
    m = _NUM_RE.search(norm_ans(s).replace(",", ""))
    return float(m.group()) if m else None

if USE_CTX_MATCH and tarball:
    with tarfile.open(tarball, "r:bz2") as tf:
        for split in ("train", "validation", "test"):
            with tf.extractfile(f"squad_bn/{split}.json") as fh:
                data = json.load(fh)
            for art in data["data"]:
                for para in art["paragraphs"]:
                    qas = []
                    for qa in para["qas"]:
                        answers = tuple(a["text"] for a in qa.get("answers", []) if a.get("text"))
                        if answers:
                            qas.append((frozenset(bm25_tokenize(qa["question"])), answers))
                    if not qas:
                        continue
                    k = normalize(para["context"])
                    PARA.setdefault(k, []).extend(qas)
                    PARA_PREFIX.setdefault(k[:200], []).extend(qas)
    print(f"paragraph index: {len(PARA)} unique paragraphs")

def para_qas(context):
    k = normalize(context)
    return PARA.get(k) or PARA_PREFIX.get(k[:200]) or []

def pool_match(response, answers):
    """Generous match (no echo guard): equality/containment either way, or equal numbers."""
    r = norm_ans(response)
    if not r:
        return False
    rn = _num(response)
    for a in answers:
        g = norm_ans(a)
        if len(g) < 2:
            continue
        if g == r or g in r or r in g:
            return True
        an = _num(a)
        if rn is not None and an is not None and abs(rn - an) < 1e-6:
            return True
    return False

from difflib import SequenceMatcher as _SM

def resp_agree(a, b):
    """Do two responses plausibly express the same answer?"""
    x, y = norm_ans(a), norm_ans(b)
    if not x or not y:
        return False
    if x == y or x in y or y in x:
        return True
    nx, ny = _num(a), _num(b)
    if nx is not None and ny is not None:
        return abs(nx - ny) < 1e-6
    return _SM(None, x, y).ratio() >= 0.75

def _tok_match(a, b):
    """Prefix-tolerant token equality - absorbs Bangla inflection
    (কীবোর্ডের ~ কীবোর্ড, আবিষ্কারক ~ আবিষ্কার)."""
    if a == b:
        return True
    return len(a) >= 4 and len(b) >= 4 and a[:4] == b[:4]

def soft_overlap(q_toks, p_toks):
    if not p_toks:
        return 0.0
    hits = sum(1 for p in p_toks if any(_tok_match(p, q) for q in q_toks))
    return hits / len(p_toks)

SOFT_MIN = 0.60
for df in (labeled, test):
    hit, soft, contain, miss = [], [], [], []
    for row in df.itertuples():
        qas = para_qas(row.context) if row.grounded else []
        if not qas:
            hit.append(False); soft.append(False); contain.append(False); miss.append(False)
            continue
        hit.append(True)
        pt = set(bm25_tokenize(row.prompt_bn))
        best_ov, best_ans = 0.0, ()
        for q_toks, answers in qas:
            ov = soft_overlap(q_toks, pt)
            if ov > best_ov:
                best_ov, best_ans = ov, answers
        is_soft = best_ov >= SOFT_MIN
        soft.append(is_soft)
        contain.append(bool(is_soft and contains_gold(row.response_bn, row.prompt_bn, set(best_ans))))
        pool = set()
        for _, answers in qas:
            pool.update(answers)
        miss.append(not pool_match(row.response_bn, pool))
    df["ctx_hit"], df["ctx_soft"] = hit, soft
    df["ctx_contain"], df["ctx_miss"] = contain, miss
    # v5.1: NO ref_text fill - soft-matched refs feed the tier-1 override only,
    # never the prompts (a 0.60 overlap can pick the wrong question from the
    # right paragraph; containment corroboration protects the tier, prompts had none)

for name, df in (("labeled", labeled), ("test", test)):
    g = df[df.grounded]
    print(f"{name}: paragraph-matched {int(g.ctx_hit.sum())}/{len(g)} grounded | "
          f"soft prompt match {int(g.ctx_soft.sum())} | "
          f"containment {int((g.ctx_soft & g.ctx_contain).sum())} | "
          f"pool-miss {int((g.ctx_hit & g.ctx_miss).sum())}")


paragraph index: 21577 unique paragraphs
labeled: paragraph-matched 83/130 grounded | soft prompt match 80 | containment 45 | pool-miss 35
test: paragraph-matched 938/1361 grounded | soft prompt match 937 | containment 546 | pool-miss 370


In [14]:
# ---- Layer 1f: grounded numeric context-verification (v6.4) ------------------
_NUM = re.compile(r"\d+(?:\.\d+)?")
def _digits(s):
    return set(_NUM.findall(nfc(s).translate(BN_DIGITS)))

def ctx_num_contradiction(response, context):
    """Grounded numeric contradiction: response has a number, the context DOES contain
    digits, and some response number is absent from the context -> hallucination (0).
    Returns True (contradiction), False (numbers consistent), or None (abstain:
    no numbers in the response, or the context has no digits to compare against)."""
    rn = _digits(response)
    if not rn:
        return None
    cn = _digits(context)
    if not cn:
        return None
    return not rn.issubset(cn)

CTXCOV_STOP = set("কী কি কে কোন কত কোথায় কবে কেন হয় হলো ছিল এর ও এবং থেকে জন্য করে "
                  "একটি টি টির এই সেই তার অথবা কয়টি".split())
def ctx_coverage(response, context):
    """Share of response content-tokens (prefix-4, inflection-tolerant) found in the
    context. Low coverage weakly signals an unsupported response; used as a stack
    feature only (not override-grade on its own)."""
    rt = [t for t in bm25_tokenize(response) if t not in CTXCOV_STOP and len(t) > 1]
    if not rt:
        return 1.0
    ct = {t[:4] for t in bm25_tokenize(context)}
    return sum(1 for t in rt if t[:4] in ct) / len(rt)

for df in (labeled, test):
    g = df.grounded.values
    df["ctx_num_bad"] = [bool(g[i]) and (ctx_num_contradiction(r, c) is True)
                         for i, (r, c) in enumerate(zip(df.response_bn, df.context))]
    df["ctx_cov"] = [ctx_coverage(r, c) if g[i] else 1.0
                     for i, (r, c) in enumerate(zip(df.response_bn, df.context))]

lg = labeled[labeled.grounded]
fire = lg[lg.ctx_num_bad]
if len(fire):
    print(f"Layer 1f (labeled grounded): numeric-contradiction fires n={len(fire)} "
          f"precision(0)={(fire.label == 0).mean():.3f} | "
          f"recall of grounded halluc {int((fire.label == 0).sum())}/{int((lg.label == 0).sum())}")
print(f"Layer 1f (test grounded): {int(test.ctx_num_bad.sum())} rows flagged as "
      f"numeric contradictions -> candidate label 0 (gated in section 7)")


Layer 1f (labeled grounded): numeric-contradiction fires n=13 precision(0)=1.000 | recall of grounded halluc 13/47
Layer 1f (test grounded): 130 rows flagged as numeric contradictions -> candidate label 0 (gated in section 7)


In [15]:
# ---- Layer 1d: closed-book external references ------------------------------
CB = {}
if USE_CB_REFS and CB_FILES:
    Q_HINT = re.compile(r"question|prompt|proshno|প্রশ্ন|problem|body|text|stem|mcq", re.I)
    A_HINT = re.compile(r"answer|\bans\b|উত্তর|solution|correct|সঠিক|label|output|option|choice", re.I)
    _seen = set()
    for path in CB_FILES:
        stem = os.path.splitext(os.path.basename(path))[0]
        if stem in _seen:
            continue
        df_c = None
        try:
            if path.endswith(".parquet"): df_c = pd.read_parquet(path)
            elif path.endswith(".csv"):   df_c = pd.read_csv(path)
            elif path.endswith(".jsonl"): df_c = pd.read_json(path, lines=True)
            elif path.endswith(".json"):  df_c = pd.read_json(path)
        except Exception as e:
            print("skip", path, "->", e)
            continue
        if df_c is None or df_c.empty:
            continue
        _seen.add(stem)
        str_cols = [c for c in df_c.columns if df_c[c].dtype == object]
        q_col = next((c for c in str_cols if Q_HINT.search(str(c))), None)
        # v6.11: prefer an exact answer column. `options` also matches A_HINT and, in
        # azminetoushikwasi/bangla-bcs-qs, precedes `answer` - which silently mapped the
        # LIST OF CHOICES as the gold answer. That dataset is handled properly by Layer 1g.
        A_STRICT = re.compile(r"^(answer|ans|correct(_answer)?|solution|উত্তর)$", re.I)
        a_col = (next((c for c in df_c.columns if A_STRICT.search(str(c)) and c != q_col), None)
                 or next((c for c in df_c.columns if A_HINT.search(str(c)) and c != q_col), None))
        if q_col is None or a_col is None:
            print(f"{os.path.basename(path)}: could not map columns {list(df_c.columns)} - skipped")
            continue
        _s = df_c[a_col].dropna().head(20)
        if any(isinstance(v, (list, tuple)) or (isinstance(v, str) and v.strip().startswith("["))
               for v in _s):
            print(f"{os.path.basename(path)}: answer column {a_col!r} holds option LISTS - skipped")
            continue
        if "option" in str(a_col).lower():
            print(f"{os.path.basename(path)}: refusing to use {a_col!r} as a gold answer - skipped")
            continue
        n0 = len(CB)
        for q, a in zip(df_c[q_col], df_c[a_col]):
            q, a = str(q).strip(), str(a).strip()
            if q and a and a.lower() != "nan":
                CB.setdefault(normalize(q), set()).add(a)
        print(f"{os.path.basename(path)}: q={q_col!r} a={a_col!r} -> +{len(CB) - n0} questions")
    print(f"closed-book reference lookup: {len(CB)} questions")

for df in (labeled, test):
    cb_hit, cb_contain, cb_miss = [], [], []
    for row in df.itertuples():
        golds = CB.get(row.p_norm) if not row.grounded else None
        if not golds:
            cb_hit.append(False); cb_contain.append(False); cb_miss.append(False)
            continue
        cb_hit.append(True)
        cb_contain.append(bool(contains_gold(row.response_bn, row.prompt_bn, golds)))
        cb_miss.append(not pool_match(row.response_bn, golds))
    df["cb_hit"], df["cb_contain"], df["cb_miss"] = cb_hit, cb_contain, cb_miss
    # v5.1: NO ref_text fill (16 of 20 harmful 1->0 flips traced to cb-ref prompts);
    # the layer's remaining value is the printed overlap sweep + gated tiers

for name, df in (("labeled", labeled), ("test", test)):
    c = df[~df.grounded]
    print(f"{name}: closed-book matched {int(c.cb_hit.sum())}/{len(c)} | "
          f"containment {int(c.cb_contain.sum())} | pool-miss {int((c.cb_hit & c.cb_miss).sum())}")


bangla_math.parquet: q='Problem' a='Answer' -> +1205 questions
closed-book reference lookup: 1205 questions
labeled: closed-book matched 0/169 | containment 0 | pool-miss 0
test: closed-book matched 0/1155 | containment 0 | pool-miss 0


In [16]:
# ---- encoder: LOAD-ONLY (training excluded - attach the clf_finetuned checkpoint) ----
labeled["p_clf"] = np.nan
test["p_clf"]    = np.nan
CLF_OK = False

if USE_CLF:
    _CLF_SAVED = None
    for _r in ("/kaggle/input", WORK_DIR, "."):
        if not os.path.isdir(_r):
            continue
        for _dp, _dn, _fn in os.walk(_r):
            if os.path.basename(_dp) == CLF_SAVE_NAME and "config.json" in _fn:
                _CLF_SAVED = _dp
                break
        if _CLF_SAVED:
            break
    assert _CLF_SAVED, ("FINAL NOTEBOOK: encoder training is excluded - attach the "
                        f"'{CLF_SAVE_NAME}' checkpoint folder as a Kaggle dataset")
    print(f"classifier: loading fine-tuned checkpoint from {_CLF_SAVED} (no training)")

    from transformers import AutoTokenizer as _AT, AutoModelForSequenceClassification
    clf_tok = _AT.from_pretrained(_CLF_SAVED)
    clf = AutoModelForSequenceClassification.from_pretrained(
        _CLF_SAVED, num_labels=2, ignore_mismatched_sizes=True).to("cuda:0")
    clf.eval()

    try:   # BanglaBERT preprocessing best practice
        from normalizer import normalize as _bn_norm
        print("clf: using csebuetnlp normalizer")
    except Exception:
        _bn_norm = lambda s: s

    def clf_encode(ctxs, qs, rs):
        first  = [_bn_norm(f"প্রশ্ন: {str(q)[:300]} উত্তর: {str(r)[:300]}")
                  for q, r in zip(qs, rs)]
        second = [(_bn_norm(str(c)[:1200]) if c else "কোনো অনুচ্ছেদ নেই") for c in ctxs]
        try:
            return clf_tok(first, second, truncation="only_second", max_length=CLF_MAX_LEN,
                           padding=True, return_tensors="pt")
        except Exception:
            return clf_tok(first, second, truncation=True, max_length=CLF_MAX_LEN,
                           padding=True, return_tensors="pt")

    from tqdm.auto import tqdm

    @torch.inference_mode()
    def clf_scores(df, desc):
        out = np.zeros(len(df))
        for lo in tqdm(range(0, len(df), 64), desc=desc, leave=False):
            part = df.iloc[lo:lo + 64]
            enc = clf_encode(part.context.tolist(), part.prompt_bn.tolist(),
                             part.response_bn.tolist())
            enc = {k: v.to("cuda:0") for k, v in enc.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                logits = clf(**enc).logits.float()
            out[lo:lo + len(part)] = logits.softmax(-1)[:, 1].cpu().numpy()
        return out

    labeled["p_clf"] = clf_scores(labeled, "clf labeled")
    test["p_clf"]    = clf_scores(test, "clf test")
    CLF_OK = True

    from sklearn.metrics import roc_auc_score
    _g = labeled.grounded.values
    print(f"labeled AUC  all {roc_auc_score(labeled.label, labeled.p_clf):.3f} | "
          f"grounded {roc_auc_score(labeled.label[_g], labeled.p_clf[_g]):.3f} | "
          f"closed {roc_auc_score(labeled.label[~_g], labeled.p_clf[~_g]):.3f}")

    del clf
    import gc; gc.collect(); torch.cuda.empty_cache()
    print("classifier member ready (loaded, not trained); encoder freed")
else:
    print("classifier member skipped: flag off")


classifier: loading fine-tuned checkpoint from /kaggle/input/datasets/zarifmahir/trainedencoder/clf_finetuned/clf_finetuned (no training)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

clf: using csebuetnlp normalizer


clf labeled:   0%|          | 0/5 [00:00<?, ?it/s]

clf test:   0%|          | 0/40 [00:00<?, ?it/s]

labeled AUC  all 0.819 | grounded 0.972 | closed 0.571
classifier member ready (loaded, not trained); encoder freed


In [17]:
# ---- v6: cultural-default (C1) traps -----------------------------------------
# (prompt_pattern, wrong_default_pattern, correct_pattern) - all matched on the
# NFC-normalized text. wrong -> pred 0, correct -> pred 1.
C1_TRAPS = [
    # national poet is Nazrul; the global default reflex is Tagore
    (r"জাতীয়\s*কবি", r"রবীন্দ্রনাথ|ঠাকুর", r"নজরুল"),
    # national anthem IS Tagore; reverse confusion
    (r"জাতীয়\s*সংগীত|জাতীয়\s*সঙ্গীত", r"নজরুল", r"রবীন্দ্রনাথ"),
    # martial song is Nazrul
    (r"রণসংগীত|রণ\s*সঙ্গীত", r"রবীন্দ্রনাথ|ঠাকুর", r"নজরুল"),
    # Yunus won the Peace prize, not Economics
    (r"ইউনূস.*নোবেল|নোবেল.*ইউনূস", r"অর্থনীতি", r"শান্তি"),
    # ORS was developed in Bangladesh (icddr,b / Cholera Research Lab)
    (r"খাবার\s*স্যালাইন|ওরস্যালাইন|ও\.?আর\.?এস|ORS", r"যুক্তরাষ্ট্র|আমেরিকা|মার্কিন|পশ্চিমা|ইউরোপ",
     r"বাংলাদেশ|আইসিডিডিআর|icddr|কলেরা"),
    # Bangabandhu is the Father of the Nation (default reflex: Gandhi/Jinnah)
    (r"জাতির\s*(জনক|পিতা)", r"গান্ধী|জিন্নাহ", r"মুজিব"),
]

def c1_flags(prompt, response):
    """Returns (wrong_fire, right_fire, trap_index) - first matching trap wins."""
    for ti, (pp, wp, cp) in enumerate(C1_TRAPS):
        if re.search(pp, prompt):
            if re.search(wp, response, re.I):
                return True, False, ti
            if re.search(cp, response, re.I):
                return False, True, ti
    return False, False, -1

for df in (labeled, test):
    w, r_, t_ = zip(*(c1_flags(p, r) for p, r in zip(df.prompt_bn, df.response_bn)))
    df["c1_wrong"], df["c1_right"], df["c1_trap"] = list(w), list(r_), list(t_)

# validation: a trap that mislabels even one labeled row is disabled
C1_ENABLED = set(range(len(C1_TRAPS)))
for ti in range(len(C1_TRAPS)):
    m = labeled.c1_trap == ti
    if not m.any():
        continue
    fires = labeled[m]
    bad = int(((fires.c1_wrong) & (fires.label == 1)).sum()
              + ((fires.c1_right) & (fires.label == 0)).sum())
    print(f"trap {ti}: {len(fires)} labeled fires, {bad} wrong -> "
          f"{'kept' if bad == 0 else 'DISABLED'}")
    if bad > 0:
        C1_ENABLED.discard(ti)

test["c1_wrong"] = test.c1_wrong & test.c1_trap.isin(C1_ENABLED)
test["c1_right"] = test.c1_right & test.c1_trap.isin(C1_ENABLED)
print(f"test fires: {int(test.c1_wrong.sum())} wrong-default -> 0 | "
      f"{int(test.c1_right.sum())} correct-default -> 1")


test fires: 0 wrong-default -> 0 | 0 correct-default -> 1


In [18]:
# ---- Layer 1g: BCS answer-key tiers (v6.8) -----------------------------------
BCS_GOLD_PAIRS, BCS_DIST_PAIRS = set(), set()
BCS_TERM_GOLD, BCS_TERM_DIST = {}, {}

for df in (labeled, test):
    for c in ("bcs_gold", "bcs_dist", "bcs_tgold", "bcs_tdist"):
        df[c] = False

if BCS_RAW:
    for q, gold, dist in BCS_RAW:
        qn = normalize(q)
        BCS_GOLD_PAIRS.add((qn, norm_ans(gold)))
        for d in dist:
            BCS_DIST_PAIRS.add((qn, norm_ans(d)))
        m = QUOTE_RE.search(nfc(q))          # quote-only: never substring-match a term
        if m:
            t = normalize(m.group(1))
            if len(t) >= 3:
                BCS_TERM_GOLD.setdefault(t, set()).add(gold)
                BCS_TERM_DIST.setdefault(t, set()).update(dist)
    BCS_DIST_PAIRS -= BCS_GOLD_PAIRS
    # a term whose answer key conflicts across exams is untrustworthy -> drop it
    _amb = {t for t, g in BCS_TERM_GOLD.items() if len({norm_ans(x) for x in g}) > 1}
    for t in _amb:
        BCS_TERM_GOLD.pop(t, None); BCS_TERM_DIST.pop(t, None)
    for t, ds in BCS_TERM_DIST.items():
        BCS_TERM_DIST[t] = {d for d in ds
                            if norm_ans(d) not in {norm_ans(g) for g in BCS_TERM_GOLD.get(t, ())}}
    print(f"BCS tiers: {len(BCS_GOLD_PAIRS)} gold pairs | {len(BCS_DIST_PAIRS)} distractor pairs "
          f"| {len(BCS_TERM_GOLD)} quoted terms ({len(_amb)} ambiguous dropped)")

    def _bcs_term(prompt):
        m = QUOTE_RE.search(prompt)
        if not m:
            return None
        t = normalize(m.group(1))
        return t if t in BCS_TERM_GOLD else None

    for df in (labeled, test):
        keys = list(zip(df.p_norm, df.response_bn.map(norm_ans)))
        df["bcs_gold"] = [k in BCS_GOLD_PAIRS for k in keys]
        df["bcs_dist"] = [k in BCS_DIST_PAIRS for k in keys]
        terms = [_bcs_term(p) for p in df.prompt_bn]
        df["bcs_tgold"] = [bool(t) and contains_gold(r, p, BCS_TERM_GOLD[t])
                           for t, p, r in zip(terms, df.prompt_bn, df.response_bn)]
        df["bcs_tdist"] = [bool(t) and any(norm_ans(r) == norm_ans(o)
                                           for o in BCS_TERM_DIST.get(t, ()))
                           for t, r in zip(terms, df.response_bn)]
        # gold evidence outranks a distractor match
        df.loc[df.bcs_gold, "bcs_dist"] = False
        df.loc[df.bcs_tgold, "bcs_tdist"] = False

    for nm, col, want in (("bcs_gold", "bcs_gold", 1), ("bcs_dist", "bcs_dist", 0),
                          ("bcs_tgold", "bcs_tgold", 1), ("bcs_tdist", "bcs_tdist", 0)):
        n = int(labeled[col].sum())
        if n:
            prec = (labeled.label[labeled[col]] == want).mean()
            warn = "" if prec >= 0.95 else "   <-- WARN: labeled precision < 0.95"
            print(f"  labeled {nm:9s} n={n:2d} precision={prec:.3f}{warn}")
        else:
            print(f"  labeled {nm:9s} 0 fires (applied on test on construction)")
    _tot = int((test.bcs_gold | test.bcs_dist | test.bcs_tgold | test.bcs_tdist).sum())
    _cl = int(((test.bcs_gold | test.bcs_dist | test.bcs_tgold | test.bcs_tdist)
               & ~test.grounded).sum())
    print(f"  test fires: ->1 {int((test.bcs_gold | test.bcs_tgold).sum())} | "
          f"->0 {int((test.bcs_dist | test.bcs_tdist).sum())} | "
          f"{_tot} rows total ({_cl} closed-book)")
else:
    print("BCS answer-key tiers: dataset unavailable - layer off")


BCS tiers: 6537 gold pairs | 19483 distractor pairs | 1276 quoted terms (132 ambiguous dropped)
  labeled bcs_gold  n=15 precision=1.000
  labeled bcs_dist  n= 9 precision=1.000
  labeled bcs_tgold n= 6 precision=1.000
  labeled bcs_tdist n= 3 precision=1.000
  test fires: ->1 154 | ->0 174 | 328 rows total (324 closed-book)


In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load_judge(path, name, device=None):
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side    = "left"   # last-position logits must align across a batch
    tok.truncation_side = "left"   # never cut the answer being judged
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    cfg_path = os.path.join(path, "config.json") if isinstance(path, str) and os.path.isdir(path) else None
    if cfg_path and os.path.exists(cfg_path):
        with open(cfg_path) as f:
            prequant = "quantization_config" in json.load(f)
    else:
        prequant = False
    kw = dict(low_cpu_mem_usage=True,
              device_map=({"": device} if device is not None else "auto"))
    if name == "tiger":
        kw["attn_implementation"] = "eager"
    if not prequant:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    try:
        mdl = AutoModelForCausalLM.from_pretrained(path, **kw)
    except (ValueError, KeyError) as e:
        print(f"{name}: Auto mapping failed -> Gemma3ForConditionalGeneration:", e)
        from transformers import Gemma3ForConditionalGeneration
        mdl = Gemma3ForConditionalGeneration.from_pretrained(path, **kw)
    mdl.eval()
    print(f"{name}: {mdl.__class__.__name__} | prequant={prequant} | "
          f"{mdl.get_memory_footprint()/1e9:.1f} GB")
    return mdl, tok

N_GPU = torch.cuda.device_count()
model, tokenizer = load_judge(MODEL_PATH, "tiger", device=0 if (NEED_QWEN and N_GPU > 1) else None)

EOT_ID   = tokenizer.convert_tokens_to_ids("<end_of_turn>")
STOP_IDS = [tokenizer.eos_token_id, EOT_ID]

qwen = qwen_tok = None
if NEED_QWEN:
    qwen, qwen_tok = load_judge(QWEN_PATH, "qwen", device=1 if N_GPU > 1 else 0)
    if N_GPU == 1:
        print("WARNING: both judges on one GPU - expect small batches")

config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

qwen: Qwen3_5ForCausalLM | prequant=False | 7.5 GB


In [20]:
INSTR_GROUNDED = (
    'তুমি একজন নির্ভুল তথ্য-যাচাইকারী। নিচে একটি অনুচ্ছেদ, একটি প্রশ্ন এবং একটি প্রদত্ত উত্তর দেওয়া আছে। '
    'শুধুমাত্র অনুচ্ছেদের তথ্যের ভিত্তিতে বিচার করো প্রদত্ত উত্তরটি সঠিক কি না। '
    'ভুল উত্তর প্রায়ই সঠিক উত্তরের খুব কাছাকাছি হয় — নাম, সাল, সংখ্যা বা তারিখ সামান্য বদলানো থাকে; '
    'তাই প্রতিটি খুঁটিনাটি অনুচ্ছেদের সাথে মিলিয়ে দেখো। '
    'শুধু "হ্যাঁ" (সঠিক) অথবা "না" (ভুল) লিখে উত্তর দাও।'
)
INSTR_CLOSED = (
    'তুমি একজন নির্ভুল তথ্য-যাচাইকারী। নিচে একটি প্রশ্ন এবং একটি প্রদত্ত উত্তর দেওয়া আছে। '
    'তোমার জ্ঞান ব্যবহার করে বিচার করো প্রদত্ত উত্তরটি তথ্যগতভাবে সঠিক কি না। '
    'ভুল উত্তর প্রায়ই সঠিক উত্তরের খুব কাছাকাছি হয় — নাম, সাল বা সংখ্যা সামান্য বদলানো থাকে। '
    'প্রশ্নগুলো সাধারণত বাংলাদেশের প্রেক্ষাপটের; বাংলাদেশি প্রেক্ষাপটের উত্তর ও বৈশ্বিক উত্তর আলাদা হলে '
    'বাংলাদেশি প্রেক্ষাপট ধরেই বিচার করো। '
    'শুধু "হ্যাঁ" (সঠিক) অথবা "না" (ভুল) লিখে উত্তর দাও।'
)
QUESTION_SUFFIX = 'উত্তরটি কি সঠিক? শুধু "হ্যাঁ" অথবা "না" লিখো।'

REF_NOTE = (
    'নির্ভরযোগ্য সূত্র থেকে প্রাপ্ত রেফারেন্স উত্তর: {refs}\n'
    'প্রদত্ত উত্তরটি রেফারেন্স উত্তরের সাথে মিলিয়ে দেখো — বানান, রূপভেদ, সংক্ষিপ্ত রূপ বা '
    'সংখ্যার ফরম্যাট আলাদা হলেও অর্থ একই হলে উত্তরটি সঠিক; অর্থ আলাদা হলে ভুল।'
)

FEWSHOT_GROUNDED = [
    ('ক্যারেবীয় দ্বীপপুঞ্জের অধিকাংশ অংশই ক্যারিবীয় প্লেটে অবস্থিত, এই অঞ্চল ৭০০-এর অধিক দ্বীপপুঞ্জ, '
     'ক্ষুদে দ্বীপ, প্রবাল প্রাচীর ও বেলে প্রাচীর দ্বারা গঠিত। ক্যারিবীয় অঞ্চল উত্তরে গ্র্যাটার এন্টিলস '
     'এবং দক্ষিণে লেসার এন্টিলস দ্বারা বিভাজিত।',
     'ক্যারেবীয় মোট কয়টি দ্বীপপুঞ্জের সমন্বয়ে গঠিত ?',
     '৭০০-এর অধিক', 'হ্যাঁ'),
    ('১৫৬৬ সালের ৫ই সেপ্টেম্বর, সুলাইমান, তিনি তখন হাঙ্গেরি অভিযানের নেতৃত্ব দেয়ার উদ্দেশ্যে '
     'কনস্টান্টিনোপল হতে রওয়ানা হয়েছিলেন, তিনি হাঙ্গেরিতে যিগেটভারের যুদ্ধে অটোম্যান বিজয়ের পূর্বেই '
     'মারা যান এবং প্রধান উজির ফিরে যাবার সময় দ্বিতীয় সেলিমের অভিষেকের সুবিধার্থে তার মৃত্যুর খবর গোপন রাখেন।',
     'প্রথম সুলাইমানের পর কে উসমানীয় সাম্রাজ্যের সম্রাট হন ?',
     'প্রথম সুলাইমানের পর উসমানীয় সাম্রাজ্যের সম্রাট হন তৃতীয় মুরাদ।', 'না'),
]
FEWSHOT_CLOSED = [
    ('', 'এডেন কোন দেশের সমুদ্রবন্দর?', 'ইয়েমেন', 'হ্যাঁ'),
    ('', 'এডেন কোন দেশের সমুদ্রবন্দর?', 'কাতার',  'না'),
]

def row_block(context, prompt, response, ref=""):
    parts = []
    if context:
        parts.append("অনুচ্ছেদ: " + context[:MAX_CTX_CHARS])
    parts.append("প্রশ্ন: " + prompt)
    parts.append("প্রদত্ত উত্তর: " + response)
    if ref:
        parts.append(REF_NOTE.format(refs=ref))
    parts.append(QUESTION_SUFFIX)
    return "\n\n".join(parts)

def build_messages(context, prompt, response, ref=""):
    grounded = bool(context)
    instr = INSTR_GROUNDED if grounded else INSTR_CLOSED
    shots = FEWSHOT_GROUNDED if grounded else FEWSHOT_CLOSED
    msgs = []
    for i, (c, q, a, verdict) in enumerate(shots):
        user = row_block(c, q, a)
        if i == 0:
            user = instr + "\n\n" + user
        msgs.append({"role": "user", "content": user})
        msgs.append({"role": "assistant", "content": verdict})
    msgs.append({"role": "user", "content": row_block(context, prompt, response, ref)})
    return msgs

def manual_gemma(messages):
    """Hand-built Gemma chat format (fallback when no chat template is exposed)."""
    out = ""
    for m in messages:
        role = "model" if m["role"] == "assistant" else "user"
        out += f"<start_of_turn>{role}\n{m['content']}<end_of_turn>\n"
    return out + "<start_of_turn>model\n"

try:
    _probe = tokenizer.apply_chat_template(
        [{"role": "user", "content": "hi"}], tokenize=False, add_generation_prompt=True)
    HAS_TEMPLATE = "<start_of_turn>" in _probe
except Exception:
    HAS_TEMPLATE = False
print("using repo chat template:", HAS_TEMPLATE)

def render_chat(msgs):
    if HAS_TEMPLATE:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        if tokenizer.bos_token and text.startswith(tokenizer.bos_token):
            text = text[len(tokenizer.bos_token):]   # tokenizer re-adds <bos>
        return text
    return manual_gemma(msgs)

def build_prompt(context, prompt, response, ref=""):
    return render_chat(build_messages(context, prompt, response, ref))

YES_ID = tokenizer.encode("হ্যাঁ", add_special_tokens=False)[0]
NO_ID  = tokenizer.encode("না",   add_special_tokens=False)[0]
assert YES_ID != NO_ID, "verdict words share a first token - pick different verbalizers"
print(f"YES token {YES_ID} ({tokenizer.decode([YES_ID])!r}) | NO token {NO_ID} ({tokenizer.decode([NO_ID])!r})")


# ---- v6: English-instruction cross-lingual prompts (scored on Qwen) ----------
# Content stays Bengali; the judging frame is English. Multilingual models are
# markedly more reliable when the task instruction and verdict space are English
# (Sain 2024; organizers' cross-lingual hint).
EN_INSTR_GROUNDED = (
    "You are a meticulous fact-checker. Below is a passage in Bengali, a question in Bengali, "
    "and a candidate answer. Judge ONLY from the passage whether the candidate answer is correct. "
    "Wrong answers are often very close to the right one - a name, year, number or date slightly "
    "altered - so verify every detail against the passage. Reply with exactly one word: "
    '"Yes" (correct) or "No" (wrong).'
)
EN_INSTR_CLOSED = (
    "You are a meticulous fact-checker. Below is a question in Bengali and a candidate answer. "
    "Silently translate them to English, recall the relevant fact, then judge whether the "
    "candidate answer is factually correct. The questions are usually set in a Bangladeshi "
    "context; when the Bangladeshi answer differs from the global default, judge by the "
    "Bangladeshi context. Wrong answers are often very close to the right one - a name, year "
    'or number slightly altered. Reply with exactly one word: "Yes" (correct) or "No" (wrong).'
)
EN_SUFFIX = 'Is the candidate answer correct? Reply "Yes" or "No" only.'

def row_block_en(context, prompt, response):
    parts = []
    if context:
        parts.append("Passage (Bengali): " + context[:MAX_CTX_CHARS])
    parts.append("Question (Bengali): " + prompt)
    parts.append("Candidate answer: " + response)
    parts.append(EN_SUFFIX)
    return "\n\n".join(parts)

def build_messages_en(context, prompt, response, ref=""):
    grounded = bool(context)
    instr = EN_INSTR_GROUNDED if grounded else EN_INSTR_CLOSED
    shots = FEWSHOT_GROUNDED if grounded else FEWSHOT_CLOSED
    msgs = []
    for i, (c, q, a, verdict) in enumerate(shots):
        user = row_block_en(c, q, a)
        if i == 0:
            user = instr + "\n\n" + user
        msgs.append({"role": "user", "content": user})
        msgs.append({"role": "assistant", "content": "Yes" if verdict == "হ্যাঁ" else "No"})
    msgs.append({"role": "user", "content": row_block_en(context, prompt, response)})
    return msgs


using repo chat template: True
YES token 194443 ('হ্যা') | NO token 4072 ('না')


In [21]:
QWEN_YES = QWEN_NO = None
if NEED_QWEN:
    _think_kw = True
    try:
        qwen_tok.apply_chat_template([{"role": "user", "content": "hi"}],
                                     tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False)
    except TypeError:
        _think_kw = False
        print("qwen template lacks enable_thinking kwarg - using template default")

    def qwen_render(msgs):
        if _think_kw:
            return qwen_tok.apply_chat_template(msgs, tokenize=False,
                                                add_generation_prompt=True,
                                                enable_thinking=False)
        return qwen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    QWEN_YES = qwen_tok.encode("হ্যাঁ", add_special_tokens=False)[0]
    QWEN_NO  = qwen_tok.encode("না",   add_special_tokens=False)[0]
    assert QWEN_YES != QWEN_NO, "verdict words collide on first token in Qwen's vocab"
    print(f"qwen verdict ids {QWEN_YES}/{QWEN_NO} "
          f"({qwen_tok.decode([QWEN_YES])!r} / {qwen_tok.decode([QWEN_NO])!r})")


# ---- v6: English verdict tokens for the cross-lingual pass -------------------
QWEN_YES_EN = QWEN_NO_EN = None
if USE_XLING_EN and qwen_tok is not None:
    QWEN_YES_EN = qwen_tok.encode("Yes", add_special_tokens=False)[0]
    QWEN_NO_EN  = qwen_tok.encode("No",  add_special_tokens=False)[0]
    assert QWEN_YES_EN != QWEN_NO_EN
    print(f"qwen EN verdict ids {QWEN_YES_EN}/{QWEN_NO_EN} "
          f"({qwen_tok.decode([QWEN_YES_EN])!r} / {qwen_tok.decode([QWEN_NO_EN])!r})")


qwen verdict ids 151318/83558 ('হ' / 'ন')
qwen EN verdict ids 9175/2665 ('Yes' / 'No')


In [22]:
from tqdm.auto import tqdm

def _ctx_of(r):
    """ctx_override column wins when present (may be '' -> closed-book prompt);
    NaN/absent means 'use the row's own context'."""
    co = getattr(r, "ctx_override", None)
    if co is None or (isinstance(co, float) and math.isnan(co)):
        return r.context
    return co

@torch.inference_mode()
def score_with(df, mdl, tok, render, yes_id, no_id, desc="scoring", builder=None):
    builder = builder or build_messages
    prompts = [render(builder(_ctx_of(r), r.prompt_bn, r.response_bn,
                              getattr(r, "ref_text", "") or ""))
               for r in df.itertuples()]
    probs = np.zeros(len(prompts), dtype=np.float64)
    start, bs = 0, BATCH_SIZE
    pbar = tqdm(total=len(prompts), desc=desc, leave=False)
    while start < len(prompts):
        batch = prompts[start:start + bs]
        try:
            enc = tok(batch, return_tensors="pt", padding=True,
                      truncation=True, max_length=MAX_LEN)
            enc = {k: v.to(mdl.device) for k, v in enc.items()}
            logits = mdl(**enc, logits_to_keep=1, use_cache=False).logits[:, -1, :]
            pair = logits[:, [yes_id, no_id]].float().softmax(dim=-1)
            probs[start:start + len(batch)] = pair[:, 0].cpu().numpy()
            start += len(batch)
            pbar.update(len(batch))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            print(f"{desc}: OOM -> batch {bs}")
    pbar.close()
    torch.cuda.empty_cache()
    return probs

def score_tiger(df, desc="tiger"):
    return score_with(df, model, tokenizer, render_chat, YES_ID, NO_ID, desc)

def score_qwen(df, desc="qwen"):
    return score_with(df, qwen, qwen_tok, qwen_render, QWEN_YES, QWEN_NO, desc)

def score_qwen_en(df, desc="qwen-en"):
    return score_with(df, qwen, qwen_tok, qwen_render, QWEN_YES_EN, QWEN_NO_EN,
                      desc, builder=build_messages_en)

W_TIGER     = 1.0     # 2-judge blend weight, chosen in section 7
ENSEMBLE_ON = False   # set in section 7
STACK_ON    = False   # any track using the stack (set in section 7)
STACK_ON_TRACK = {"grounded": False, "closed": False}   # v6.6: per-track selection
META        = {"grounded": "blend", "closed": "blend"}  # v6.7: chosen meta-judge per track
BLEND3_W    = {}      # v6.7: per-track 3-way judge weights {"p_t":..,"p_q":..,"p_qe":..}
STACK       = {}      # per-track: {"grounded": (model, cols, thr), "closed": (...)}

def score_blend(df, desc="blend"):
    """v5.1 2-judge blend - still used for the re-judge passes (rel-route /
    retrieval), whose overridden contexts invalidate the clf/EN features.
    2xT4: this blend needs only p_t + p_q (never p_qe), so Tiger (GPU0) and Qwen
    (GPU1) can run concurrently -> wall time drops from 2 passes to 1."""
    if (ENSEMBLE_ON and qwen is not None and torch.cuda.device_count() > 1
            and qwen.device != model.device):
        _pt, _pq, _ = score_judges(df, desc, qe_rows=np.zeros(len(df), dtype=bool))
        if _pq is None:
            return _pt
        return W_TIGER * _pt + (1.0 - W_TIGER) * _pq
    p = score_tiger(df, desc + "-t")
    if ENSEMBLE_ON:
        p = W_TIGER * p + (1.0 - W_TIGER) * score_qwen(df, desc + "-q")
    return p

def judge_frame(df, p_t, p_q=None, p_qe=None):
    """Assemble the meta-judge feature frame for a scored row set."""
    F = pd.DataFrame({"p_t": p_t}, index=df.index)
    if p_q is not None:
        F["p_q"] = p_q
    if p_qe is not None:
        F["p_qe"] = p_qe
    if CLF_OK:
        F["p_clf"] = df.p_clf.values
    # v6.3: cheap register-robust lexical features (starter-notebook model C);
    # they enter the same CV-gated stack, so they can only help
    rt = [{t[:4] for t in bm25_tokenize(r)} for r in df.response_bn]
    ct = [{t[:4] for t in bm25_tokenize(p + " " + (c or ""))}
          for p, c in zip(df.prompt_bn, df.context)]
    F["lex_jac"] = [len(a & b) / max(1, len(a | b)) for a, b in zip(rt, ct)]
    F["r_len"] = df.response_bn.str.len().values / 100.0
    # v6.4: grounded context-verification features (0 for closed rows) - the stack
    # gates them, so they only help where a passage exists
    if "ctx_cov" in df.columns:
        F["ctx_cov"] = np.where(df.grounded.values, df.ctx_cov.values, 1.0)
        F["ctx_numbad"] = np.where(df.grounded.values, df.ctx_num_bad.values.astype(float), 0.0)
    return F

def stack_prob(track, F):
    mdl, cols, _ = STACK[track]
    return mdl.predict_proba(F[cols].values)[:, 1]

def blend3_prob(track, p_t, p_q=None, p_qe=None):
    """v6.7: per-track linear blend over the three judge signals. Unlike the global
    2-way blend it can differ per track AND can use the English cross-lingual pass -
    which the closed track (the bottleneck) otherwise never sees."""
    w = BLEND3_W[track]
    p = w.get("p_t", 0.0) * p_t
    if p_q is not None and "p_q" in w:
        p = p + w["p_q"] * p_q
    if p_qe is not None and "p_qe" in w:
        p = p + w["p_qe"] * p_qe
    return p

def cached_scores(df, path, desc):
    """Score tiger (+qwen bn/en if enabled), caching RAW per-model scores so blend
    weights / the stacker can change without rescoring. Rescored if rows changed."""
    cache = None
    if os.path.exists(path):
        cache = pd.read_parquet(path)
        if set(cache.id) != set(df.id):
            print(f"{os.path.basename(path)}: row set changed - rescoring")
            cache = None
    if cache is None:
        _pt, _pq, _pqe = score_judges(df, desc)          # 2xT4: Tiger || Qwen
        cache = pd.DataFrame({"id": df.id.values, "p_t": _pt})
        if _pq is not None:
            cache["p_q"] = _pq
        if _pqe is not None and not np.isnan(_pqe).all():
            cache["p_qe"] = _pqe
        cache.to_parquet(path)
    return cache


# ---- v6.2: batched Qwen generation (used by math CoT + translate-verify) ------
@torch.inference_mode()
def _gen_qwen_batch(users, max_new_tokens, n=1, temperature=0.7, bs=None, desc=None):
    """Generate for many user prompts at once; returns a list of n-reply lists.
    Left padding is already set on qwen_tok. Batch size halves on OOM."""
    bs = bs or GEN_BATCH
    texts = [qwen_render([{"role": "user", "content": u}]) for u in users]
    replies, lo = [], 0
    pbar = tqdm(total=len(texts), desc=desc, leave=False) if desc else None
    while lo < len(texts):
        chunk = texts[lo:lo + bs]
        try:
            enc = qwen_tok(chunk, return_tensors="pt", padding=True).to(qwen.device)
            kw = dict(max_new_tokens=max_new_tokens,
                      pad_token_id=qwen_tok.pad_token_id or qwen_tok.eos_token_id)
            if n > 1:
                kw.update(do_sample=True, temperature=temperature, top_p=0.9,
                          num_return_sequences=n)
            else:
                kw.update(do_sample=False)
            out = qwen.generate(**enc, **kw)
            dec = qwen_tok.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                        skip_special_tokens=True)
            for i in range(len(chunk)):
                replies.append(dec[i * n:(i + 1) * n])
            lo += len(chunk)
            if pbar:
                pbar.update(len(chunk))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            print(f"qwen gen: OOM -> batch {bs}")
    if pbar:
        pbar.close()
    torch.cuda.empty_cache()
    return replies


# ---- 2xT4: concurrent judge scoring across both GPUs --------------------------
# Tiger is loaded on GPU0 and Qwen on GPU1 (see load_judge), but they were scored
# SEQUENTIALLY - so one T4 idled while the other worked. score_judges runs Tiger
# (GPU0) and Qwen's Bengali+English passes (GPU1) in two threads: CUDA launches are
# async and PyTorch drops the GIL inside the forward pass, so both cards run at once.
# NOTE: nn.DataParallel is NOT applicable here - it splits ONE model's batch across
# GPUs; we have two DIFFERENT models already pinned to separate devices.
# Falls back to sequential on a single GPU / when Qwen is off. qe_rows scores the
# English pass on a subset only (the closed track's blend3 often ignores p_qe).
from concurrent.futures import ThreadPoolExecutor as _TPE

def _score_qe_subset(df, qe_rows, desc):
    if qe_rows is None:
        return score_qwen_en(df, desc)
    mask = np.asarray(qe_rows, dtype=bool)
    out = np.full(len(df), np.nan, dtype=np.float64)
    idx = np.flatnonzero(mask)
    if len(idx):
        out[idx] = score_qwen_en(df.iloc[idx], desc)
    return out

def score_judges(df, desc="score", qe_rows=None):
    """(p_t, p_q|None, p_qe|None) - Tiger and Qwen scored concurrently on 2 GPUs."""
    do_q  = (qwen is not None) and USE_ENSEMBLE
    do_qe = (qwen is not None) and USE_XLING_EN
    res = {}
    def _run_tiger():
        with torch.cuda.device(model.device):
            res["p_t"] = score_tiger(df, desc + "-t")
    def _run_qwen():
        with torch.cuda.device(qwen.device if qwen is not None else model.device):
            if do_q:
                res["p_q"] = score_qwen(df, desc + "-q")
            if do_qe:
                res["p_qe"] = _score_qe_subset(df, qe_rows, desc + "-qe")
    parallel = (torch.cuda.device_count() > 1 and qwen is not None
                and (do_q or do_qe) and qwen.device != model.device)
    if parallel:
        with _TPE(max_workers=2) as _ex:
            _fa = _ex.submit(_run_tiger); _fb = _ex.submit(_run_qwen)
            _fa.result(); _fb.result()
    else:
        _run_tiger(); _run_qwen()
    return res.get("p_t"), res.get("p_q"), res.get("p_qe")


In [23]:
import sympy

TRIVIA_OVERRIDE = re.compile(r'সালে|সাল\s|প্রেসিডেন্ট|বিশ্বকাপ|আদমশুমারি|অর্থনৈতিক সমীক্ষা|সুনামি|জনশুমারি')
# v6.15 BUGFIX: `\*\*` matched trailing markdown bold ("...পুরস্কার পান-**"),
# routing pure GK questions (Einstein's Nobel, blood vessels, protein) into the
# MATH path - so a biology question got a math verdict. Require digits on BOTH
# sides so a real exponent (2**3) still matches but '-**' / '**৫' (bold) does not.
MATH_OPERATORS  = re.compile(r'[=+×÷√^]|\d\s*\*\*\s*\d|\bx\s*এর মান')
# v6.15: families is_math never recognised. Work-rate word problems
# ("রহিম একা একটি কাজ ৪২ দিনে...") carry no operator and no old keyword, so 23 real
# math rows were judged as ordinary trivia by the closed blend - which is 80% Tiger,
# and Tiger is the WORST math judge in BenHalluEval (BHS 53.45, 85.5% miss rate).
# Geometry is added too: it was only reaching math via the buggy `**` above.
MATH_KEYWORDS   = re.compile(r'সম্ভাবনা|শতকরা|সুদ|অনুপাত|গুণিতক|মৌলিক সংখ্যা|ধারাটির|সেট\s*A|ক্ষতি|লাভ.*শতকরা|উচ্চতা.*হলে|গ\s*\.?\s*সা\s*\.?\s*গু|ল\s*\.?\s*সা\s*\.?\s*গু'
                             r'|কাজ(?:টি)?\s*\S*\s*দিনে|একা\s*\S*\s*দিনে|দুজনে\s*একসাথে|একত্রে\s*কাজ'
                             r'|ত্রিভুজ|বৃত্ত|ক্ষেত্রফল|পরিসীমা|ব্যাসার্ধ')

def is_math(prompt):
    if GRAMMAR_Q.search(prompt):        # v4: idiom/grammar questions are never math
        return False
    if TRIVIA_OVERRIDE.search(prompt):
        return False
    return bool(MATH_OPERATORS.search(prompt) or MATH_KEYWORDS.search(prompt))

def bn2en(s):
    return nfc(s).translate(BN_DIGITS)

def first_number(s):
    m = re.search(r"-?\d+(?:\.\d+)?", bn2en(s).replace(",", ""))
    return float(m.group()) if m else None

EXPR_RE  = re.compile(r"[0-9][0-9+\-*/×÷^().\s]{2,}[0-9)]")
RATIO_RE = re.compile(r"(\d+)\s*[:ঃ]\s*(\d+)")
SUM_RE   = re.compile(r"(?:সমষ্টি|মোট|যোগফল)\D{0,20}?(\d+(?:\.\d+)?)")
# v4 templates
PCT_RE   = re.compile(r"(\d+(?:\.\d+)?)\s*(?:এর|টাকার)\s*(\d+(?:\.\d+)?)\s*(?:%|শতাংশ)")
PCT_VETO = re.compile(r"বৃদ্ধি|হ্রাস|কমে|বেড়ে|ছাড়|কমিশন|লাভ|ক্ষতি|সুদ")
GCD_RE   = re.compile(r"গ\s*\.?\s*সা\s*\.?\s*গু")
LCM_RE   = re.compile(r"ল\s*\.?\s*সা\s*\.?\s*গু")

def eval_expr(prompt):
    p = bn2en(prompt).replace("×", "*").replace("÷", "/").replace("^", "**")
    m = EXPR_RE.search(p)
    if not m:
        return None
    frag = m.group().strip()
    if not re.search(r"[+\-*/]", frag):
        return None
    try:
        return float(sympy.sympify(frag, rational=False))
    except Exception:
        return None

def sympy_verdict(prompt, response):
    """1 / 0 / None (abstain). Only fires on clean parses."""
    r = first_number(response)
    if r is None:
        return None
    v = eval_expr(prompt)
    if v is not None:
        return 1 if abs(v - r) < 1e-6 else 0
    p = bn2en(prompt)
    # v4: percentage template ("X এর Y%") - vetoed if change/interest wording present
    m = PCT_RE.search(p)
    if m and not PCT_VETO.search(p):
        v = float(m.group(1)) * float(m.group(2)) / 100.0
        return 1 if abs(v - r) < 1e-6 else 0
    # v4: gcd / lcm on the listed integers
    if (GCD_RE.search(p) or LCM_RE.search(p)) and "অনুপাত" not in p:
        nums = [int(x) for x in re.findall(r"\d+", p) if int(x) > 0]
        if 2 <= len(nums) <= 4:
            v = float(sympy.igcd(*nums)) if GCD_RE.search(p) else float(sympy.ilcm(*nums))
            return 1 if abs(v - r) < 1e-6 else 0
        return None
    rm, sm = RATIO_RE.search(p), SUM_RE.search(p)
    if rm and sm:
        a, b, S = float(rm.group(1)), float(rm.group(2)), float(sm.group(1))
        if a + b == 0:
            return None
        pa, pb = S * a / (a + b), S * b / (a + b)
        if not any(abs(r - x) < 1e-6 for x in (pa, pb)):
            return 0
        if re.search(r"ছোট|কম", p):
            return 1 if abs(r - min(pa, pb)) < 1e-6 else 0
        if re.search(r"বড়|বেশি", p):
            return 1 if abs(r - max(pa, pb)) < 1e-6 else 0
    return None


In [24]:
QA1_OK = QA0_OK = False
labeled["qa_match"] = np.nan; labeled["qa_span"] = ""
test["qa_match"]    = np.nan; test["qa_span"]    = ""

if USE_QA_VERIFY:
    from transformers import AutoModelForQuestionAnswering
    from difflib import SequenceMatcher

    qa_tok   = AutoTokenizer.from_pretrained(QA_PATH)
    qa_model = AutoModelForQuestionAnswering.from_pretrained(
        QA_PATH, torch_dtype=torch.float16).to("cuda:0").eval()

    @torch.inference_mode()
    def qa_span(question, context, max_len=384, stride=128, max_ans=30):
        enc = qa_tok(question, context, truncation="only_second", max_length=max_len,
                     stride=stride, return_overflowing_tokens=True,
                     return_offsets_mapping=True, padding="max_length", return_tensors="pt")
        offsets = enc.pop("offset_mapping")
        enc.pop("overflow_to_sample_mapping", None)
        seq_ids = [enc.sequence_ids(i) for i in range(enc["input_ids"].shape[0])]
        enc = {k: v.to("cuda:0") for k, v in enc.items()}
        out = qa_model(**enc)
        best_text, best_score, min_null = "", -1e9, 1e9
        for i in range(out.start_logits.shape[0]):
            sl = out.start_logits[i].float().cpu().numpy()
            el = out.end_logits[i].float().cpu().numpy()
            min_null = min(min_null, float(sl[0] + el[0]))
            mask = np.array([sid != 1 for sid in seq_ids[i]])
            sl2, el2 = sl.copy(), el.copy()
            sl2[mask] = -1e9; el2[mask] = -1e9
            for s_ in np.argsort(sl2)[-10:][::-1]:
                for e_ in np.argsort(el2)[-10:][::-1]:
                    if e_ < s_ or e_ - s_ + 1 > max_ans:
                        continue
                    sc = float(sl2[s_] + el2[e_])
                    if sc > best_score:
                        off = offsets[i]
                        best_score = sc
                        best_text = context[int(off[s_][0]):int(off[e_][1])]
        found = bool(best_text) and best_score > min_null
        return (best_text if found else ""), found

    def qa_match_score(span, response):
        a, b = norm_ans(span), norm_ans(response)
        if not a or not b:
            return 0.0
        if a == b or a in b or b in a:
            return 1.0
        na, nb = first_number(a), first_number(b)
        if na is not None and nb is not None:
            return 1.0 if abs(na - nb) < 1e-6 else 0.0
        return SequenceMatcher(None, a, b).ratio()

    for df, dname in ((labeled, "labeled"), (test, "test")):
        g_idx = df.index[df.grounded] if dname == "labeled" else df.index[df.grounded & (df.ref_text == "")]
        for i in tqdm(g_idx, desc=f"qa {dname}"):
            span, found = qa_span(df.at[i, "prompt_bn"], df.at[i, "context"][:2000])
            if found:
                df.at[i, "qa_span"]  = span
                df.at[i, "qa_match"] = qa_match_score(span, df.at[i, "response_bn"])
    print("qa spans found: labeled", int(labeled.qa_match.notna().sum()),
          "| test", int(test.qa_match.notna().sum()))

def qa_tier1_mask(df):   # span found, response matches it strongly -> candidate label 1
    return df.qa_match.notna() & (df.qa_match >= 0.95)

def qa_tier0_mask(df):   # span found, numeric contradiction with response -> candidate label 0
    num_span = df.qa_span.map(lambda s: first_number(s) is not None)
    num_resp = df.response_bn.map(lambda s: first_number(s) is not None)
    return df.qa_match.notna() & (df.qa_match == 0.0) & num_span & num_resp


In [25]:
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

EXEMPLAR_KEYS = {(normalize(q), normalize(a)) for (_, q, a, _) in FEWSHOT_GROUNDED + FEWSHOT_CLOSED}
val = labeled[[ (p, r) not in EXEMPLAR_KEYS
                for p, r in zip(labeled.p_norm, labeled.r_norm) ]].reset_index(drop=True)
print(f"validation rows: {len(val)} (excluded {len(labeled) - len(val)} few-shot exemplars)")

GRID = np.arange(0.05, 0.96, 0.05)

def f1_hall(y_true, pred):
    return f1_score(y_true, pred, pos_label=0, zero_division=0)

def metric_f1(y_true, pred):     # v6: the scored metric is macro F1
    return f1_score(y_true, pred, average="macro", zero_division=0)

def best_threshold(p_yes, y):
    scores = [metric_f1(y, (p_yes > t).astype(int)) for t in GRID]
    return float(GRID[int(np.argmax(scores))])

def cv_threshold(p_yes, y, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_ts, fold_f1s = [], []
    for tr, va in cv.split(np.zeros(len(y)), y):
        t = best_threshold(p_yes[tr], y[tr])
        fold_ts.append(t)
        fold_f1s.append(metric_f1(y[va], (p_yes[va] > t).astype(int)))
    return float(np.median(fold_ts)), float(np.mean(fold_f1s)), fold_ts

g_mask = val.grounded.values
c_mask = ~g_mask
y = val.label.values

# v6.17: exclude math rows from the CLOSED-track FIT (not from application).
# At test time math rows keep the closed meta-judge verdict only provisionally -
# sympy and the math-CoT pass override most of them - so fitting weights and
# thresholds on the ~13 labeled math rows (9/4 split, judges ~random on math)
# optimises the judge for rows it does not finally decide and drags the closed
# threshold around. Weights/thresholds are fitted on non-math closed rows; math
# rows still RECEIVE the chosen judge's probabilities as their pre-CoT verdict.
math_mask = ((~val.grounded) & val.prompt_bn.map(is_math)).values
c_fit = c_mask & ~math_mask
print(f"closed-track fit rows: {int(c_fit.sum())} "
      f"({int(math_mask.sum())} math rows excluded from fitting, still scored)")

# ---- pass 1: plain prompts, all judge signals (v6.19: guarded cache) ----------
# These 3 judge passes on the 295 labeled rows cost ~31 min and depend ONLY on the
# judge prompts + models + context length - NOT on banks, tiers or thresholds. So we
# cache them and let all the downstream FITTING re-run live. The guard key covers the
# labeled rows, every judge prompt/few-shot, the model ids, MAX_CTX_CHARS/MAX_LEN and
# the USE_ENSEMBLE/USE_XLING_EN flags; any change -> cache miss -> re-score. If you
# edit the scoring FUNCTIONS themselves (row_block/score_with), delete val_scores.json.
import hashlib as _hl
_val_key = _hl.sha256(("|".join((val.p_norm + "\x1f" + val.r_norm).tolist())).encode()).hexdigest()[:16]
_pk_parts = [INSTR_GROUNDED, INSTR_CLOSED, QUESTION_SUFFIX, REF_NOTE,
             EN_INSTR_GROUNDED, EN_INSTR_CLOSED, EN_SUFFIX,
             repr(FEWSHOT_GROUNDED), repr(FEWSHOT_CLOSED),
             str(MODEL_PATH), str(QWEN_PATH), str(MAX_CTX_CHARS), str(MAX_LEN),
             str(bool(USE_ENSEMBLE)), str(bool(USE_XLING_EN))]
_prompt_key = _hl.sha256("\x1e".join(_pk_parts).encode()).hexdigest()[:16]
_VS_LOAD = find_file("val_scores.json")
_VS_SAVE = os.path.join(WORK_DIR, "val_scores.json")

vt = vq = vqe = None
if USE_VAL_SCORE_CACHE and _VS_LOAD:
    try:
        _d = json.load(open(_VS_LOAD, encoding="utf-8"))
        if (_d.get("val_key") == _val_key and _d.get("prompt_key") == _prompt_key
                and _d.get("n") == len(val)):
            vt  = np.asarray(_d["p_t"], dtype=np.float64)
            vq  = np.asarray(_d["p_q"],  dtype=np.float64) if _d.get("p_q")  is not None else None
            vqe = np.asarray(_d["p_qe"], dtype=np.float64) if _d.get("p_qe") is not None else None
            print(f"validation scores: LOADED from cache {_VS_LOAD} - pass-1 scoring skipped")
        else:
            print("validation scores: cache present but guard MISMATCH (prompts/models/rows "
                  "changed) -> re-scoring")
    except Exception as _e:
        print("validation scores: cache read failed -> re-scoring:", _e); vt = None
if vt is None:
    vt, vq, vqe = score_judges(val.assign(ref_text=""), "val")
    # v6.19: ALWAYS save the scores (decoupled from the load flag) so the artifact
    # exists to attach later, even while USE_VAL_SCORE_CACHE is False and validation
    # runs fresh every time. Loading/skipping only happens when the flag is True.
    try:
        json.dump({"val_key": _val_key, "prompt_key": _prompt_key, "n": len(val),
                   "p_t": vt.tolist(),
                   "p_q": (vq.tolist() if vq is not None else None),
                   "p_qe": (vqe.tolist() if vqe is not None else None)},
                  open(_VS_SAVE, "w"))
        print(f"validation scores: SAVED -> {_VS_SAVE}  "
              f"(download + attach as a Kaggle dataset, then set USE_VAL_SCORE_CACHE=True "
              f"to skip ~31 min on the final run)")
    except Exception as _e:
        print("validation scores: save failed (non-fatal):", _e)

def track_fit(p):
    tg, f1g, _ = cv_threshold(p[g_mask], y[g_mask])
    tc, f1c, _ = cv_threshold(p[c_fit], y[c_fit])
    ng, nc = int(g_mask.sum()), int(c_fit.sum())
    return tg, tc, (f1g * ng + f1c * nc) / (ng + nc), f1g, f1c

# v5.1 baseline: tiger only, then the 2-judge blend sweep
tg, tc, best_w_score, f1g, f1c = track_fit(vt)
print(f"w=1.00 (tiger only) | grounded cvF1 {f1g:.4f} | closed cvF1 {f1c:.4f} | weighted {best_w_score:.4f}")
if vq is not None:
    # v6.7: the old grid floored at 0.40 and never tested Qwen-heavy / Qwen-only blends,
    # even though grounded kept improving as tiger's share fell.
    for w in (0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0):
        p_w = w * vt + (1 - w) * vq
        tg_, tc_, s_, f1g_, f1c_ = track_fit(p_w)
        print(f"w={w:.2f}              | grounded cvF1 {f1g_:.4f} | closed cvF1 {f1c_:.4f} | weighted {s_:.4f}")
        if s_ > best_w_score + 0.003:
            W_TIGER, ENSEMBLE_ON = w, True
            tg, tc, best_w_score, f1g, f1c = tg_, tc_, s_, f1g_, f1c_
print(f"blend gate: {('ENABLED w=%.2f' % W_TIGER) if ENSEMBLE_ON else 'off (tiger only)'}")

THRESH_BLEND = {"grounded": tg, "closed": tc}           # blend-scale (re-judge passes)
val_blend = (W_TIGER * vt + (1 - W_TIGER) * vq) if ENSEMBLE_ON else vt

# ---- v6.7: per-track 3-way judge blend (tiger / qwen-bn / qwen-en) ------------
# All three signals are already scored, so this is free. Crucially it lets the CLOSED
# track use p_qe (the English cross-lingual pass) without needing the stack, which the
# closed track rejects. Weights are a simplex grid, thresholds by the same CV routine.
BLEND3 = {}
_sig, _nm = [vt], ["p_t"]
if vq is not None:  _sig.append(vq);  _nm.append("p_q")
if vqe is not None: _sig.append(vqe); _nm.append("p_qe")
if len(_sig) >= 2:
    if len(_sig) == 2:
        _combos = [(a / 10, 1 - a / 10) for a in range(11)]
    else:
        _combos = [(a / 10, b / 10, (10 - a - b) / 10)
                   for a in range(11) for b in range(11 - a)]
    for _tr, _m in (("grounded", g_mask), ("closed", c_fit)):
        _best = None
        for _w in _combos:
            _p = sum(wi * si for wi, si in zip(_w, _sig))
            _t, _f1, _ = cv_threshold(_p[_m], y[_m])
            if _best is None or _f1 > _best[2]:
                _best = (_w, _t, _f1, _p)
        BLEND3[_tr] = _best
        print(f"blend3 {_tr:9s} | w={ {k: round(v,2) for k,v in zip(_nm,_best[0])} } "
              f"| t={_best[1]:.2f} | cv macroF1 {_best[2]:.4f}")

# ---- v6.22: manual blend3 weight override (per track) -------------------------
for _tr_m, _w_m in (BLEND3_MANUAL or {}).items():
    if _tr_m not in BLEND3:
        print(f"blend3 MANUAL: track {_tr_m!r} unavailable - ignored"); continue
    _wvec = tuple(float(_w_m.get(nm, 0.0)) for nm in _nm)
    if abs(sum(_wvec) - 1.0) > 1e-6:
        print(f"blend3 MANUAL {_tr_m}: weights sum to {sum(_wvec):.2f} (not 1.0) - "
              f"threshold refit absorbs the scale, but prefer weights summing to 1")
    _mm = g_mask if _tr_m == "grounded" else c_fit
    _pp = sum(wi * si for wi, si in zip(_wvec, _sig))
    _tt, _ff, _ = cv_threshold(_pp[_mm], y[_mm])
    _auto = BLEND3[_tr_m]
    BLEND3[_tr_m] = (_wvec, _tt, _ff, _pp)
    print(f"blend3 {_tr_m} MANUAL w={dict(zip(_nm, _wvec))} -> t={_tt:.2f} cv {_ff:.4f}  "
          f"(grid best: w={dict(zip(_nm, _auto[0]))} cv {_auto[2]:.4f})")

# ---- v6: per-track stacked meta-judge, gated against the blend ----------------
VF = judge_frame(val, vt, vq, vqe)
STACK_COLS = list(VF.columns)
print("stack features:", STACK_COLS)

def oof_stack(F, yy, C, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(yy))
    for tr, va in cv.split(F, yy):
        m = LogisticRegression(C=C, max_iter=2000).fit(F[tr], yy[tr])
        oof[va] = m.predict_proba(F[va])[:, 1]
    return oof

stack_cv = {}
if len(STACK_COLS) >= 2:
    for track, m_ in (("grounded", g_mask), ("closed", c_fit)):
        F_, y_ = VF.values[m_], y[m_]
        best = None
        for C in (0.1, 0.3, 1.0, 3.0):
            oof = oof_stack(F_, y_, C)
            t_, cvf1, _ = cv_threshold(oof, y_)
            if best is None or cvf1 > best[0]:
                best = (cvf1, C, t_, oof)
        cvf1, C, t_, oof = best
        mdl = LogisticRegression(C=C, max_iter=2000).fit(F_, y_)
        STACK[track] = (mdl, STACK_COLS, t_)
        stack_cv[track] = (cvf1, oof)
        print(f"stack {track:9s} | C={C} | cv macroF1 {cvf1:.4f} | t={t_:.2f} | "
              f"coef {dict(zip(STACK_COLS, np.round(mdl.coef_[0], 2)))}")
    ng, nc = int(g_mask.sum()), int(c_fit.sum())
    stack_score = (stack_cv["grounded"][0] * ng + stack_cv["closed"][0] * nc) / (ng + nc)
    print(f"stack weighted cvF1 {stack_score:.4f} vs blend {best_w_score:.4f} "
          f"(informational - selection is per-track)")
# v6.6/v6.7: PER-TRACK selection among {2-way blend, 3-way blend, stack}. The stack
# can win big on one track and lose on the other (p_clf is trained on grounded QA:
# labeled AUC 0.97 grounded vs 0.58 closed), and the best judge weights differ per
# track too. An all-or-nothing gate throws away real wins on both counts. A challenger
# must beat the 2-way blend baseline by >0.005 on its own track to be adopted.
THRESH, CV_F1 = {}, {}
val_probs = np.zeros(len(val))
for _tr, _m, _fm, _b2 in (("grounded", g_mask, g_mask, f1g),
                          ("closed", c_mask, c_fit, f1c)):
    _opts = [("blend", _b2, THRESH_BLEND[_tr], val_blend)]
    if _tr in BLEND3:
        _opts.append(("blend3", BLEND3[_tr][2], BLEND3[_tr][1], BLEND3[_tr][3]))
    if _tr in stack_cv:
        _full = np.zeros(len(val)); _full[_fm] = stack_cv[_tr][1]   # honest OOF probs
        _ex = _m & ~_fm      # math rows: excluded from the fit, so direct model
        if _ex.any():        # predictions on them are out-of-sample = honest
            _full[_ex] = stack_prob(_tr, VF[_ex])
        _opts.append(("stack", stack_cv[_tr][0], STACK[_tr][2], _full))
    if _tr in (BLEND3_MANUAL or {}) and _tr in BLEND3:   # v6.22: manual weights win
        _name, _f1, _thr, _p = "blend3", BLEND3[_tr][2], BLEND3[_tr][1], BLEND3[_tr][3]
    else:
        _name, _f1, _thr, _p = max(_opts, key=lambda o: o[1])
        if _name != "blend" and _f1 <= _b2 + 0.005:      # margin over the incumbent
            _name, _f1, _thr, _p = _opts[0]
    META[_tr] = _name
    THRESH[_tr], CV_F1[_tr] = _thr, _f1
    val_probs[_m] = _p[_m]
    STACK_ON_TRACK[_tr] = (_name == "stack")
    if _name == "blend3":
        BLEND3_W[_tr] = {k: v for k, v in zip(_nm, BLEND3[_tr][0]) if v > 0}
STACK_ON = any(STACK_ON_TRACK.values())
_expected = (CV_F1["grounded"] * int(g_mask.sum()) + CV_F1["closed"] * int(c_mask.sum())) / len(val)
print(f"meta-judge per track: {META} | thresholds {THRESH}")
print(f"  blend3 weights used: {BLEND3_W if BLEND3_W else 'none'}")
print(f"  selected weighted cvF1 {_expected:.4f}  (blend-everywhere was {best_w_score:.4f})")

# ---- v6.12: F1(label=0) threshold refinement (CV-gated) -------------------------
# The organizers' Phase-2 preview states the scored metric is BINARY F1 ON THE
# HALLUCINATED CLASS (label=0). Scorer selection above stays macro-tuned (a good
# regularizer on 299 rows), but the decision threshold is the one knob that trades
# the scored metric directly. Per track: CV-compare the macro-chosen threshold vs
# an F1(0)-refit threshold UNDER CV F1(label=0); adopt only on a >=0.005 gain.
GRID_T = np.arange(0.05, 0.96, 0.05)

def _cv_f1h_at(p, yy, t_fixed=None, n_splits=5, seed=42):
    """Mean fold F1(label=0); threshold refit per fold on F1(0) unless fixed."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    f1s, ts = [], []
    for tr, va in cv.split(np.zeros(len(yy)), yy):
        if t_fixed is None:
            t_ = float(GRID_T[int(np.argmax([f1_hall(yy[tr], (p[tr] > t).astype(int))
                                             for t in GRID_T]))])
        else:
            t_ = t_fixed
        ts.append(t_)
        f1s.append(f1_hall(yy[va], (p[va] > t_).astype(int)))
    return float(np.mean(f1s)), float(np.median(ts))

for _tr, _m in (("grounded", g_mask), ("closed", c_fit)):
    _pv, _yv = val_probs[_m], y[_m]
    _f1h_base, _ = _cv_f1h_at(_pv, _yv, t_fixed=THRESH[_tr])
    _f1h_ref, _t_ref = _cv_f1h_at(_pv, _yv)
    if _f1h_ref > _f1h_base + 0.005:
        print(f"F1(0) threshold refinement {_tr}: t {THRESH[_tr]:.2f} -> {_t_ref:.2f} "
              f"(cv F1(0) {_f1h_base:.4f} -> {_f1h_ref:.4f}) ADOPTED")
        THRESH[_tr] = _t_ref
    else:
        print(f"F1(0) threshold refinement {_tr}: keep t={THRESH[_tr]:.2f} "
              f"(cv F1(0) {_f1h_base:.4f} vs refit {_f1h_ref:.4f})")

# ---- pass 2: reference-augmented prompts (squad + grammar refs) --------------
THRESH["ref"] = 0.95
vm = val[val.ref_text != ""].reset_index(drop=True)
vm_probs = np.array([])
if len(vm):
    vm_probs = score_blend(vm, "val-ref")
    resid = ~vm.ref_contain.values
    y_res, p_res = vm.label.values[resid], vm_probs[resid]
    if len(y_res) >= 25 and min((y_res == 0).sum(), (y_res == 1).sum()) >= 5:
        t_ref, cv_ref, fold_ts = cv_threshold(p_res, y_res)
        THRESH["ref"] = t_ref
        print(f"{'ref':9s} n={len(y_res):3d} residuals | fold thresholds {[f'{x:.2f}' for x in fold_ts]} "
              f"| cv macroF1 {cv_ref:.4f} | chosen t={t_ref:.2f}")
    else:
        print(f"ref branch: only {len(y_res)} residual rows / too one-sided - keeping conservative t=0.95")

# ---- v4 gate: context-relevance rerouting ------------------------------------
USE_REL_FINAL, REL_CUT = False, None
alt = pd.Series(dtype=float)
if USE_REL_ROUTE:
    base_pred_g = (val_probs[g_mask] > THRESH["grounded"]).astype(int)
    f1_base = metric_f1(y[g_mask], base_pred_g)
    cand = val[val.grounded & (val.ref_text == "") & (val.ctx_rel < 0.45)]
    if len(cand) >= 10:
        p_alt = score_blend(cand.assign(ctx_override="", ref_text=""), "val-relroute")
        alt = pd.Series(p_alt, index=cand.index)
        gidx, best_rel = np.flatnonzero(g_mask), -1.0
        for cut in (0.15, 0.30, 0.45):
            pred = base_pred_g.copy()
            n_r = 0
            for pos_i, vi in enumerate(gidx):
                if vi in alt.index and val.at[vi, "ctx_rel"] < cut:
                    pred[pos_i] = int(alt[vi] > THRESH_BLEND["closed"])
                    n_r += 1
            f1_c = metric_f1(y[g_mask], pred)
            print(f"rel-route cut={cut:.2f} rows={n_r} macroF1={f1_c:.4f} (base {f1_base:.4f})")
            if f1_c > f1_base + 0.01 and f1_c > best_rel:
                USE_REL_FINAL, REL_CUT, best_rel = True, cut, f1_c
    else:
        print(f"rel-route: only {len(cand)} low-relevance grounded rows - cannot evaluate")
print(f"rel-route gate: {('ENABLED cut=%.2f' % REL_CUT) if USE_REL_FINAL else 'off'}")

# ---- v4 gate: grammar containment tier ---------------------------------------
GRAM1_OK = False
if USE_GRAMMAR:
    gl = val[val.gram_hit]
    n_t = int(gl.gram_contain.sum())
    if n_t >= 8:
        prec = (gl.label[gl.gram_contain] == 1).mean()
        GRAM1_OK = prec >= 0.90
        print(f"grammar containment tier: n={n_t} precision={prec:.3f} -> {'ENABLED' if GRAM1_OK else 'off'}")
    else:
        print(f"grammar containment tier: n={n_t} too small - off (ref-augmented prompts still active)")

# ---- v6.11 gate: field-aware idiom tiers ---------------------------------------
if USE_IDIOM_TIERS and IDM_LIT:
    _iv = val[val.idiom_v.notna()]
    _im1, _im0 = (_iv.idiom_v == 1), (_iv.idiom_v == 0)
    if _im1.sum() >= 5:
        _ip = (_iv.label[_im1] == 1).mean()
        IDIOM1_OK = bool(_ip >= 0.90)
        print(f"idiom tier1 (asked-field match -> 1): n={int(_im1.sum())} "
              f"precision={_ip:.3f} -> {'ENABLED' if IDIOM1_OK else 'off'}")
    else:
        print(f"idiom tier1: n={int(_im1.sum())} too small - off")
    if _im0.sum() >= 4:
        _ip = (_iv.label[_im0] == 0).mean()
        IDIOM0_OK = bool(_ip >= 0.90)
        print(f"idiom tier0 (swap-trap match -> 0): n={int(_im0.sum())} "
              f"precision={_ip:.3f} -> {'ENABLED' if IDIOM0_OK else 'off'}")
    else:
        print(f"idiom tier0: n={int(_im0.sum())} too small - off")

# ---- v5 gates: ctx tiers, cb tiers, group rule B ------------------------------
CTX1_OK = CTX0_OK = False
if USE_CTX_MATCH:
    vg2 = val[val.grounded]
    t1 = vg2.ctx_soft & vg2.ctx_contain
    t0 = vg2.ctx_hit & vg2.ctx_miss
    if t1.sum() >= 15:
        p1 = (vg2.label[t1] == 1).mean()
        CTX1_OK = p1 >= 0.90
        print(f"ctx tier1 (soft containment -> 1): n={int(t1.sum())} precision={p1:.3f} -> {'ENABLED' if CTX1_OK else 'off'}")
    else:
        print(f"ctx tier1: n={int(t1.sum())} too small - off")
    if t0.sum() >= 15:
        p0 = (vg2.label[t0] == 0).mean()
        CTX0_OK = p0 >= 0.90
        print(f"ctx tier0 (paragraph pool-miss -> 0): n={int(t0.sum())} precision={p0:.3f} -> {'ENABLED' if CTX0_OK else 'off'}")
    else:
        print(f"ctx tier0: n={int(t0.sum())} too small - off")

CB1_OK = CB0_OK = False
if USE_CB_REFS:
    vc2 = val[~val.grounded]
    t1 = vc2.cb_contain
    t0 = vc2.cb_hit & vc2.cb_miss
    if t1.sum() >= 10:
        p1 = (vc2.label[t1] == 1).mean()
        CB1_OK = p1 >= 0.90
        print(f"cb tier1 (gold containment -> 1): n={int(t1.sum())} precision={p1:.3f} -> {'ENABLED' if CB1_OK else 'off'}")
    else:
        print(f"cb tier1: n={int(t1.sum())} too small - off")
    if t0.sum() >= 10:
        p0 = (vc2.label[t0] == 0).mean()
        CB0_OK = p0 >= 0.90
        print(f"cb tier0 (pool-miss -> 0): n={int(t0.sum())} precision={p0:.3f} -> {'ENABLED' if CB0_OK else 'off'}")
    else:
        print(f"cb tier0: n={int(t0.sum())} too small - off")

# ---- v6.2 gates: BenHalluEval exact-pair tiers --------------------------------
def _bh_gate(mask, want, name, min_n=8, min_prec=0.95):
    n = int(mask.sum())
    if n >= min_n:
        prec = (val.label[mask] == want).mean()
        ok = bool(prec >= min_prec)
        print(f"{name}: n={n} precision={prec:.3f} -> {'ENABLED' if ok else 'off'}")
        return ok
    print(f"{name}: n={n} too small - off")
    return False

def _bh_exact(mask, want, name):
    """Exact (prompt,response) matches to VERIFIED answers - always applied, same
    trust level as the ungated labeled exact-lookup. Labeled precision is a printed
    sanity check, not an on/off switch (a gate could silently drop a 100%-precise
    deterministic layer)."""
    n = int(mask.sum())
    if n:
        prec = (val.label[mask] == want).mean()
        warn = "" if prec >= 0.95 else "  <-- WARN: labeled precision < 0.95, inspect data!"
        print(f"{name}: n={n} labeled precision={prec:.3f} -> ALWAYS ON{warn}")
    else:
        print(f"{name}: 0 labeled fires (still applied on test)")
    return True

CNM_OK = False
if int(val.ctx_num_bad.sum()) >= 8:
    _p = (val.label[val.ctx_num_bad] == 0).mean()
    CNM_OK = bool(_p >= 0.90)
    print(f"ctx numeric-contradiction -> 0: n={int(val.ctx_num_bad.sum())} "
          f"precision={_p:.3f} -> {'ENABLED' if CNM_OK else 'off'}")
else:
    print(f"ctx numeric-contradiction: n={int(val.ctx_num_bad.sum())} too small - off")

BH1_OK  = _bh_exact(val.bh_gold, 1, "bh gold-pair -> 1") if USE_BENHALLU else False
BH0_OK  = _bh_exact(val.bh_hall, 0, "bh hall-pair -> 0") if USE_BENHALLU else False
BHM1_OK = _bh_gate(val.bh_m1,   1, "bh model-correct pair -> 1")   # weaker: stays gated
BHM0_OK = _bh_gate(val.bh_m0,   0, "bh model-wrong pair -> 0")

GROUPB_OK = False
if USE_GROUPS:
    _pairs = []
    for _pn, _idx in val.groupby("p_norm").groups.items():
        if len(_idx) < 2:
            continue
        _sub = val.loc[_idx]
        _anc = _sub[(_sub.label == 1) & (_sub.ref_contain | _sub.gram_contain
                                          | (_sub.ctx_soft & _sub.ctx_contain) | _sub.cb_contain)]
        if not len(_anc):
            continue
        _ar = _anc.response_bn.iloc[0]
        for _j in _idx:
            if _j in _anc.index:
                continue
            if not resp_agree(val.at[_j, "response_bn"], _ar):
                _pairs.append(int(val.at[_j, "label"]))
    if len(_pairs) >= 10:
        _p0 = float(np.mean([l == 0 for l in _pairs]))
        GROUPB_OK = _p0 >= 0.90
        print(f"group rule B: n={len(_pairs)} anchor-conflicting rows, P(label=0)={_p0:.3f} -> {'ENABLED' if GROUPB_OK else 'off'}")
    else:
        print(f"group rule B: n={len(_pairs)} too small - off (rule A always applies)")

# ---- v6.6 diagnostic: stack vs blend on the rows the meta-judge really decides ----
# Exact tiers override most grounded val rows (bh gold/hall pairs, refs, ctx tiers),
# so a track-wide CV can pick the stack for rows it never sees at test time. This
# reports the comparison restricted to the rows that actually reach the meta-judge.
# NOTE `wrong_match` is deliberately absent: it exists only on `test` (a response matching
# a known-wrong answer *from* the labeled set is self-referential on labeled). Built
# defensively so a layer that disabled itself can never crash this diagnostic again.
_det = np.zeros(len(val), dtype=bool)
for _c in ("bh_gold", "bh_hall", "ref_contain", "gram_contain", "cb_contain", "ctx_num_bad",
           "bcs_gold", "bcs_dist", "bcs_tgold", "bcs_tdist"):
    if _c in val.columns:
        _det |= val[_c].fillna(False).astype(bool).values
for _a, _b in (("ctx_soft", "ctx_contain"), ("ctx_hit", "ctx_miss")):
    if _a in val.columns and _b in val.columns:
        _det |= (val[_a].fillna(False).astype(bool) & val[_b].fillna(False).astype(bool)).values
if "stack_cv" in dir() and stack_cv:
    _oof = np.zeros(len(val))
    _oof[g_mask] = stack_cv["grounded"][1]
    _oof[c_fit] = stack_cv["closed"][1]
    for _tr, _m, _bthr in (("grounded", g_mask, THRESH_BLEND["grounded"]),
                           ("closed",   c_fit, THRESH_BLEND["closed"])):
        _sel = _m & ~_det
        if _sel.sum() >= 20:
            _fs = metric_f1(y[_sel], (_oof[_sel] > STACK[_tr][2]).astype(int))
            _fb = metric_f1(y[_sel], (val_blend[_sel] > _bthr).astype(int))
            print(f"judge-decided {_tr:9s} (n={int(_sel.sum()):3d}, tiers excluded): "
                  f"stack {_fs:.4f} vs blend {_fb:.4f}  -> {'stack' if _fs > _fb else 'blend'} better here")
        else:
            print(f"judge-decided {_tr:9s}: only {int(_sel.sum())} rows after tier exclusion "
                  f"- track-wide CV is the only estimate available")

# ---- graft 3 gate: retrieval-augmented closed-track strategy ------------------
USE_RET_FINAL, RET_CUT = False, None
THRESH["closed_ret"] = THRESH_BLEND["closed"]
if USE_RETRIEVAL:
    vc = val[~val.grounded].reset_index(drop=True)
    # v6.6: the re-judge probs (p_rej) come from score_blend, so the plain probs must be
    # blend-scale too - mixing stack-scale and blend-scale probs into one array would
    # make the fitted threshold meaningless when the closed track selects the stack.
    vc_plain = val_blend[c_mask]
    hit = (vc.ret_text != "").values
    if hit.sum() >= 20:
        vh = vc[hit].reset_index(drop=True)
        p_rej = score_blend(vh.assign(ctx_override=vh.ret_text.values, ref_text=""), "val-rej")
        all_conf = pd.concat([labeled.ret_conf, test.ret_conf]).dropna().values
        best = (CV_F1["closed"], None, THRESH_BLEND["closed"])
        for pct in (60, 75, 90):
            cut = float(np.percentile(all_conf, pct))
            use_rej = hit.copy()
            use_rej[hit] = vh.ret_conf.values >= cut
            s = vc_plain.copy()
            s[use_rej] = p_rej[vh.ret_conf.values >= cut]
            t, cv_f1, _ = cv_threshold(s, vc.label.values)
            print(f"retrieval pct={pct} (cut={cut:.1f}, rows={int(use_rej.sum())}) cv macroF1={cv_f1:.4f}")
            if cv_f1 > best[0] + 0.005:
                best = (cv_f1, cut, t)
        if best[1] is not None:
            USE_RET_FINAL, RET_CUT, THRESH["closed_ret"] = True, best[1], best[2]
        print(f"retrieval gate: {('ENABLED cut=%.1f t=%.2f' % (RET_CUT, THRESH['closed_ret'])) if USE_RET_FINAL else 'off (no CV gain)'}")
    else:
        print("retrieval gate: too few labeled closed rows with hits - off")

# ---- graft 2 gate: QA tier precision on labeled grounded rows -----------------
if USE_QA_VERIFY:
    vg = val[val.grounded]
    t1, t0 = qa_tier1_mask(vg), qa_tier0_mask(vg)
    if t1.sum() >= 15:
        prec1 = (vg.label[t1] == 1).mean()
        QA1_OK = prec1 >= 0.90
        print(f"QA tier1 (match>=.95 -> 1): n={int(t1.sum())} precision={prec1:.3f} -> {'ENABLED' if QA1_OK else 'off'}")
    else:
        print(f"QA tier1: n={int(t1.sum())} too small - off")
    if t0.sum() >= 15:
        prec0 = (vg.label[t0] == 0).mean()
        QA0_OK = prec0 >= 0.90
        print(f"QA tier0 (numeric contradiction -> 0): n={int(t0.sum())} precision={prec0:.3f} -> {'ENABLED' if QA0_OK else 'off'}")
    else:
        print(f"QA tier0: n={int(t0.sum())} too small - off")

# ---- graft 1: sympy verdicts on labeled math rows ------------------------------
val["sympy_v"] = np.nan
if USE_SYMPY:
    for i in val.index[(~val.grounded) & val.prompt_bn.map(is_math)]:
        v = sympy_verdict(val.at[i, "prompt_bn"], val.at[i, "response_bn"])
        if v is not None:
            val.at[i, "sympy_v"] = v
    n_s = int(val.sympy_v.notna().sum())
    if n_s:
        acc = (val.sympy_v.dropna() == val.label[val.sympy_v.notna()]).mean()
        print(f"sympy on labeled math: {n_s} verdicts, accuracy {acc:.3f}")

# ---- v6 gate: translate-then-verify on uncertain rows --------------------------
XLING_INSTR_CLOSED = (
    "You are a bilingual Bengali-English fact-checker. Follow these steps:\n"
    "1. Translate the question and the candidate answer into English.\n"
    "2. Answer the question yourself in English from your own knowledge "
    "(assume a Bangladeshi context where relevant).\n"
    "3. Compare your answer with the candidate answer.\n"
    'End with exactly one line: "Final verdict: Yes" if the candidate answer is correct, '
    'or "Final verdict: No" if it is wrong.'
)
XLING_INSTR_GROUNDED = (
    "You are a bilingual Bengali-English fact-checker. Follow these steps:\n"
    "1. Translate the question and the candidate answer into English.\n"
    "2. Find what the Bengali passage says about the question and state it in English.\n"
    "3. Compare the passage's answer with the candidate answer.\n"
    'End with exactly one line: "Final verdict: Yes" if the candidate answer is correct '
    'according to the passage, or "Final verdict: No" if it is wrong.'
)
VERDICT_EN_RE = re.compile(r"Final verdict\s*:\s*\"?(Yes|No)", re.I)

def xling_user(context, prompt_bn, response_bn):
    if context:
        return (XLING_INSTR_GROUNDED + "\n\nPassage (Bengali): " + context[:900]
                + "\nQuestion (Bengali): " + prompt_bn
                + "\nCandidate answer: " + response_bn)
    return (XLING_INSTR_CLOSED + "\n\nQuestion (Bengali): " + prompt_bn
            + "\nCandidate answer: " + response_bn)

def xling_parse(replies):
    for rep_ in replies:
        m = VERDICT_EN_RE.findall(rep_)
        if m:
            return 1 if m[-1].lower() == "yes" else 0
    return None

def uncertainty(df_rows, probs):
    thr = np.where(df_rows.grounded.values, THRESH["grounded"], THRESH["closed"])
    return np.abs(probs - thr)

XLING_OK = False
val_xv = {}
if USE_XLING_COT:
    elig = ((val.ref_text == "").values & ~val.prompt_bn.map(is_math).values
            & ~val.gram_contain.values & ~(val.ctx_soft & val.ctx_contain).values
            & ~val.cb_contain.values
            & ~val.idiom_v.notna().values
            & ~val.bh_gold.values & ~val.bh_hall.values)
    d = uncertainty(val, val_probs)
    order = np.argsort(d)
    pick = [i for i in order if elig[i]][:XLING_VAL_N]
    users = [xling_user(val.at[i, "context"], val.at[i, "prompt_bn"], val.at[i, "response_bn"])
             for i in pick]
    reps = _gen_qwen_batch(users, max_new_tokens=200, n=1, desc="val xling")
    for i, rep_ in zip(pick, reps):
        val_xv[i] = xling_parse(rep_)
    parsed = {i: v for i, v in val_xv.items() if v is not None}
    if len(parsed) >= 30:
        yy   = np.array([y[i] for i in parsed])
        vv   = np.array([parsed[i] for i in parsed])
        bb   = np.array([int(val_probs[i] > (THRESH["grounded"] if g_mask[i] else THRESH["closed"]))
                         for i in parsed])
        acc_v, acc_b = float((vv == yy).mean()), float((bb == yy).mean())
        XLING_OK = acc_v >= acc_b + 0.03
        print(f"xling gate: n={len(parsed)} parsed | verdict acc {acc_v:.3f} vs base {acc_b:.3f} "
              f"-> {'ENABLED' if XLING_OK else 'off'}")
    else:
        print(f"xling gate: only {len(parsed)} parseable verdicts - off")

# ---- full-pipeline simulation (mirrors test-time order exactly) ----------------
sim = np.where(g_mask, val_probs > THRESH["grounded"],
                       val_probs > THRESH["closed"]).astype(int)
# v6.18 mirror: math rows take the Qwen-only base (sympy below still outranks it;
# the CoT pass is not simulated on labeled - same as before)
if QWEN_ONLY_MATH and vq is not None:
    sim[math_mask] = (vq[math_mask] > 0.5).astype(int)
if USE_REL_FINAL:
    for vi in np.flatnonzero(g_mask):
        if vi in alt.index and val.at[vi, "ctx_rel"] < REL_CUT and val.at[vi, "ref_text"] == "":
            sim[vi] = int(alt[vi] > THRESH_BLEND["closed"])
if USE_RET_FINAL:
    hitmask = (val.ret_text != "").values & c_mask & (val.ret_conf.fillna(-1).values >= RET_CUT)
    vr = val[hitmask].reset_index(drop=True)
    if len(vr):
        p_r = score_blend(vr.assign(ctx_override=vr.ret_text.values, ref_text=""), "sim-rej")
        sim[np.flatnonzero(hitmask)] = (p_r > THRESH["closed_ret"]).astype(int)
if XLING_OK:
    for i, v in val_xv.items():
        if v is not None:
            sim[i] = int(v)
if USE_SYMPY:
    sv = val.sympy_v.notna().values
    sim[sv] = val.sympy_v.values[sv].astype(int)
if QA1_OK:
    m = (qa_tier1_mask(val) & val.grounded & (val.ref_text == "")).values
    sim[m] = 1
if QA0_OK:
    m = (qa_tier0_mask(val) & val.grounded & (val.ref_text == "")).values
    sim[m] = 0
if len(vm):
    pos = np.flatnonzero((val.ref_text != "").values)
    sim[pos] = np.where(vm.ref_contain.values, 1, (vm_probs > THRESH["ref"]).astype(int))
if GRAM1_OK:
    m = (val.gram_hit & val.gram_contain).values
    sim[m] = 1
if IDIOM1_OK:
    sim[(val.idiom_v == 1).values] = 1
if IDIOM0_OK:
    sim[(val.idiom_v == 0).values] = 0
if CTX1_OK:
    m = (val.ctx_soft & val.ctx_contain & val.grounded).values
    sim[m] = 1
if CTX0_OK:
    m = (val.ctx_hit & val.ctx_miss & val.grounded).values
    sim[m] = 0
if CNM_OK:
    sim[val.ctx_num_bad.values] = 0
if CB1_OK:
    sim[val.cb_contain.values] = 1
if CB0_OK:
    sim[(val.cb_hit & val.cb_miss).values] = 0
if BCS_RAW:
    sim[(val.bcs_dist | val.bcs_tdist).values] = 0
    sim[(val.bcs_gold | val.bcs_tgold).values] = 1
if BHM1_OK:
    sim[val.bh_m1.values] = 1
if BHM0_OK:
    sim[val.bh_m0.values] = 0
if BH1_OK:
    sim[val.bh_gold.values] = 1
if BH0_OK:
    sim[val.bh_hall.values] = 0
sim[(val.c1_wrong & val.c1_trap.isin(C1_ENABLED)).values] = 0
sim[(val.c1_right & val.c1_trap.isin(C1_ENABLED)).values] = 1
if GROUPB_OK:
    _strong1 = (val.ref_contain | val.gram_contain | (val.ctx_soft & val.ctx_contain) | val.cb_contain | ((val.idiom_v == 1) & IDIOM1_OK))
    for _pn, _idx in val.groupby("p_norm").groups.items():
        if len(_idx) < 2:
            continue
        _anc = [j for j in _idx if _strong1.loc[j] and sim[j] == 1]
        if not _anc:
            continue
        _ar = val.at[_anc[0], "response_bn"]
        for _j in _idx:
            if _strong1.loc[_j] or sim[_j] == 0:
                continue
            if not resp_agree(val.at[_j, "response_bn"], _ar):
                sim[_j] = 0

print(f"\nfull-pipeline simulation on labeled set: macro F1 = {metric_f1(val.label.values, sim):.4f}")
print(f"F1(label=0) (v4/v5 reference metric)   : {f1_hall(val.label.values, sim):.4f}")
print("references: v3 LB 0.806 | v4 LB 0.816 | v5 LB 0.799 | v5.1 LB 0.818")
print()
print(classification_report(val.label.values, sim, target_names=["hallucinated (0)", "faithful (1)"]))


validation rows: 295 (excluded 4 few-shot exemplars)
closed-track fit rows: 154 (13 math rows excluded from fitting, still scored)


val-t:   0%|          | 0/295 [00:00<?, ?it/s]

val-q:   0%|          | 0/295 [00:00<?, ?it/s]

val-qe:   0%|          | 0/295 [00:00<?, ?it/s]

validation scores: SAVED -> /kaggle/working/val_scores.json  (download + attach as a Kaggle dataset, then set USE_VAL_SCORE_CACHE=True to skip ~31 min on the final run)
w=1.00 (tiger only) | grounded cvF1 0.8277 | closed cvF1 0.7090 | weighted 0.7629
w=0.90              | grounded cvF1 0.8445 | closed cvF1 0.6770 | weighted 0.7530
w=0.80              | grounded cvF1 0.8352 | closed cvF1 0.6935 | weighted 0.7578
w=0.70              | grounded cvF1 0.8352 | closed cvF1 0.7118 | weighted 0.7678
w=0.60              | grounded cvF1 0.8377 | closed cvF1 0.7166 | weighted 0.7716
w=0.50              | grounded cvF1 0.8377 | closed cvF1 0.7293 | weighted 0.7785
w=0.40              | grounded cvF1 0.8377 | closed cvF1 0.7365 | weighted 0.7824
w=0.30              | grounded cvF1 0.8377 | closed cvF1 0.6724 | weighted 0.7475
w=0.20              | grounded cvF1 0.8468 | closed cvF1 0.6915 | weighted 0.7620
w=0.10              | grounded cvF1 0.8468 | closed cvF1 0.5936 | weighted 0.7085
w=0.00     

...

val-rej-t:   0%|          | 0/167 [00:00<?, ?it/s]

retrieval pct=60 (cut=25.2, rows=67) cv macroF1=0.7195
retrieval pct=75 (cut=29.3, rows=33) cv macroF1=0.7181
retrieval pct=90 (cut=37.1, rows=12) cv macroF1=0.7478
retrieval gate: ENABLED cut=37.1 t=0.60
sympy on labeled math: 1 verdicts, accuracy 1.000


sim-rej-q:   0%|          | 0/12 [00:00<?, ?it/s]

sim-rej-t:   0%|          | 0/12 [00:00<?, ?it/s]


full-pipeline simulation on labeled set: macro F1 = 0.8696
F1(label=0) (v4/v5 reference metric)   : 0.8550
references: v3 LB 0.806 | v4 LB 0.816 | v5 LB 0.799 | v5.1 LB 0.818

                  precision    recall  f1-score   support

hallucinated (0)       0.88      0.84      0.85       134
    faithful (1)       0.87      0.90      0.88       161

        accuracy                           0.87       295
       macro avg       0.87      0.87      0.87       295
    weighted avg       0.87      0.87      0.87       295



In [26]:
MATH_INSTR = (
    'তুমি একজন গণিত শিক্ষক। নিচের গাণিতিক প্রশ্নটি ধাপে ধাপে নিজে সমাধান করো, '
    'তারপর তোমার ফলাফলের সাথে প্রদত্ত উত্তরটি মিলিয়ে দেখো। '
    'সবশেষে ঠিক এই ফরম্যাটে এক লাইনে রায় দাও — "চূড়ান্ত রায়: হ্যাঁ" (প্রদত্ত উত্তর সঠিক হলে) '
    'অথবা "চূড়ান্ত রায়: না" (প্রদত্ত উত্তর ভুল হলে)।'
)
MATH_INSTR_EN = (
    "You are a careful math teacher. The problem below is in Bengali. "
    "Solve it yourself step by step in English (translate Bengali number words carefully), "
    "then compare your result with the given answer. "
    'End with exactly one line: "Final verdict: Yes" if the given answer is correct, '
    'or "Final verdict: No" if it is wrong.'
)
VERDICT_RE   = re.compile(r'চূড়ান্ত রায়\s*[:ঃ]?\s*"?(হ্যাঁ|না)')
FALLBACK_RE  = re.compile(r'হ্যাঁ|(?<!\w)না(?!\w)')
VERDICT_EN_MATH = re.compile(r"Final verdict\s*:\s*\"?(Yes|No)", re.I)

@torch.inference_mode()
def _cot(mdl, tok, render, stop_ids, prompt_bn, response_bn):
    user = MATH_INSTR + "\n\nপ্রশ্ন: " + prompt_bn + "\nপ্রদত্ত উত্তর: " + response_bn
    text = render([{"role": "user", "content": user}])
    enc = tok(text, return_tensors="pt").to(mdl.device)
    out = mdl.generate(**enc, max_new_tokens=192, do_sample=False,
                       eos_token_id=stop_ids, pad_token_id=tok.pad_token_id)
    reply = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    m = VERDICT_RE.findall(reply)
    if not m:
        m = FALLBACK_RE.findall(reply.strip()[-80:])
    if m:
        return 1 if m[-1] == "হ্যাঁ" else 0
    return None

def math_user(prompt_bn, response_bn):
    return (MATH_INSTR_EN + "\n\nProblem (Bengali): " + prompt_bn
            + "\nGiven answer: " + response_bn)

def parse_math_votes(replies):
    """Majority vote over parsed verdicts from the sampled chains; tie/none -> None."""
    votes = []
    for rep_ in replies:
        m = VERDICT_EN_MATH.findall(rep_)
        if m:
            votes.append(1 if m[-1].lower() == "yes" else 0)
    if not votes:
        return None
    s = sum(votes)
    if s * 2 == len(votes):       # tie -> abstain, let the Tiger fallback decide
        return None
    return int(s * 2 > len(votes))


In [27]:
def chunk_file(i):
    return os.path.join(CHECKPOINT_DIR, f"chunk_{i:04d}.parquet")

n_chunks = math.ceil(len(test) / CHUNK_SIZE)
print(f"{len(test)} rows in {n_chunks} chunks of {CHUNK_SIZE}")

def _track_uses_qe(tr):
    if not USE_XLING_EN:
        return False
    if META.get(tr) == "stack":
        return True
    if META.get(tr) == "blend3":
        return "p_qe" in BLEND3_W.get(tr, {})
    return False
_QE_USE = {tr: _track_uses_qe(tr) for tr in ("grounded", "closed")}
if USE_XLING_EN and not all(_QE_USE.values()):
    print(f"2xT4: English-Qwen (p_qe) used per track = {_QE_USE}; skipped where unused")

for i in range(n_chunks):
    if os.path.exists(chunk_file(i)):
        print(f"chunk {i} already scored, skipping")
        continue
    lo, hi = i * CHUNK_SIZE, min((i + 1) * CHUNK_SIZE, len(test))
    part = test.iloc[lo:hi]
    t0 = time.time()
    _qe_mask = None
    if USE_XLING_EN:
        _qe_mask = np.where(part.grounded.values, _QE_USE["grounded"], _QE_USE["closed"])
    _pt, _pq, _pqe = score_judges(part, f"chunk{i}", qe_rows=_qe_mask)
    dfc = pd.DataFrame({"id": part.id.values, "p_t": _pt})
    if _pq is not None:
        dfc["p_q"] = _pq
    if _pqe is not None and not np.isnan(_pqe).all():
        dfc["p_qe"] = _pqe
    dfc.to_parquet(chunk_file(i))
    dt = time.time() - t0
    print(f"chunk {i}: {len(part)} rows in {dt:.0f}s ({len(part)/dt:.2f} rows/s), "
          f"eta {(n_chunks - i - 1) * dt / 60:.0f} min")

all_scores = pd.concat([pd.read_parquet(chunk_file(i)) for i in range(n_chunks)], ignore_index=True)
test = test.drop(columns=["p_t", "p_q", "p_qe", "p_yes", "p_final"], errors="ignore").merge(
    all_scores, on="id", how="left")
assert test.p_t.notna().all(), "some test rows were not scored - delete bad chunk files and re-run"

# blend-scale probability (ref rows were scored with ref-augmented prompts)
p_blend = test.p_t.values.copy()
if ENSEMBLE_ON and "p_q" in test.columns:
    p_blend = W_TIGER * p_blend + (1.0 - W_TIGER) * test.p_q.values
test["p_yes"] = p_blend

# v6: stacked probability for plain-prompt (non-ref) rows
p_final = p_blend.copy()
if STACK_ON or any(v == "blend3" for v in META.values()):
    plain = (test.ref_text == "").values
    TF = judge_frame(test, test.p_t.values,
                     test.p_q.values if "p_q" in test.columns else None,
                     test.p_qe.values if "p_qe" in test.columns else None)
    for track, m_ in (("grounded", plain & test.grounded.values),
                      ("closed",   plain & ~test.grounded.values)):
        if not m_.any():
            continue
        if META[track] == "stack":
            p_final[m_] = stack_prob(track, TF[m_])
        elif META[track] == "blend3":
            p_final[m_] = blend3_prob(
                track, test.p_t.values[m_],
                test.p_q.values[m_] if "p_q" in test.columns else None,
                test.p_qe.values[m_] if "p_qe" in test.columns else None)
        else:
            continue   # 2-way blend already in p_final
        print(f"{META[track]} meta-judge applied to {int(m_.sum())} {track} rows")
test["p_final"] = p_final

# base thresholds (ref rows -> blend + ref threshold; others -> final + track threshold)
ref_rows = (test.ref_text != "").values
pred = np.where(test.grounded.values,
                p_final > THRESH["grounded"], p_final > THRESH["closed"]).astype(int)
pred[ref_rows] = (p_blend[ref_rows] > THRESH["ref"]).astype(int)
test["pred"] = pred

def cached_blend(df, path, desc):
    cache = cached_scores(df, path, desc)
    p = cache.p_t.values.copy()
    if ENSEMBLE_ON and "p_q" in cache.columns:
        p = W_TIGER * p + (1.0 - W_TIGER) * cache.p_q.values
    return cache.id.values, p

# v4: context-relevance rerouting for grounded rows with useless contexts
if USE_REL_FINAL:
    rr = test[test.grounded & (test.ref_text == "") & (test.ctx_rel < REL_CUT)]
    if len(rr):
        ids, p_alt = cached_blend(rr.assign(ctx_override="", ref_text=""),
                                  os.path.join(CHECKPOINT_DIR, "relroute.parquet"), "relroute")
        s = pd.Series(p_alt, index=ids)
        m = test.id.isin(s.index)
        test.loc[m, "pred"] = (test.loc[m, "id"].map(s) > THRESH_BLEND["closed"]).astype(int)
        print(f"rel-route applied to {int(m.sum())} grounded rows (ctx_rel < {REL_CUT})")

# graft 3: retrieval re-judge for confident closed rows
if USE_RET_FINAL:
    rj = test[(~test.grounded) & (test.ret_text != "") & (test.ret_conf.fillna(-1) >= RET_CUT)]
    if len(rj):
        ids, p_rej = cached_blend(rj.assign(ctx_override=rj.ret_text.values, ref_text=""),
                                  os.path.join(CHECKPOINT_DIR, "rejudge.parquet"), "rejudge")
        s = pd.Series(p_rej, index=ids)
        m = test.id.isin(s.index)
        test.loc[m, "pred"] = (test.loc[m, "id"].map(s) > THRESH["closed_ret"]).astype(int)
        print(f"retrieval re-judge applied to {int(m.sum())} closed rows")

# v6: translate-then-verify on the most uncertain unresolved rows (gated + capped)
math_mask = (~test.grounded) & test.prompt_bn.map(is_math) & (test.ref_text == "")

# v6.18: Qwen-only base verdict for math rows (replaces the 80%-Tiger blend3 base;
# survives only where the Qwen-EN CoT verdict below is unparseable)
if QWEN_ONLY_MATH and "p_q" in test.columns:
    _mm = math_mask.values
    test.loc[_mm, "pred"] = (test.p_q.values[_mm] > 0.5).astype(int)
    print(f"Qwen-only math base: {int(_mm.sum())} rows set from p_q > 0.5 (Tiger removed from math)")

if XLING_OK:
    elig = ((test.ref_text == "").values & ~math_mask.values
            & ~test.ref_contain.values & ~test.wrong_match.values
            & ~test.idiom_v.notna().values
            & ~(test.gram_hit & test.gram_contain).values
            & ~(test.ctx_soft & test.ctx_contain).values
            & ~(test.ctx_hit & test.ctx_miss).values
            & ~test.cb_contain.values & ~(test.cb_hit & test.cb_miss).values
            & ~test.bh_gold.values & ~test.bh_hall.values
            & ~test.c1_wrong.values & ~test.c1_right.values
            & test.lookup_label.isna().values)
    d = np.abs(p_final - np.where(test.grounded.values, THRESH["grounded"], THRESH["closed"]))
    order = np.argsort(d)
    xl_cap = max(XLING_COT_CAP, min(2 * XLING_COT_CAP, int(0.18 * len(test))))
    pick = [int(j) for j in order if elig[j]][:xl_cap]
    XL_CKPT = os.path.join(CHECKPOINT_DIR, "xling_verdicts.json")
    xverd = {}
    if os.path.exists(XL_CKPT):
        with open(XL_CKPT, encoding="utf-8") as f:
            xverd = json.load(f)
        print(f"resuming: {len(xverd)} xling verdicts already checkpointed")
    todo = [j for j in pick if str(int(test.at[j, "id"])) not in xverd]
    SLICE = 48
    for lo in tqdm(range(0, len(todo), SLICE), desc="test xling (batched)"):
        rows = todo[lo:lo + SLICE]
        users = [xling_user(test.at[j, "context"], test.at[j, "prompt_bn"],
                            test.at[j, "response_bn"]) for j in rows]
        reps = _gen_qwen_batch(users, max_new_tokens=200, n=1)
        for j, rep_ in zip(rows, reps):
            xverd[str(int(test.at[j, "id"]))] = xling_parse(rep_)
        with open(XL_CKPT, "w", encoding="utf-8") as f:
            json.dump(xverd, f)
    flipped = 0
    for j in pick:
        v = xverd.get(str(int(test.at[j, "id"])))
        if v is not None and int(v) != test.at[j, "pred"]:
            test.at[j, "pred"] = int(v)
            flipped += 1
    print(f"xling pass: {len(pick)} rows judged, {flipped} predictions flipped")
else:
    print("xling pass: gate off")

# graft 1: sympy overrides on math rows, then CoT only where sympy abstained
sympy_resolved = pd.Series(False, index=test.index)
if USE_SYMPY:
    n_s = 0
    for row in test[math_mask].itertuples():
        v = sympy_verdict(row.prompt_bn, row.response_bn)
        if v is not None:
            test.at[row.Index, "pred"] = int(v)
            sympy_resolved.at[row.Index] = True
            n_s += 1
    print(f"sympy resolved {n_s} of {int(math_mask.sum())} math rows")

if USE_MATH_COT:
    cot_mask = math_mask & ~sympy_resolved
    print(f"math CoT pass over {int(cot_mask.sum())} rows...")
    MATH_CKPT = os.path.join(CHECKPOINT_DIR, "math_verdicts.json")
    verdicts = {}
    if os.path.exists(MATH_CKPT):
        with open(MATH_CKPT, encoding="utf-8") as f:
            verdicts = json.load(f)
        print(f"resuming: {len(verdicts)} verdicts already checkpointed")
    todo = [row for row in test[cot_mask].itertuples() if str(row.id) not in verdicts]
    SLICE = 48
    for lo in tqdm(range(0, len(todo), SLICE), desc="math CoT (batched)"):
        rows = todo[lo:lo + SLICE]
        if qwen is not None:
            users = [math_user(r.prompt_bn, r.response_bn) for r in rows]
            reps = _gen_qwen_batch(users, max_new_tokens=MATH_COT_TOKENS, n=MATH_SC_N)
            for r, rep_ in zip(rows, reps):
                verdicts[str(r.id)] = parse_math_votes(rep_)
        else:
            for r in rows:
                verdicts[str(r.id)] = None
        if not QWEN_ONLY_MATH:   # v6.18: unparseable rows keep the p_q base instead
            for r in rows:   # Tiger Bengali CoT fallback where Qwen abstained or tied
                if verdicts[str(r.id)] is None:
                    verdicts[str(r.id)] = _cot(model, tokenizer, render_chat, STOP_IDS,
                                               r.prompt_bn, r.response_bn)
        with open(MATH_CKPT, "w", encoding="utf-8") as f:
            json.dump(verdicts, f)
    overridden = 0
    for row in test[cot_mask].itertuples():
        v = verdicts.get(str(row.id))
        if v is not None and int(v) != test.at[row.Index, "pred"]:
            test.at[row.Index, "pred"] = int(v)
            overridden += 1
    print(f"CoT changed {overridden} logit verdicts")

# graft 2: QA tiers on grounded rows without a reference
if QA1_OK:
    m = qa_tier1_mask(test) & test.grounded & (test.ref_text == "")
    test.loc[m, "pred"] = 1
    print(f"QA tier1 -> 1 on {int(m.sum())} rows")
if QA0_OK:
    m = qa_tier0_mask(test) & test.grounded & (test.ref_text == "")
    test.loc[m, "pred"] = 0
    print(f"QA tier0 -> 0 on {int(m.sum())} rows")

# string layers; v4 grammar containment; exact lookup wins over everything
test.loc[test.ref_contain, "pred"] = 1
test.loc[test.wrong_match, "pred"] = 0
if GRAM1_OK:
    m = test.gram_hit & test.gram_contain
    test.loc[m, "pred"] = 1
    print(f"grammar containment -> 1 on {int(m.sum())} rows")
if IDIOM1_OK:
    m = test.idiom_v == 1
    test.loc[m, "pred"] = 1
    print(f"idiom tier1 (asked-field match) -> 1 on {int(m.sum())} rows")
if IDIOM0_OK:
    m = test.idiom_v == 0
    test.loc[m, "pred"] = 0
    print(f"idiom tier0 (swap trap) -> 0 on {int(m.sum())} rows")
if CTX1_OK:
    m = test.ctx_soft & test.ctx_contain & test.grounded
    test.loc[m, "pred"] = 1
    print(f"ctx tier1 -> 1 on {int(m.sum())} rows")
if CTX0_OK:
    m = test.ctx_hit & test.ctx_miss & test.grounded
    test.loc[m, "pred"] = 0
    print(f"ctx tier0 -> 0 on {int(m.sum())} rows")
if CB1_OK:
    m = test.cb_contain
    test.loc[m, "pred"] = 1
    print(f"cb tier1 -> 1 on {int(m.sum())} rows")
if CB0_OK:
    m = test.cb_hit & test.cb_miss
    test.loc[m, "pred"] = 0
    print(f"cb tier0 -> 0 on {int(m.sum())} rows")

# v6.4: grounded numeric context-contradiction (gated in section 7) - applied
# before the exact tiers so a verified gold/lookup match still wins
if CNM_OK:
    m = test.ctx_num_bad
    test.loc[m, "pred"] = 0
    print(f"ctx numeric-contradiction -> 0 on {int(m.sum())} grounded rows")

# v6.8: BCS answer-key tiers (mostly closed-book). Placed before the BenHallu tiers
# and the exact lookup, so better-validated evidence still wins.
if BCS_RAW:
    _pre = test.pred.copy()
    test.loc[test.bcs_dist | test.bcs_tdist, "pred"] = 0
    test.loc[test.bcs_gold | test.bcs_tgold, "pred"] = 1
    print(f"BCS tiers: {int((test.pred != _pre).sum())} predictions flipped "
          f"(->1 {int((test.bcs_gold | test.bcs_tgold).sum())}, "
          f"->0 {int((test.bcs_dist | test.bcs_tdist).sum())})")

# v6.2/v6.3: BenHalluEval exact-pair tiers. The KEY diagnostic is how many
# predictions these FLIP vs the judge baseline - if ~0, BenHallu is redundant
# with the judge/refs; if large, it is doing real work.
_bh_pre = test.pred.copy()
if BHM1_OK: test.loc[test.bh_m1, "pred"] = 1
if BHM0_OK: test.loc[test.bh_m0, "pred"] = 0
if BH1_OK:  test.loc[test.bh_gold, "pred"] = 1
if BH0_OK:  test.loc[test.bh_hall, "pred"] = 0
_flip_gold = int(((test.pred != _bh_pre) & test.bh_gold).sum())
_flip_hall = int(((test.pred != _bh_pre) & test.bh_hall).sum())
_matches = int((test.bh_gold | test.bh_hall).sum())
print(f"BenHalluEval tiers: {_matches} exact test matches "
      f"({int(test.bh_gold.sum())} gold->1, {int(test.bh_hall.sum())} hall->0) | "
      f"{_flip_gold + _flip_hall} predictions FLIPPED vs the judge "
      f"(gold {_flip_gold}, hall {_flip_hall})")
if not USE_BENHALLU:
    print("  !!! USE_BENHALLU is False - the benhallu-data attachment was NOT found - "
          "these tiers contributed NOTHING. Attach the dataset. !!!")
elif _matches < 500:
    print(f"  !! only {_matches} test matches (expected ~900) - check the attachment/columns")

# v6: cultural-default traps (validated in 3g)
if int(test.c1_wrong.sum()):
    test.loc[test.c1_wrong, "pred"] = 0
    print(f"C1 wrong-default -> 0 on {int(test.c1_wrong.sum())} rows")
if int(test.c1_right.sum()):
    test.loc[test.c1_right, "pred"] = 1
    print(f"C1 correct-default -> 1 on {int(test.c1_right.sum())} rows")

n_lookup = int(test.lookup_label.notna().sum())
test["pred"] = test.lookup_label.fillna(test.pred).astype(int)

# ---- v5: within-test group consistency (final layer) --------------------------
if USE_GROUPS:
    strong1 = (test.ref_contain | test.gram_contain
               | (test.ctx_soft & test.ctx_contain) | test.cb_contain
               | (test.lookup_label == 1) | (test.bh_gold & BH1_OK)
               | ((test.idiom_v == 1) & IDIOM1_OK))
    protected = (strong1 | test.lookup_label.notna() | test.wrong_match
                 | ((test.idiom_v == 0) & IDIOM0_OK)
                 | (test.bh_hall & BH0_OK))
    n_a = n_b = 0
    for pn, idx in test.groupby("p_norm").groups.items():
        if len(idx) < 2:
            continue
        sub = test.loc[idx]
        # Rule A: identical (prompt, response) rows share one label
        for rn, jdx in sub.groupby("r_norm").groups.items():
            if len(jdx) < 2:
                continue
            s2 = test.loc[jdx]
            if s2.pred.nunique() > 1:
                if strong1.loc[jdx].any():
                    lab = 1
                elif s2.wrong_match.any():
                    lab = 0
                else:
                    lab = int(s2.pred.mode().iloc[0])
                test.loc[jdx, "pred"] = lab
                n_a += len(jdx)
        # Rule B (gated): confirmed-faithful anchor -> conflicting responses are 0
        if GROUPB_OK:
            anc = sub[strong1.loc[idx] & (test.loc[idx, "pred"] == 1)]
            if len(anc):
                ar = anc.response_bn.iloc[0]
                for j in idx:
                    if protected.loc[j] or test.at[j, "pred"] == 0:
                        continue
                    if not resp_agree(test.at[j, "response_bn"], ar):
                        test.at[j, "pred"] = 0
                        n_b += 1
    print(f"group consistency: rule A unified {n_a} rows | rule B flipped {n_b} rows")

print(f"\nlabel sources: {n_lookup} exact lookup | {int(test.ref_contain.sum())} gold containment -> 1 | "
      f"{int(test.wrong_match.sum())} known-wrong -> 0 | "
      f"{int(((test.ref_text != '') & ~test.ref_contain & ~test.wrong_match).sum())} ref-judge | rest judged")


2516 rows in 13 chunks of 200


chunk0-q:   0%|          | 0/200 [00:00<?, ?it/s]

chunk0-t:   0%|          | 0/200 [00:00<?, ?it/s]

chunk0-qe:   0%|          | 0/200 [00:00<?, ?it/s]

chunk 0: 200 rows in 286s (0.70 rows/s), eta 57 min


...

rejudge-q:   0%|          | 0/121 [00:00<?, ?it/s]

rejudge-qe:   0%|          | 0/121 [00:00<?, ?it/s]

retrieval re-judge applied to 121 closed rows
Qwen-only math base: 130 rows set from p_q > 0.5 (Tiger removed from math)
xling pass: gate off
sympy resolved 4 of 130 math rows
math CoT pass over 126 rows...


math CoT (batched):   0%|          | 0/3 [00:00<?, ?it/s]

CoT changed 0 logit verdicts
idiom tier1 (asked-field match) -> 1 on 76 rows
idiom tier0 (swap trap) -> 0 on 40 rows
ctx tier1 -> 1 on 546 rows
ctx numeric-contradiction -> 0 on 130 grounded rows
BCS tiers: 22 predictions flipped (->1 154, ->0 174)
BenHalluEval tiers: 901 exact test matches (550 gold->1, 351 hall->0) | 10 predictions FLIPPED vs the judge (gold 1, hall 9)
group consistency: rule A unified 0 rows | rule B flipped 0 rows

label sources: 11 exact lookup | 679 gold containment -> 1 | 1 known-wrong -> 0 | 668 ref-judge | rest judged


In [28]:
# ---- v6.10: match audit dump (diagnostic; does NOT affect predictions) --------
AUDIT, UNRESOLVED = [], []

def _why_match(resp, prompt, golds):
    """Which gold string fired contains_gold, and via which sub-rule."""
    r, q = norm_ans(resp), norm_ans(prompt)
    for g0 in golds or ():
        g = norm_ans(g0)
        if len(g) < 2:
            continue
        if g == r:
            return g0, "exact"
        if g in r and g not in q:
            return g0, "gold_in_response"
        if r in g and len(r) >= SUB_RATIO * len(g):
            return g0, "response_in_gold"
    return None, None

_bh_gold_map, _bh_hall_map = {}, {}
if "bh_qa" in globals():
    for _pn, _a in zip(bh_qa.pn, bh_qa.correct_answer):
        if str(_a).strip() and str(_a).lower() != "nan":
            _bh_gold_map.setdefault(_pn, set()).add(str(_a).strip())
    for _pn, _h in zip(bh_hall.pn, bh_hall.hallucinated_answer):
        if str(_h).strip() and str(_h).lower() != "nan":
            _bh_hall_map.setdefault(_pn, set()).add(str(_h).strip())

def _rec(row, tier, asserts, source, matched, how=""):
    AUDIT.append({
        "id": int(row.id), "tier": tier, "tier_says": asserts,
        "final_pred": int(row.pred), "agrees": int(int(row.pred) == asserts),
        "grounded": bool(row.grounded),
        "prompt_bn": row.prompt_bn, "response_bn": row.response_bn,
        "source_dataset": source, "matched_answer": matched, "match_rule": how,
    })

for row in test.itertuples():
    pn = row.p_norm
    srcs = ",".join(sorted(GOLD_SRC.get(pn, set()))) or ""

    if pd.notna(getattr(row, "lookup_label", np.nan)):
        _rec(row, "labeled_exact_lookup", int(row.lookup_label), "dataset samples.json",
             row.response_bn, "exact (prompt,response)")
    if getattr(row, "ref_contain", False):
        g, how = _why_match(row.response_bn, row.prompt_bn, row.ref_pool)
        _rec(row, "gold_containment", 1, srcs, g, how)
    if getattr(row, "wrong_match", False):
        _rec(row, "known_wrong", 0, "dataset samples.json",
             " | ".join(sorted(globals().get("LABELED_WRONG", {}).get(pn, ()))), "exact response match")
    if getattr(row, "bh_gold", False):
        _rec(row, "benhallu_gold_pair", 1, "benhallu",
             " | ".join(sorted(_bh_gold_map.get(pn, ()))), "exact (prompt,response)")
    if getattr(row, "bh_hall", False):
        _rec(row, "benhallu_hallucinated_pair", 0, "benhallu",
             " | ".join(sorted(_bh_hall_map.get(pn, ()))), "exact (prompt,response)")
    if getattr(row, "bcs_gold", False):
        _rec(row, "bcs_correct_option", 1, "bangla-bcs-qs", row.response_bn, "exact (prompt,option)")
    if getattr(row, "bcs_dist", False):
        _rec(row, "bcs_distractor", 0, "bangla-bcs-qs", row.response_bn, "exact (prompt,distractor)")
    if getattr(row, "bcs_tgold", False):
        _t = _bcs_term(row.prompt_bn) if "_bcs_term" in globals() else None
        g, how = _why_match(row.response_bn, row.prompt_bn, BCS_TERM_GOLD.get(_t, set()))
        _rec(row, "bcs_term_gold", 1, f"bangla-bcs-qs[term={_t}]", g, how)
    if getattr(row, "bcs_tdist", False):
        _t = _bcs_term(row.prompt_bn) if "_bcs_term" in globals() else None
        _rec(row, "bcs_term_distractor", 0, f"bangla-bcs-qs[term={_t}]",
             row.response_bn, "exact distractor")
    if getattr(row, "gram_contain", False):
        _rec(row, "grammar_containment", 1, "bangla grammar refs", row.ref_text, "containment")
    if getattr(row, "cb_contain", False):
        _rec(row, "closed_book_ref", 1, "cb_refs",
             " | ".join(sorted(globals().get("CB", {}).get(pn, ()))), "containment")
    if getattr(row, "ctx_num_bad", False):
        _rec(row, "ctx_numeric_contradiction", 0, "(row context)",
             row.response_bn, "number absent from passage")

    # prompt matched a dataset question, but the response matched nothing
    if srcs and not (getattr(row, "ref_contain", False) or getattr(row, "bh_hall", False)
                     or getattr(row, "wrong_match", False)):
        UNRESOLVED.append({
            "id": int(row.id), "grounded": bool(row.grounded), "prompt_bn": row.prompt_bn,
            "response_bn": row.response_bn, "source_dataset": srcs,
            "dataset_answers": " | ".join(sorted(GOLD.get(pn, set()))[:4]),
            "final_pred": int(row.pred),
        })

audit = pd.DataFrame(AUDIT)
unres = pd.DataFrame(UNRESOLVED)
audit.to_csv("match_audit.csv", index=False)
unres.to_csv("match_audit_unresolved.csv", index=False)
print(f"match_audit.csv            : {len(audit)} tier fires over {audit.id.nunique() if len(audit) else 0} distinct test rows")
print(f"match_audit_unresolved.csv : {len(unres)} rows whose prompt matched but response did not\n")

if len(audit):
    summ = (audit.groupby("tier")
                 .agg(fires=("id", "size"), asserts_1=("tier_says", "sum"),
                      agrees_with_final=("agrees", "sum"))
                 .sort_values("fires", ascending=False))
    print(summ.to_string())
    conflict = audit[audit.agrees == 0]
    print(f"\ntier disagrees with the final prediction on {len(conflict)} fires "
          f"(expected: a later, better-validated tier overrode it)")
    if len(conflict):
        print(conflict.groupby(["tier", "tier_says", "final_pred"]).size().to_string())
    # rows where two tiers assert OPPOSITE labels - these are the real red flags
    per = audit.groupby("id").tier_says.nunique()
    bad = per[per > 1].index
    print(f"\nrows with CONTRADICTORY tiers: {len(bad)}"
          + ("  <-- inspect these first" if len(bad) else "  (none)"))
    if len(bad):
        print(audit[audit.id.isin(bad)][["id", "tier", "tier_says", "matched_answer"]].to_string()[:1500])
    print("\n--- sample of matched rows (verify the answers line up) ---")
    cols = ["id", "tier", "prompt_bn", "response_bn", "matched_answer", "source_dataset", "match_rule"]
    print(audit[cols].head(12).to_string(max_colwidth=34))


match_audit.csv            : 2056 tier fires over 1253 distinct test rows
match_audit_unresolved.csv : 230 rows whose prompt matched but response did not

                            fires  asserts_1  agrees_with_final
tier                                                           
gold_containment              679        679                675
benhallu_gold_pair            550        550                550
benhallu_hallucinated_pair    351          0                351
bcs_distractor                140          0                140
ctx_numeric_contradiction     130          0                129
bcs_correct_option            126        126                126
bcs_term_distractor            36          0                 36
bcs_term_gold                  29         29                 29
labeled_exact_lookup           11         10                 11
grammar_containment             3          3                  3
known_wrong                     1          0                  1

tier disagre

In [29]:
sub = sample[["id"]].copy()
sub["label"] = sub.id.map(test.set_index("id")["pred"])
assert sub.label.notna().all(), "some test ids missing predictions"
assert sub.label.isin([0, 1]).all(), "labels must be 0 or 1"
sub["label"] = sub.label.astype(int)
sub.to_csv("submission.csv", index=False)

print("saved submission.csv |", len(sub), "rows")
print("\npredicted label distribution:")
print(sub.label.value_counts(normalize=True).rename("share").round(4))
print("\nper-track prediction rates:")
print(test.groupby("grounded").pred.mean().rename("P(pred=1)").round(4))

run_config = {
    "version": "v6.3-online",
    "base": "v6.2 + Phase-2 source expansion (XL-Sum news, searched corpora), clf normalizer+compass ckpt, lexical stack feats, benhallu-data attachment",
    "model": MODEL_PATH, "qwen": QWEN_PATH, "encoder": CLF_PATH,
    "thresholds": THRESH, "thresholds_blend": THRESH_BLEND,
    "gates": {
        "ensemble": ENSEMBLE_ON, "w_tiger": W_TIGER,
        "stack": STACK_ON, "stack_cols": STACK_COLS if STACK_ON else [],
        "clf_member": CLF_OK,
        "xling_en_pass": USE_XLING_EN, "xling_cot": XLING_OK,
        "sympy": USE_SYMPY, "qa_tier1": QA1_OK, "qa_tier0": QA0_OK,
        "retrieval": USE_RET_FINAL, "ret_cut": RET_CUT,
        "rel_route": USE_REL_FINAL, "rel_cut": REL_CUT,
        "grammar_tier": GRAM1_OK, "grammar_terms": len(GRAM),
        "ctx_tier1": CTX1_OK, "ctx_tier0": CTX0_OK,
        "cb_tier1": CB1_OK, "cb_tier0": CB0_OK, "cb_questions": len(CB),
        "bh_gold": BH1_OK, "bh_hall": BH0_OK, "bh_m1": BHM1_OK, "bh_m0": BHM0_OK,
        "group_rule_b": GROUPB_OK,
        "c1_traps_enabled": sorted(C1_ENABLED),
    },
    "batch_size": BATCH_SIZE, "max_context_chars": MAX_CTX_CHARS,
    "n_lookup_labels": n_lookup,
    "n_ref_rows": int((test.ref_text != "").sum()),
    "n_gold_containment": int(test.ref_contain.sum()),
    "n_known_wrong": int(test.wrong_match.sum()),
    "n_grammar_refs": int(test.gram_hit.sum()),
    "n_ctx_matched": int(test.ctx_hit.sum()),
    "n_cb_matched": int(test.cb_hit.sum()),
    "n_c1_fires": int(test.c1_wrong.sum() + test.c1_right.sum()),
}
def _json_default(o):
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)

with open("run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, ensure_ascii=False, default=_json_default)
print("\n" + json.dumps(run_config, indent=2, ensure_ascii=False, default=_json_default))


saved submission.csv | 2516 rows

predicted label distribution:
label
1    0.5421
0    0.4579
Name: share, dtype: float64

per-track prediction rates:
grounded
False    0.4727
True     0.6010
Name: P(pred=1), dtype: float64

{
  "version": "v6.3-online",
  "base": "v6.2 + Phase-2 source expansion (XL-Sum news, searched corpora), clf normalizer+compass ckpt, lexical stack feats, benhallu-data attachment",
  "model": "md-nishat-008/TigerLLM-9B-it",
  "qwen": "Qwen/Qwen3.5-9B",
  "encoder": "csebuetnlp/banglabert",
  "thresholds": {
    "grounded": 0.35000000000000003,
    "closed": 0.7500000000000001,
    "ref": 0.6500000000000001,
    "closed_ret": 0.6000000000000001
  },
  "thresholds_blend": {
    "grounded": 0.9500000000000001,
    "closed": 0.6000000000000001
  },
  "gates": {
    "ensemble": true,
    "w_tiger": 0.4,
    "stack": true,
    "stack_cols": [
      "p_t",
      "p_q",
      "p_qe",
      "p_clf",
      "lex_jac",
      "r_len",
      "ctx_cov",
      "ctx_numbad"
    ]